# One-Phonon Diffuse Scattering — Fit to Experimental Data (Mac1, P4₃, model 7tx0)

SARS-CoV-2 NSP3 macrodomain (Mac1) in the P4₃ crystal form. There are two chains in
the asymmetric unit and four symmetry operations, so the unit cell holds **eight
rigid bodies**. The rigid-body lattice model, its density, its contacts and its
symmetry-respecting spring parameterization are exactly those of the synthetic
7tx0 notebook. Here the stiffnesses are refined against a measured diffuse map: the
merged, scaled diffuse intensities produced by **mdx2** (Meisburger & Ando 2024)
from the CHESS 2025-11-05 data.

**The structure and the data come from different crystals.** The electron density,
the bodies and the contacts come from PDB 7tx0. The diffuse map was measured on
another crystal of the same form. That is acceptable for a rigid-body lattice
model, which needs the molecule's transform and its packing, but two consequences are
checked explicitly rather than assumed:

* **Unit cell.** The two cells differ slightly. The model is evaluated at the data's
  Miller indices with the 7tx0 cell, and the difference is reported.
* **Indexing.** Point group 4 is merohedral: the lattice has 4/mmm symmetry, but the
  crystal only 4. Two crystals indexed independently can therefore differ by the
  twinning operator, e.g. $(h,k,l)\to(k,h,-l)$, and nothing in the processing
  fixes the choice. The notebook tests every alternative indexing against the
  structure (halo brightness against $|F_{\rm cell}(\mathbf G)|^2$) and uses the one the
  data support.

**What mdx2 delivers.** `mdx2.merge` writes a NeXus file (`merged.nxs`) holding an
`hkl_table`: one row per voxel of a Miller-index grid subdivided `ndiv` times along
each axis, restricted to the reciprocal-space asymmetric unit of the Laue group. The
columns are `intensity`, `intensity_error` and `count`, and, when the merge was
split, `group_0_intensity` and `group_1_intensity` (two independent half-datasets).
The geometry (`crystal`: cell and UB matrix; `symmetry`: Laue operators and ASU) is
either in the same file or in the `geometry.nxs` written by
`mdx2.import_geometry`. The file is read directly with `h5py`, so neither mdx2 nor
DIALS is needed here.

**Pipeline:** structure, symmetry and rigid bodies → hybrid density, partitioned
among the bodies → transforms and $G(\mathbf q)$ → contacts, orbits and the prior →
dynamical matrix → **reading the mdx2 map** (geometry, indexing test, resolution
diagnostics, half-dataset CC) → halo selection → calibration of the prior's scale →
cross-validated λ → refinement → what the data determine (sloppiness, posterior,
elastic tensor) → maps, CC by resolution against the half-dataset ceiling, ADPs,
mode animations.

**Requirements.** `gemmi`, `numpy`, `scipy`, `h5py`, `matplotlib`, `scikit-image`,
`imageio` and `finufft`.

In [1]:
import io
import os
import pathlib
import urllib.request
from itertools import product

import numpy as np
import gemmi
import h5py
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.lines import Line2D
import scipy.constants as const
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components as sc_connected_components
from scipy.interpolate import griddata
from scipy.optimize import minimize
from scipy.spatial.transform import Rotation as Rot
import imageio
import imageio.v2 as iio2
from skimage.measure import marching_cubes
from IPython.display import Image as IPImage, display
import gc

try:
    import finufft
    HAVE_FINUFFT = True
except ImportError:
    HAVE_FINUFFT = False
    print("finufft not found (pip install finufft): the molecular transform falls back to "
          "direct summation, which is exact but slow at this resolution.")

np.set_printoptions(precision=4, suppress=True)

def fig_to_image(fig, dpi=72):
    """Rasterize a matplotlib figure to an RGB array, for building GIFs."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor=fig.get_facecolor())
    buf.seek(0)
    return iio2.imread(buf)[:, :, :3]

In [2]:
PDB_ID     = "7tx0"
PDB_FILE   = pathlib.Path(f"{PDB_ID}.cif")
SF_CIF     = pathlib.Path(f"{PDB_ID}-sf.cif")
DOWNLOAD_IF_MISSING = True            # fetch the two PDB files from RCSB if they are absent

# --- The mdx2 data -----------------------------------------------------------------
# merged.nxs of the CHESS 2025-11-05 dataset; in the mac1-east-west project it is
# <project_directory>/chess_20251105/mac1_merge/mdx2/merged.nxs
MDX2_FILE     = pathlib.Path("chess_20251105/mac1_merge/mdx2/merged.nxs")
MDX2_GEOMETRY = None      # NeXus file with 'crystal' and 'symmetry' if merged.nxs lacks them
                          # (e.g. .../mdx2/geometry.nxs); None searches next to MDX2_FILE
DATA_COLUMN   = 'intensity'   # merged intensity column; its error is DATA_COLUMN + '_error'
                              # ('intensity_error' for 'intensity')
REINDEX       = 'auto'    # 'auto': test the alternative indexings and use the best;
                          # 'none'; or an explicit matrix M with h_model = M h_data
REINDEX_STRICT = False    # True: stop instead of reindexing automatically

# --- Fitting mode -------------------------------------------------------------------
# FIT_MODE = 'halo' (Meisburger-like): whole reciprocal cells around N_HALO_SELECT bright
#   halos, with natural 1/sigma^2 weights; whole halos are held out and the lambda folds
#   are grouped by reciprocal-lattice point, so no halo is split between training and test.
# FIT_MODE = 'both': a stratified mid-zone sample plus halo voxels within HALO_R_MAX.
FIT_MODE = 'halo'
assert FIT_MODE in ('both', 'halo'), FIT_MODE
# prefix for every figure, GIF and cache file this notebook writes
OUT_PREFIX = f"experimental_{PDB_ID}_" + ("halo_" if FIT_MODE == 'halo' else "")

# --- Rigid bodies -------------------------------------------------------------------
BODY_CHAINS       = None   # asymmetric-unit chains that become rigid bodies; None = every
                           # chain with polymer residues (A and B for 7tx0)
LIGANDS_TO_BODIES = True   # ligands join the body whose polymer atoms are nearest
PHASES_FROM_FULL_MODEL = True   # model phases from the complete model (waters, H)

# --- Resolution limits (Å; smaller = finer) -----------------------------------------
# Required order: D_MIN_MODEL_DENSITY <= D_MIN_TRANSFORM < D_FIT_MIN < D_MIN_HALO.
D_MIN_TRANSFORM     = 0.84   # reflections used for the hybrid density, hence for F, L, G
FILL_MISSING        = True   # fill unmeasured reflections with bin-scaled model amplitudes
RATE_TRANSFORM      = 2.5    # density-grid oversampling (spacing ~ D_MIN_TRANSFORM/rate)
D_MIN_MODEL_DENSITY = 0.84   # calculated model density: source of the phases
RATE_MODEL_DENSITY  = 1.5
UNWRAP_IMAGE        = 1      # lattice images searched when assigning voxels to bodies
TRANSFORM_EPS       = 1e-10  # requested relative accuracy of the NUFFT transforms
VOXEL_CHUNK         = 1_000_000

D_FIT_MIN, D_FIT_MAX   = 1.9, 12.0   # shell collected from the map for fitting
D_MIN_HALO, D_MAX_HALO = 2.0, 2.5    # window for the CENTRES of the halos that can be
                                     # selected (Meisburger et al. 2020: 400 bright halos
                                     # between 2 and 2.5 Å)
D_READ_MIN, D_READ_MAX = 1.9, 20.0   # shell kept in memory from the map (fitting, maps, CC)

# --- Contacts -----------------------------------------------------------------------
CONTACT_CUTOFF = 4.0
MIN_CONTACTS   = 5
MAX_IMAGE      = 1

# --- Diffuse model --------------------------------------------------------------------
D_MIN_DIFFUSE  = 3.0    # Å — limit of the model-only prior maps
NPLOT_HK       = 6
TEMPERATURE    = 300.0  # K — used only for unit conversions
HALO_R_MAX     = 0.03   # Å⁻¹ — near-Bragg radius (FIT_MODE = 'both'); a* = 0.071 Å⁻¹

# --- Voxel selection ------------------------------------------------------------------
BRAGG_EXCL_CART = 0.010   # Å⁻¹: the Bragg-centred voxel itself is excluded (it holds the
                          # residual Bragg peak); with ndiv = 4 along a the first
                          # off-Bragg voxel is at a*/4 = 0.018 Å⁻¹
N_FIT_MAX      = 6000     # mid-zone points (FIT_MODE = 'both')
HOLDOUT_FRAC   = 0.15
N_RES_STRATA   = 6
N_HALO_STRATA  = 8
N_HALO_SHELLS  = 5
N_HALO_FIT     = 3000     # halo voxels (FIT_MODE = 'both')
HALO_EXTENT    = 'cell'   # FIT_MODE = 'halo': 'cell' (the whole reciprocal cell around G)
                          # or 'sphere' (within HALO_R_MAX)
HALO_SAMPLING  = 'uniform'
N_HALO_SELECT  = 400
N_HALO_FIT_ONLY = 40000   # halo voxels fitted when FIT_MODE = 'halo' (None: all)
HALO_HOLDOUT_FRAC = 0.15
HALO_WEIGHT    = 1.0
SIGMA_FLOOR_PCT = 5.0     # percentile floor on sigma

# --- Isotropic background ---------------------------------------------------------------
# I_model(q) = I_TDS(q; K) + b(|q|), b piecewise linear on N_BG_KNOTS knots over the |q|
# range of the fitting points, solved exactly for every K. mdx2 maps are total
# scattering (solvent, Compton, liquid-like motion are all in them), so the background
# is on by default.
USE_BACKGROUND = True
N_BG_KNOTS     = 20 if FIT_MODE == 'both' else 6

# --- Prior and regularization ------------------------------------------------------------
PRIOR_K_PER_PAIR = 1.0    # k_BT/Å² per atom pair; only the prior's SHAPE matters, its
                          # magnitude is calibrated against the data before the fit
LAMBDA_PRIOR     = 3e-1   # selected by cross-validation below
REG_EIG_FLOOR    = 1e-10

# --- Bulk solvent ------------------------------------------------------------------------
SOLVENT_PROBE = 1.1
SOLVENT_EDGE  = 0.7
SOLVENT_SHELL = 1.5

CHUNK_EVAL = 4000
BZ_NGRID   = 20

def _fetch(path, url):
    if path.exists():
        return
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(f"{path} not found (set DOWNLOAD_IF_MISSING = True to fetch it)")
    print(f"downloading {url} -> {path}")
    urllib.request.urlretrieve(url, path)

In [3]:
_fetch(PDB_FILE, f"https://files.rcsb.org/download/{PDB_ID.upper()}.cif")
_fetch(SF_CIF,   f"https://files.rcsb.org/download/{PDB_ID.upper()}-sf.cif")
if not pathlib.Path(MDX2_FILE).exists():
    raise FileNotFoundError(
        f"{MDX2_FILE} not found. It is produced by mdx2_process.sh in the mac1-east-west "
        f"project (raw images from OSN via sync_raw_data.sh, then DIALS and mdx2); set "
        f"MDX2_FILE to its location.")

# --- Consistency of the resolution limits -------------------------------------------------
if D_MIN_MODEL_DENSITY > D_MIN_TRANSFORM:
    print(f"WARNING: D_MIN_MODEL_DENSITY = {D_MIN_MODEL_DENSITY} Å is coarser than "
          f"D_MIN_TRANSFORM = {D_MIN_TRANSFORM} Å.")
if min(D_FIT_MIN, D_MIN_HALO) < 1.2*D_MIN_TRANSFORM:
    print(f"WARNING: fitting to {min(D_FIT_MIN, D_MIN_HALO)} Å with the transform built to "
          f"{D_MIN_TRANSFORM} Å; lower D_MIN_TRANSFORM first.")
assert D_READ_MIN <= D_FIT_MIN and D_READ_MAX >= D_FIT_MAX, "the read shell must contain the fitting shell"

# --- Structure, unit cell and space group ------------------------------------------
st_full = gemmi.read_structure(str(PDB_FILE))     # complete model: used for the phases
st_full.setup_entities()
st = st_full.clone()                              # rigid-body atoms: no H, no waters
st.remove_hydrogens()
st.remove_waters()
st.remove_empty_chains()

cell = st.cell
A_orth  = np.array(cell.orth.mat.tolist())         # orthogonalization matrix (columns a, b, c)
A_inv   = np.linalg.inv(A_orth)
a1, a2, a3 = A_orth[:, 0], A_orth[:, 1], A_orth[:, 2]
B_recip = 2*np.pi*A_inv.T                          # reciprocal lattice (columns)

SG = st.find_spacegroup()
SYM_OPS = [(np.array(op.rot, float)/op.DEN, np.array(op.tran, float)/op.DEN)
           for op in SG.operations()]              # fractional: x -> W x + w
SYM_TRIPLETS = [op.triplet() for op in SG.operations()]
N_OPS = len(SYM_OPS)
assert np.allclose(SYM_OPS[0][0], np.eye(3)) and np.allclose(SYM_OPS[0][1], 0), \
    "the first symmetry operation is expected to be the identity"
SYM_ROT_CART = [A_orth @ W @ A_inv for W, _ in SYM_OPS]   # Cartesian rotation of each op
for R in SYM_ROT_CART:
    assert np.allclose(R @ R.T, np.eye(3), atol=1e-8) and np.linalg.det(R) > 0, \
        "a symmetry operation is not a proper rotation in Cartesian space"

print(f"Cell: {cell.a:.3f}×{cell.b:.3f}×{cell.c:.3f} Å  "
      f"α={cell.alpha:.2f} β={cell.beta:.2f} γ={cell.gamma:.2f}°   Volume: {cell.volume:.0f} Å³")
print(f"Space group: {SG.hm} (No. {SG.number}), {N_OPS} operations:")
for k, t in enumerate(SYM_TRIPLETS):
    print(f"   s={k}: {t}")

FileNotFoundError: chess_20251105/mac1_merge/mdx2/merged.nxs not found. It is produced by mdx2_process.sh in the mac1-east-west project (raw images from OSN via sync_raw_data.sh, then DIALS and mdx2); set MDX2_FILE to its location.

## Rigid Bodies and Their Symmetry Images

**Atoms of a body.** Each asymmetric-unit chain listed in `BODY_CHAINS` becomes one
rigid body: its polymer atoms plus, with `LIGANDS_TO_BODIES`, every ligand residue
whose nearest polymer atom belongs to that chain. The search runs over all
symmetry mates and lattice translations, because a ligand deposited under chain B's
label can sit against a symmetry copy of B. Such a ligand is moved into chain B's
own frame by the inverse operation. Hydrogens and waters are not part of any body.
Atoms are weighted by occupancy in the mass, the inertia and the electron count, so
alternate conformations are counted once.

**Images.** For operation $s$, $(W_s,\mathbf w_s)$ in fractional coordinates, body
$b=(s,c)$ occupies

$$\mathbf f\mapsto W_s\mathbf f+\mathbf w_s+\mathbf m_b,$$

with the integer shift $\mathbf m_b$ that puts its centre of mass in $[0,1)^3$. In
Cartesian coordinates this is $\mathbf r\mapsto R_b\mathbf r+\mathbf t_b$ with
$R_b=AW_sA^{-1}$ (a proper rotation) and $\mathbf t_b=A(\mathbf w_s+\mathbf m_b)$.
Each chain is first shifted by a lattice vector so that its own centre of mass is in
the cell, so the asymmetric-unit bodies have $R_b=I$, $\mathbf t_b=0$. Bodies are
numbered $b=s\,N_{\rm chain}+c$, so bodies $0,\dots,N_{\rm chain}-1$ are the
asymmetric unit.

**Generalized coordinates.** $u_b=(\boldsymbol\Omega_b,\mathbf v_b)$ about the
body's own centre of mass, so the mass matrix is block-diagonal,
$M=\mathrm{diag}(J_b,m_bI)$, with $J_b=R_bJ_cR_b^{\mathsf T}$. All bodies of one chain
have the same mass and inertia eigenvalues.

In [ ]:
ATOMIC_MASS = {'C': 12.011, 'N': 14.007, 'O': 15.999, 'S': 32.06, 'P': 30.974,
               'SE': 78.96, 'H': 1.008, 'FE': 55.845, 'ZN': 65.38, 'CA': 40.078,
               'MG': 24.305, 'CL': 35.45, 'NA': 22.99, 'K': 39.098}
VDW_RADIUS = {'C': 1.70, 'N': 1.55, 'O': 1.52, 'S': 1.80, 'P': 1.80, 'H': 1.20,
              'SE': 1.90, 'FE': 2.00, 'ZN': 1.39, 'CA': 2.31, 'MG': 1.73, 'CL': 1.75,
              'NA': 2.27, 'K': 2.75}

if BODY_CHAINS is None:
    BODY_CHAINS = [ch.name for ch in st[0] if len(ch.get_polymer()) > 0]
N_CHAIN = len(BODY_CHAINS)
CHAIN_INDEX = {c: i for i, c in enumerate(BODY_CHAINS)}
print(f"rigid-body chains of the asymmetric unit: {BODY_CHAINS}")

def _atoms_of(residues, chain_name, is_ligand):
    rows = []
    for res in residues:
        for atom in res:
            if atom.occ <= 0:
                continue
            el = atom.element.name.upper()
            rows.append(dict(pos=np.array(atom.pos.tolist()), el=el, Z=atom.element.atomic_number,
                             occ=float(atom.occ), b=float(atom.b_iso),
                             U=np.array(atom.aniso.as_mat33().tolist()), aniso=bool(atom.aniso.nonzero()),
                             name=atom.name, res=(chain_name, res.seqid.num, res.seqid.icode, res.name),
                             lig=is_ligand))
    return rows

_rows = {c: [] for c in BODY_CHAINS}
_ligands = []
for ch in st[0]:
    poly = ch.get_polymer()
    poly_ids = {(r.seqid.num, r.seqid.icode) for r in poly}
    if ch.name in _rows:
        _rows[ch.name] += _atoms_of(poly, ch.name, False)
    for res in ch:
        if (res.seqid.num, res.seqid.icode) not in poly_ids and not res.is_water():
            _ligands.append((ch.name, res))

# --- Ligands: nearest polymer atom over every symmetry mate and lattice translation ----
_n_tr = np.array(list(product(range(-2, 3), repeat=3)), float)
_img_pts, _img_lab = [], []
for c in BODY_CHAINS:
    f_c = np.array([r['pos'] for r in _rows[c]]) @ A_inv.T
    for s, (W, w) in enumerate(SYM_OPS):
        fs = f_c @ W.T + w
        for n in _n_tr:
            _img_pts.append((fs + n) @ A_orth.T)
            _img_lab.append((c, s, tuple(n.astype(int)), len(fs)))
_img_tree = cKDTree(np.vstack(_img_pts))
_img_owner = np.concatenate([np.full(nl[3], k) for k, nl in enumerate(_img_lab)])
LIGAND_LOG = []
for ch_name, res in _ligands:
    if not LIGANDS_TO_BODIES:
        LIGAND_LOG.append((ch_name, res.name, res.seqid.num, 'dropped', None, None, None)); continue
    P = np.array([a.pos.tolist() for a in res if a.occ > 0])
    if len(P) == 0:
        continue
    d, j = _img_tree.query(P)
    c, s, n, _ = _img_lab[_img_owner[j[np.argmin(d)]]]
    W, w = SYM_OPS[s]
    rows = _atoms_of([res], c, True)
    Winv = np.linalg.inv(W); Rinv = A_orth @ Winv @ A_inv
    for r in rows:      # into chain c's own frame: f_c = W^-1 (f - w - n)
        r['pos'] = A_orth @ (Winv @ (A_inv @ r['pos'] - w - np.array(n, float)))
        r['U'] = Rinv @ r['U'] @ Rinv.T
    _rows[c] += rows
    LIGAND_LOG.append((ch_name, res.name, res.seqid.num, c, s, tuple(int(x) for x in n), float(d.min())))
del _img_pts, _img_tree

if LIGAND_LOG:
    print(f"\n{len(LIGAND_LOG)} ligand residue(s):")
    for ch_name, rn, num, c, s, n, dmin in LIGAND_LOG:
        if c == 'dropped':
            print(f"   {ch_name}/{rn}{num}: dropped (LIGANDS_TO_BODIES = False)")
        else:
            moved = '' if (s == 0 and n == (0, 0, 0)) else f"  (moved by s={s}, n={n} into chain {c}'s frame)"
            print(f"   {ch_name}/{rn}{num}: -> body chain {c}, nearest polymer atom {dmin:.2f} Å{moved}")

# --- Per-chain atom arrays, centre of mass, inertia ----------------------------------
CHAINS = {}
for c in BODY_CHAINS:
    rows = _rows[c]
    pos = np.array([r['pos'] for r in rows])
    occ = np.array([r['occ'] for r in rows])
    el  = [r['el'] for r in rows]
    mass = np.array([ATOMIC_MASS.get(e, 12.0) for e in el])*occ
    com = (mass[:, None]*pos).sum(0)/mass.sum()
    shift = -np.floor(A_inv @ com + 1e-9)               # lattice shift: COM into the cell
    pos = pos + A_orth @ shift
    com = com + A_orth @ shift
    d = pos - com
    J = (mass[:, None, None]*((d**2).sum(1)[:, None, None]*np.eye(3) - d[:, :, None]*d[:, None, :])).sum(0)
    CHAINS[c] = dict(pos=pos, occ=occ, el=el, mass_at=mass, m=float(mass.sum()), com=com, J=J,
                     Z=np.array([r['Z'] for r in rows])*occ,
                     vdw=np.array([VDW_RADIUS.get(e, 1.70) for e in el]),
                     b=np.array([r['b'] for r in rows]), U=np.array([r['U'] for r in rows]),
                     aniso=np.array([r['aniso'] for r in rows]), names=[r['name'] for r in rows],
                     res=[r['res'] for r in rows], lig=np.array([r['lig'] for r in rows]),
                     shift=tuple(int(x) for x in shift))

# --- Bodies: every symmetry image of every chain ------------------------------------
BODIES = []
for s, (W, w) in enumerate(SYM_OPS):
    for c in BODY_CHAINS:
        ch = CHAINS[c]
        f_com = W @ (A_inv @ ch['com']) + w
        m_b = -np.floor(f_com + 1e-9)
        tau = w + m_b
        R = A_orth @ W @ A_inv
        BODIES.append(dict(chain=c, op=s, W=W, tau=tau, R=R, t=A_orth @ tau,
                           com=R @ ch['com'] + A_orth @ tau, m=ch['m'], J=R @ ch['J'] @ R.T))
N_BODY = len(BODIES)
N_DOF  = 6*N_BODY
ASU_BODY = {c: CHAIN_INDEX[c] for c in BODY_CHAINS}
for b in BODIES[:N_CHAIN]:
    assert b['op'] == 0 and np.allclose(b['t'], 0), "asymmetric-unit bodies must be untransformed"

def body_positions(b):
    """Cartesian atom positions of body b (cell 0)."""
    B = BODIES[b]
    return CHAINS[B['chain']]['pos'] @ B['R'].T + B['t']

def S_of(R):
    """6x6 action of a proper rotation on (Omega, v)."""
    S = np.zeros((6, 6)); S[:3, :3] = R; S[3:, 3:] = R
    return S

M_mat = np.zeros((N_DOF, N_DOF))
for b, B in enumerate(BODIES):
    M_mat[6*b:6*b+3, 6*b:6*b+3] = B['J']
    M_mat[6*b+3:6*b+6, 6*b+3:6*b+6] = B['m']*np.eye(3)
Msq     = np.linalg.cholesky(M_mat)
Msq_inv = np.linalg.inv(Msq)
m_cell  = sum(B['m'] for B in BODIES)

kBT_SI     = const.k*TEMPERATURE
omega_unit = np.sqrt(kBT_SI/(const.atomic_mass*(1e-10)**2))
freq_unit  = omega_unit/(2*np.pi)/1e12       # THz per sqrt(reduced stiffness/mass)

Z_ASU   = float(sum(CHAINS[c]['Z'].sum() for c in BODY_CHAINS))
Z_model = N_OPS*Z_ASU                        # electrons of all rigid-body atoms in the cell
print(f"\n{'chain':>5s} {'atoms':>6s} {'ligand atoms':>13s} {'mass (amu)':>11s} {'electrons':>10s} "
      f"{'alt. conf. atoms':>17s} {'ANISOU':>7s}  lattice shift")
for c in BODY_CHAINS:
    ch = CHAINS[c]
    print(f"{c:>5s} {len(ch['pos']):6d} {int(ch['lig'].sum()):13d} {ch['m']:11.0f} {ch['Z'].sum():10.0f} "
          f"{int((ch['occ'] < 0.999).sum()):17d} {ch['aniso'].mean():7.0%}  {ch['shift']}")
print(f"\n{N_BODY} rigid bodies in the cell ({N_CHAIN} chains × {N_OPS} operations), "
      f"{N_DOF} degrees of freedom; cell mass {m_cell:.0f} amu")
print(f"{'body':>5s} {'chain':>6s} {'op':>16s} {'COM (fractional)':>26s}   cell shift")
for b, B in enumerate(BODIES):
    f = A_inv @ B['com']
    print(f"{b:5d} {B['chain']:>6s} {SYM_TRIPLETS[B['op']]:>16s}   ({f[0]:.3f}, {f[1]:.3f}, {f[2]:.3f})   "
          f"{tuple(int(x) for x in (B['tau'] - SYM_OPS[B['op']][1]).round())}")
print(f"\nFrequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")

## Electron Density for Each Rigid Body

### What has to move with each body

Write the crystal density as the eight bodies (and their lattice images) plus a
flat bulk solvent filling the rest:

$$\rho_c(\mathbf r)=\rho_{\rm sol}+\sum_{\mathbf n}\sum_b
\underbrace{\bigl[\rho_b(\mathbf r)-\rho_{\rm sol}M_b(\mathbf r)\bigr]}_{\rho_{{\rm eff},b},\ \text{moves with body }b}
(\mathbf r-\mathbf T_{\mathbf n}).$$

When body $b$ is displaced, its atoms and the hole it makes in the solvent move
together. The object each body carries is therefore its own **contrast density**
$\rho_{{\rm eff},b}$: its electron density minus the solvent it excludes. This
needs two things per body: a share of the electron density, and a share of the
excluded-volume envelope.

### Where the density comes from

As for 6o2h, the density is the **hybrid map**: measured amplitudes with phases
from the deposited model, so it contains bonding density, ordered solvent and the
real disorder, which a form-factor model does not. Its transform is
$F_0e^{-W}$, so the Debye–Waller factor is already included. Two things change for
P4₃:

1. **Symmetry expansion.** The deposited amplitudes cover only the reciprocal
   asymmetric unit. Every measured reflection is expanded over the Laue group
   ($|F(\mathbf hW_s)|=|F(\mathbf h)|$, and Friedel mates), and each expanded index takes
   the phase of the calculated model density at that index. The model density is
   symmetrized over the space group, so the phases are consistent between
   equivalent reflections and the map has the full P4₃ symmetry. Systematic
   absences are never filled.
2. **Phases from the complete model.** With `PHASES_FROM_FULL_MODEL`, the model
   density includes waters and riding hydrogens. At 0.84 Å these improve the
   phases, and the rigid bodies are still built only from protein and ligand atoms.

The grid is chosen so that every symmetry operation maps grid points onto grid
points (equal sizes along $a$ and $b$, $n_c$ divisible by 4). The partition below
is then exactly symmetric.

### Splitting the density among the bodies

Each voxel is assigned to one body and one lattice image: the one whose nearest
atom *surface* (centre distance minus van der Waals radius) is closest, over all
eight bodies and all lattice images within `UNWRAP_IMAGE`. This is the P1
unwrapping of 6o2h, generalized from "which lattice image" to "which body and which
lattice image". It is a fundamental domain of the space group: every voxel goes to
exactly one body, the images of the asymmetric-unit chains tile the cell, and a
voxel owned by body $(s,c)$ is the image of one owned by chain $c$.

Two properties make this choice adequate here, and the cells check both:

* **The boundaries lie where the density is low.** Two bodies in contact are
  separated by a 3–4 Å gap. At 0.84 Å the atomic density has decayed to the
  solvent level well before the midplane of that gap. A voxel's owner is uncertain
  only within a few tenths of an Å of the boundary, and the cell reports how much
  contrast lies there.
* **The excluded volume follows the same partition.** The envelope
  $M=\tfrac12[1-\tanh((d_{\rm surf}-p)/w)]$ is computed from the distance to the
  nearest atom surface of any body. Between two bodies in contact the envelopes
  merge, and the merged region is split at the same boundary as the density. Body
  $b$'s envelope is $M_b=M\cdot[\text{voxel owned by }b]$, so $\sum_bM_b=M$ and
  $\sum_b\rho_{{\rm eff},b}=(\rho-\rho_{\rm sol})M$ exactly.

A finer split, such as a stockholder partition by the promolecule density of each
body, would change only the few tenths of an Å next to each boundary. With atomic
Gaussians of width ~0.4 Å, a stockholder boundary is nearly as sharp as this one.

### Transforms

For each asymmetric-unit chain the contrast density, with each voxel at its
unwrapped position, is placed on a box of grid indices. $F$ and $L$ at any
fractional $hkl$ are a type-2 non-uniform FFT of that box (`finufft`, accurate to
`TRANSFORM_EPS`), which replaces the axis-by-axis contraction of the P1 notebook.
The other bodies follow from symmetry (next section).

In [ ]:
# --- Hybrid density: measured |F| (symmetry-expanded), model phases -----------------
def _good_fft_size(n):
    """Smallest even 2,3,5-smooth integer >= n (FFT-friendly)."""
    n = max(4, int(np.ceil(n)))
    while True:
        m = n
        for p in (2, 3, 5):
            while m % p == 0:
                m //= p
        if m == 1 and n % 2 == 0:
            return n
        n += 1

def symmetry_compatible_sizes(n):
    """Grid sizes such that every operation maps grid points onto grid points:
    equal sizes along axes that an operation mixes, and n_i * w_i integer."""
    n = [_good_fft_size(x) for x in n]
    for _ in range(50):
        old = list(n)
        for W, w in SYM_OPS:
            for i in range(3):
                for j in range(3):
                    if i != j and abs(W[i, j]) > 1e-9:
                        n[i] = n[j] = max(n[i], n[j])
            for i in range(3):
                while abs(n[i]*w[i] - round(n[i]*w[i])) > 1e-9:
                    n[i] = _good_fft_size(n[i] + 1)
        if n == old:
            return n
    raise RuntimeError("could not find a symmetry-compatible grid")

def hkl_key(h):
    """One int64 per integer (h, k, l) (exact, not Friedel-folded)."""
    h = np.asarray(h, np.int64) + 512
    return (h[..., 0]*1024 + h[..., 1])*1024 + h[..., 2]

def _friedel_key(h):
    h = np.asarray(h, np.int64)
    lead = np.where(h[:, 0] != 0, h[:, 0], np.where(h[:, 1] != 0, h[:, 1], h[:, 2]))
    h = h*np.where(lead < 0, -1, 1)[:, None]
    return hkl_key(h)

def systematically_absent(H):
    """True where the space group extinguishes the reflection."""
    H = np.asarray(H, float); out = np.zeros(len(H), bool)
    for W, w in SYM_OPS:
        fixed = np.all(np.abs(H @ W - H) < 1e-9, axis=1)
        ph = H @ w
        out |= fixed & (np.abs(ph - np.round(ph)) > 1e-6)
    return out

def _decode_key(k):
    k = np.asarray(k, np.int64)
    return np.stack([k//(1024*1024) - 512, (k//1024) % 1024 - 512, k % 1024 - 512], 1)

def laue_orbit_key(H):
    """One id per Laue orbit: the smallest exact key among h W_s and -h W_s."""
    H = np.asarray(H, np.int64)
    return np.min(np.stack([hkl_key(sg*(H @ W.astype(np.int64))) for W, _ in SYM_OPS for sg in (1, -1)]), axis=0)

def expand_with_phases(H, F):
    """Expand complex structure factors (crystallographic e^{+2 pi i h.x} convention) of
    one representative per orbit over the space group, F(h W_s) = F(h) e^{-2 pi i h.w_s}
    and F(-h) = conj F(h). Exact, so the map built from them has the full symmetry."""
    H = np.asarray(H, np.int64)
    Hs, Fs = [], []
    for W, w in SYM_OPS:
        Hw = H @ W.astype(np.int64)
        Fw = F*np.exp(-2j*np.pi*(H @ w))
        Hs += [Hw, -Hw]; Fs += [Fw, np.conj(Fw)]
    Hs = np.vstack(Hs); Fs = np.concatenate(Fs)
    _, first = np.unique(hkl_key(Hs), return_index=True)
    return Hs[first], Fs[first]

def _read_amplitudes(path):
    """Measured amplitudes from the first reflection block that has any. Falls back from
    F_meas_au to anomalous F+/F-, then to intensities."""
    doc = gemmi.cif.read(str(path))
    for rb in gemmi.as_refln_blocks(doc):
        lab = set(rb.column_labels())
        col = lambda t: np.array(rb.make_float_array(t), float)
        if 'F_meas_au' in lab:
            F, src = col('F_meas_au'), 'F_meas_au'
        elif 'F_meas' in lab:
            F, src = col('F_meas'), 'F_meas'
        elif {'pdbx_F_plus', 'pdbx_F_minus'} <= lab:
            Fp, Fn = col('pdbx_F_plus'), col('pdbx_F_minus')
            F = np.where(np.isfinite(Fp) & np.isfinite(Fn), 0.5*(Fp + Fn), np.where(np.isfinite(Fp), Fp, Fn))
            src = 'mean of pdbx_F_plus / pdbx_F_minus'
        elif 'intensity_meas' in lab:
            I = col('intensity_meas'); F = np.sqrt(np.where(I > 0, I, np.nan)); src = 'sqrt(intensity_meas)'
        elif {'pdbx_I_plus', 'pdbx_I_minus'} <= lab:
            Ip, In = col('pdbx_I_plus'), col('pdbx_I_minus')
            I = np.where(np.isfinite(Ip) & np.isfinite(In), 0.5*(Ip + In), np.where(np.isfinite(Ip), Ip, In))
            F = np.sqrt(np.where(I > 0, I, np.nan)); src = 'sqrt(mean pdbx_I_plus / pdbx_I_minus)'
        else:
            continue
        hkl = np.array(rb.make_miller_array(), np.int64)
        ok = np.isfinite(F) & (F > 0)
        return hkl[ok], F[ok], src, rb.block.name
    raise ValueError(f"no amplitudes or intensities found in {path}")

miller, F_meas, _amp_src, _amp_block = _read_amplitudes(SF_CIF)
q_refl = miller @ B_recip.T
d_refl = 2*np.pi/np.maximum(np.linalg.norm(q_refl, axis=1), 1e-9)
use = (d_refl >= D_MIN_TRANSFORM) & ~systematically_absent(miller)
mil_u, Fm_u = miller[use], F_meas[use]
_, _first = np.unique(laue_orbit_key(mil_u), return_index=True)      # one per orbit
mil_u, Fm_u = mil_u[_first], Fm_u[_first]
d_u = 2*np.pi/np.linalg.norm(mil_u @ B_recip.T, axis=1)
print(f"amplitudes: {_amp_src} from block '{_amp_block}': {len(miller)} reflections, "
      f"{len(mil_u)} unique to {D_MIN_TRANSFORM} Å ({d_u.min():.3f}-{d_u.max():.1f} Å)")

# --- Model phases (complete deposited model, symmetrized over the space group) -------
dc = gemmi.DensityCalculatorX()
dc.d_min = D_MIN_MODEL_DENSITY; dc.rate = RATE_MODEL_DENSITY
_phase_model = st_full if PHASES_FROM_FULL_MODEL else st
dc.set_grid_cell_and_spacegroup(_phase_model)
dc.initialize_grid()
dc.add_model_density_to_grid(_phase_model[0])
dc.grid.symmetrize_sum()
_rho_model = np.array(dc.grid, dtype=np.float64)
nu_m, nv_m, nw_m = _rho_model.shape
F_calc_grid = np.fft.ifftn(_rho_model)          # (1/N) sum rho e^{+2 pi i h.x}
del dc, _rho_model
_Fc = lambda H: F_calc_grid[H[:, 0] % nu_m, H[:, 1] % nv_m, H[:, 2] % nw_m]
assert np.all(np.array([nu_m, nv_m, nw_m]) >= 2*np.abs(mil_u).max(0) + 1), "model grid too coarse for phase lookup"
F_hyb_u = Fm_u*np.exp(1j*np.angle(_Fc(mil_u)))

# --- Fill unmeasured reflections with scaled model amplitudes (as for 6o2h) ----------
# Bookkeeping by Laue orbit: one representative per set of equivalent reflections.
_dmax_meas = float(d_u.max())
_hbox = [int(np.floor(L/D_MIN_TRANSFORM)) for L in (cell.a, cell.b, cell.c)]
_Hall = np.stack(np.meshgrid(*[np.arange(-m, m + 1) for m in _hbox], indexing='ij'), -1).reshape(-1, 3)
_dall = 2*np.pi/np.maximum(np.linalg.norm(_Hall @ B_recip.T, axis=1), 1e-12)
_keep_all = (_dall >= D_MIN_TRANSFORM) & np.any(_Hall != 0, axis=1)
_Hall = _Hall[_keep_all]
_Hall = _Hall[~systematically_absent(_Hall)]
_kall  = np.unique(laue_orbit_key(_Hall))
_kmeas = np.unique(laue_orbit_key(mil_u))
_kmiss = np.setdiff1d(_kall, _kmeas)
_Hmiss = _decode_key(_kmiss)
_dmiss = 2*np.pi/np.linalg.norm(_Hmiss @ B_recip.T, axis=1)
_n_low_missing = int((_dmiss > _dmax_meas).sum())
_Hfill = _Hmiss[_dmiss <= _dmax_meas] if FILL_MISSING else np.zeros((0, 3), np.int64)
if len(_Hfill):
    assert np.all(np.array([nu_m, nv_m, nw_m]) >= 2*np.abs(_Hfill).max(0) + 1), "model grid too coarse for the fill"
    _s2m = 1.0/d_u**2
    _edges_k = np.quantile(_s2m, np.linspace(0, 1, 21))
    _bm = np.clip(np.digitize(_s2m, _edges_k) - 1, 0, 19)
    _Fcm = np.abs(_Fc(mil_u))
    _kbin = np.array([np.sum(Fm_u[_bm == b]*_Fcm[_bm == b])/max(np.sum(_Fcm[_bm == b]**2), 1e-300)
                      for b in range(20)])
    _bf = np.clip(np.digitize(np.sum((_Hfill @ B_recip.T)**2, 1)/(2*np.pi)**2, _edges_k) - 1, 0, 19)
    F_fill_u = _kbin[_bf]*_Fc(_Hfill)
    H_rep, F_rep = np.vstack([mil_u, _Hfill]), np.concatenate([F_hyb_u, F_fill_u])
else:
    H_rep, F_rep = mil_u, F_hyb_u
mil_all, F_all = expand_with_phases(H_rep, F_rep)
del F_calc_grid; gc.collect()
print(f"Unique reflections to {D_MIN_TRANSFORM} Å (absences excluded): {len(_kmeas)} measured of "
      f"{len(_kall)} possible ({100*len(_kmeas)/max(len(_kall), 1):.1f}%); {len(_Hfill)} filled"
      f"{'' if FILL_MISSING else ' (FILL_MISSING is off)'}; {_n_low_missing} lower-resolution than any "
      f"measured reflection (> {_dmax_meas:.1f} Å) left at zero")
_dorb = 2*np.pi/np.linalg.norm(_decode_key(_kall) @ B_recip.T, axis=1)
_dbins = np.quantile(_dorb, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
for _lo, _hi in zip(_dbins[:-1], _dbins[1:]):
    _na = np.sum((_dorb >= _lo) & (_dorb < _hi + 1e-9))
    _nm = np.sum((d_u >= _lo) & (d_u < _hi + 1e-9))
    print(f"   {_hi:6.2f}–{_lo:5.2f} Å: completeness {100*min(_nm/max(_na, 1), 1):5.1f}%")
print(f"expanded over the space group: {len(mil_all)} reflections in P1")
del _Hall, _dall, _keep_all

# --- The density grid ------------------------------------------------------------------
h_lim = np.abs(mil_all).max(axis=0)
n_tr = symmetry_compatible_sizes([max(RATE_TRANSFORM*L/D_MIN_TRANSFORM, 2*hm + 1)
                                  for L, hm in zip((cell.a, cell.b, cell.c), h_lim)])
nu, nv, nw = n_tr
N_GRID = np.array(n_tr)
spacing = np.array([cell.a/nu, cell.b/nv, cell.c/nw])
F_grid = np.zeros((nu, nv, nw), dtype=complex)
F_grid[mil_all[:, 0] % nu, mil_all[:, 1] % nv, mil_all[:, 2] % nw] = F_all/cell.volume
F_grid[(-mil_all[:, 0]) % nu, (-mil_all[:, 1]) % nv, (-mil_all[:, 2]) % nw] = F_all.conj()/cell.volume
rho = np.fft.fftn(F_grid).real                 # rho(r) = (1/V) sum_h F_h e^{-2 pi i h.r}
del F_grid
N_VOX = rho.size
dV  = cell.volume/N_VOX

# The map must have the space-group symmetry on this grid: compare rho with its image
# under every operation (exact up to rounding when the grid is compatible).
_ijk = np.array(np.unravel_index(np.random.default_rng(0).choice(N_VOX, 20000, replace=False), rho.shape)).T
for W, w in SYM_OPS[1:]:
    _img = np.rint((_ijk/N_GRID) @ W.T*N_GRID + w*N_GRID).astype(int) % N_GRID
    _dev = np.abs(rho[tuple(_img.T)] - rho[tuple(_ijk.T)]).max()/rho.std()
    assert _dev < 1e-6, f"hybrid map is not symmetric under an operation (max deviation {_dev:.1e} sd)"
print(f"\nTransform grid: {nu}x{nv}x{nw} = {N_VOX} voxels, spacing {spacing.round(3)} Å; "
      f"symmetric under all {N_OPS} operations")
print(f"Reflections used: {len(mil_all)} in P1, |h|max = {tuple(int(x) for x in h_lim)}")
print(f"Density RMS {rho.std():.4f} e⁻/Å³; cell integral {rho.sum()*dV:.3f} e⁻ (~0: F(000) unmeasured)")
n_low = int((d_u > 8.0).sum())
print(f"Unique reflections beyond 8 Å: {n_low} "
      f"({'SPARSE -- the envelope may be distorted' if n_low < 30 else 'adequate'})")

In [ ]:
# --- Partition the density among bodies and lattice images ---------------------------
# Every voxel goes to the (body, lattice image) whose nearest atom SURFACE is closest.
# The atom set is all bodies in all images within +-UNWRAP_IMAGE, so the result is a
# fundamental domain of the space group (one owner per voxel, symmetric because the
# atom set and the grid are). The per-voxel record is the owner, the distance to its
# surface, and the margin to the best competing owner (how contested the voxel is).
_IMGS = np.array(list(product(range(-UNWRAP_IMAGE, UNWRAP_IMAGE + 1), repeat=3)))
N_IMG = len(_IMGS)
IMG_ZERO = int(np.where((_IMGS == 0).all(1))[0][0])
_K_NEAR = 8

def _atom_cloud(imgs):
    pos, own, vdw = [], [], []
    for b in range(N_BODY):
        P = body_positions(b); v = CHAINS[BODIES[b]['chain']]['vdw']
        for k, n in enumerate(imgs):
            pos.append(P + A_orth @ n); own.append(np.full(len(P), b*len(imgs) + k, np.int32)); vdw.append(v)
    return cKDTree(np.vstack(pos)), np.concatenate(own), np.concatenate(vdw)

def _voxel_xyz(flat):
    i, j, k = np.unravel_index(flat, (nu, nv, nw))
    return np.column_stack([i/nu, j/nv, k/nw]) @ A_orth.T

def partition(flat_idx, imgs, tree, own, vdw, chunk=VOXEL_CHUNK):
    """Owner (body*len(imgs) + image), surface distance and contest margin per voxel."""
    n = len(flat_idx)
    owner = np.empty(n, np.int32); dsurf = np.empty(n, np.float32); margin = np.empty(n, np.float32)
    for s0 in range(0, n, chunk):
        sl = slice(s0, min(s0 + chunk, n))
        dc, ic = tree.query(_voxel_xyz(flat_idx[sl]), k=_K_NEAR, workers=-1)
        ds = dc - vdw[ic]
        o = own[ic]
        best = np.argmin(ds, axis=1); rows = np.arange(len(ds))
        owner[sl] = o[rows, best]; dsurf[sl] = ds[rows, best]
        other = np.where(o != o[rows, best][:, None], ds, np.inf).min(axis=1)
        margin[sl] = np.minimum(other - ds[rows, best], 99.0)
    return owner, dsurf, margin

_tree_img, _own_img, _vdw_img = _atom_cloud(_IMGS)
OWNER, D_SURF, MARGIN = partition(np.arange(N_VOX), _IMGS, _tree_img, _own_img, _vdw_img)
OWNER_BODY = OWNER // N_IMG
OWNER_IMG  = OWNER % N_IMG
del _tree_img, _own_img, _vdw_img; gc.collect()

# Sufficiency of UNWRAP_IMAGE: repeat on a random subset with one more shell of images
# and count voxels that would be claimed from beyond the original range.
_sub = np.sort(np.random.default_rng(1).choice(N_VOX, min(N_VOX, 300_000), replace=False))
_IMGS2 = np.array(list(product(range(-UNWRAP_IMAGE - 1, UNWRAP_IMAGE + 2), repeat=3)))
_t2, _o2, _v2 = _atom_cloud(_IMGS2)
_own2, _, _ = partition(_sub, _IMGS2, _t2, _o2, _v2)
del _t2, _o2, _v2
_beyond = float((np.abs(_IMGS2[_own2 % len(_IMGS2)]).max(axis=1) > UNWRAP_IMAGE).mean())
print(f"Partition of {N_VOX} voxels among {N_BODY} bodies x {N_IMG} lattice images done.")
print(f"Widening the image search to ±{UNWRAP_IMAGE + 1} claims {_beyond:.3%} of a random subset of voxels"
      + ("" if _beyond < 1e-3 else "  -- raise UNWRAP_IMAGE"))

# Symmetry of the partition: images of one chain must own the same number of voxels.
_cnt = np.bincount(OWNER_BODY, minlength=N_BODY)
print(f"\nvoxels owned per body (images of one chain must agree):")
for c in BODY_CHAINS:
    bs = [b for b in range(N_BODY) if BODIES[b]['chain'] == c]
    v = _cnt[bs]
    print(f"   chain {c}: {v.tolist()}   spread {(v.max() - v.min())/v.mean():.1e}")
    assert (v.max() - v.min()) <= 1e-3*v.mean() + 2, "the partition is not symmetric"

### Bulk Solvent and the Contrast Density of Each Body

As for 6o2h: $\rho_{\rm sol}$ is the median map value over voxels more than
`SOLVENT_PROBE + SOLVENT_SHELL` from any atom surface, $M$ is the tanh-softened
envelope with its edge `SOLVENT_PROBE` beyond the van der Waals surface, and
$\rho_{\rm eff}=(\rho-\rho_{\rm sol})M$. The partition then gives each body its
share $\rho_{{\rm eff},b}$.

The cell reports, per body, the contrast electrons $\int\rho_{{\rm eff},b}\,dV$;
all images of one chain must agree. It also reports how much of each chain's
contrast sits in **contested voxels**, where a different body (or lattice image)
has an atom surface within 0.5 Å of the owner's. That is the part of the density
whose assignment depends on the partition rule.

The profile figure has one row per asymmetric-unit chain, using only the voxels that
chain owns. For each asymmetric-unit chain the contrast density is stored on a box of grid
indices at the voxels' unwrapped positions (the cell index plus the lattice image
times the grid size), in four channels $\rho_{\rm eff}\,dV$ and
$\rho_{\rm eff}(\mathbf r-\mathbf r_{{\rm cm},c})\,dV$, the inputs of $F$ and $L$.

In [ ]:
# --- Flat bulk-solvent level, envelope, contrast ---------------------------------------
_deep = D_SURF > (SOLVENT_PROBE + SOLVENT_SHELL)
if _deep.sum() < 0.005*N_VOX:
    print(f"WARNING: only {_deep.sum()} deep-solvent voxels; falling back to the outermost 2%.")
    _deep = D_SURF > np.quantile(D_SURF, 0.98)
rho_flat = rho.ravel()
rho_sol_map = float(np.median(rho_flat[_deep]))
M_soft = (0.5*(1.0 - np.tanh((D_SURF - SOLVENT_PROBE)/SOLVENT_EDGE))).astype(np.float64)
V_mask = float(M_soft.sum()*dV)
rho_eff = (rho_flat - rho_sol_map)*M_soft
V_solv  = cell.volume - V_mask
rho_bar = (Z_model + rho_sol_map*V_solv)/V_mask
rho_sol_abs = rho_sol_map + rho_bar
Q_contrast = float(rho_eff.sum()*dV)
print(f"Deep-solvent voxels: {_deep.sum()} ({100*_deep.mean():.1f}% of the cell); "
      f"flat solvent level rho_sol = {rho_sol_map:+.4f} e⁻/Å³ (map offset)")
print(f"Envelope volume {V_mask:.0f} Å³ ({100*V_mask/cell.volume:.0f}% of the cell); "
      f"implied absolute solvent density {rho_sol_abs:.4f} e⁻/Å³ (water ≈ 0.335, no H in the bodies)")
print(f"Contrast electrons in the cell {Q_contrast:.0f} e⁻ (model {Z_model:.0f} e⁻; "
      f"flat-solvent identity -rho_sol V = {-rho_sol_map*cell.volume:.0f} e⁻)")

# --- per body -------------------------------------------------------------------------------
Q_body = np.bincount(OWNER_BODY, weights=rho_eff*dV, minlength=N_BODY)
print(f"\n{'body':>5s} {'chain':>6s} {'op':>3s} {'contrast e⁻':>12s}")
for b in range(N_BODY):
    print(f"{b:5d} {BODIES[b]['chain']:>6s} {BODIES[b]['op']:3d} {Q_body[b]:12.1f}")
for c in BODY_CHAINS:
    q = Q_body[[b for b in range(N_BODY) if BODIES[b]['chain'] == c]]
    assert np.ptp(q) <= 1e-3*abs(q.mean()) + 1e-6, "symmetry images carry different contrast"

print(f"\ncontested voxels (another body's surface within 0.5 Å of the owner's):")
for c in BODY_CHAINS:
    m = OWNER_BODY == ASU_BODY[c]
    w = np.abs(rho_eff[m]); hard = MARGIN[m] < 0.5
    print(f"   chain {c}: {100*hard.mean():5.2f}% of its voxels, {100*w[hard].sum()/w.sum():5.2f}% of its |contrast|")

# --- Boxes for the transform of each asymmetric-unit chain ----------------------------------
BOX = {}
_FLOOR_M = 1e-6          # envelope weight below which a voxel carries no contrast
for c in BODY_CHAINS:
    sel = np.where((OWNER_BODY == ASU_BODY[c]) & (M_soft > _FLOOR_M))[0]
    ijk = np.array(np.unravel_index(sel, (nu, nv, nw))).T
    I = ijk - _IMGS[OWNER_IMG[sel]]*N_GRID     # unwrapped: a voxel next to the image at +n sits at r - T_n
    w = rho_eff[sel]*dV
    r = (I/N_GRID) @ A_orth.T
    lever = r - CHAINS[c]['com']
    I0 = I.min(0); ext = I.max(0) - I0 + 1
    Nb = ext + (ext % 2)                                   # even box sizes
    Ic = I0 + Nb//2                                        # index of mode 0
    W4 = np.zeros((4,) + tuple(Nb), complex)
    loc = tuple((I - I0).T)
    W4[0][loc] = w
    for k in range(3):
        W4[k + 1][loc] = w*lever[:, k]
    BOX[c] = dict(W4=W4, Ic=Ic, N=Nb, I=I.astype(np.int32), w=w, lever=lever)
    cen = (r*w[:, None]).sum(0)/w.sum()
    # every unwrapped voxel must sit next to the chain itself (in cell 0), not next to an image
    _dmax = float(cKDTree(CHAINS[c]['pos']).query(r)[0].max())
    assert _dmax < SOLVENT_PROBE + 7*SOLVENT_EDGE + 2.5, \
        f"chain {c}: an unwrapped voxel lies {_dmax:.1f} Å from the chain; the unwrapping is wrong"
    print(f"chain {c}: {len(sel)} voxels on a {Nb[0]}x{Nb[1]}x{Nb[2]} box "
          f"({W4.nbytes/2**20:.0f} MB); contrast centroid {np.linalg.norm(cen - CHAINS[c]['com']):.2f} Å "
          f"from the centre of mass; extent {(ext*spacing).round(1)} Å; "
          f"farthest voxel {_dmax:.1f} Å from an atom")
del sel, ijk, I, w, r, lever; gc.collect()

# --- Profiles, one row per asymmetric-unit body (on a random subset of voxels) ---------------
# Each row uses only the voxels the partition gives to that chain, so D_SURF is the distance
# to that chain's own atom surface. The two chains share one solvent level and one envelope
# rule; differences between the rows come from the density itself.
_ps = np.random.default_rng(2).choice(N_VOX, min(N_VOX, 4_000_000), replace=False)
_edges = np.linspace(-1.5, 6.0, 60)
_cen = 0.5*(_edges[:-1] + _edges[1:])
_cols_ch = ['#2a78d6', '#eb6834', '#1baf7a', '#8e5bd0']
fig, axes = plt.subplots(N_CHAIN, 3, figsize=(16, 4.0*N_CHAIN), squeeze=False)
_neg = {}
for r_, c in enumerate(BODY_CHAINS):
    p = _ps[OWNER_BODY[_ps] == ASU_BODY[c]]
    ds = D_SURF[p]
    bi = np.clip(np.digitize(ds, _edges) - 1, 0, len(_cen) - 1)
    ok = (ds >= _edges[0]) & (ds <= _edges[-1])
    cnt = np.bincount(bi[ok], minlength=len(_cen))
    prof_raw = np.where(cnt > 20, np.bincount(bi[ok], weights=rho_flat[p][ok], minlength=len(_cen))/np.maximum(cnt, 1), np.nan)
    prof_eff = np.where(cnt > 20, np.bincount(bi[ok], weights=rho_eff[p][ok], minlength=len(_cen))/np.maximum(cnt, 1), np.nan)
    inside = M_soft[p] > 0.5
    _neg[c] = float((rho_eff[p][inside] < 0).mean())
    col = _cols_ch[r_ % len(_cols_ch)]
    ax = axes[r_]
    ax[0].plot(_cen, prof_raw, lw=1.8, color=col, label=r'hybrid map $\rho$')
    ax[0].axhline(rho_sol_map, color='crimson', ls='--', lw=1.2, label=r'flat solvent level $\rho_{sol}$')
    ax[0].axhline(0, color='0.7', lw=0.8); ax[0].axvline(SOLVENT_PROBE, color='k', ls=':', lw=1, label='envelope boundary')
    ax[0].set_ylabel(r'$\rho$ (e$^-$/Å$^3$)'); ax[0].set_title(f'chain {c}: density profile, protein → solvent')
    ax[0].legend(fontsize=7)
    ax[1].plot(_cen, prof_eff, lw=1.8, color=col, label=r'contrast $\rho_{eff}$')
    ax[1].fill_between(_cen, 0, np.where(prof_eff < 0, prof_eff, 0), color='crimson', alpha=0.3,
                       label='excluded-volume deficit')
    ax[1].axhline(0, color='k', lw=0.9); ax[1].axvline(SOLVENT_PROBE, color='k', ls=':', lw=1)
    ax[1].set_ylabel(r'$\rho_{eff}$ (e$^-$/Å$^3$)'); ax[1].set_title(f'chain {c}: what moves rigidly')
    ax[1].legend(fontsize=7)
    mg = MARGIN[p][inside]
    lone = float((mg >= 6).mean())            # no other body among the nearest atoms
    ax[2].hist(mg[mg < 6], bins=120, color=col)
    ax[2].axvline(0.5, color='crimson', ls='--', lw=1, label='contested (< 0.5 Å)')
    ax[2].set_yscale('log'); ax[2].set_ylabel('voxels')
    ax[2].set_title(f'chain {c}: margin to the competing body\n'
                    f'({100*lone:.0f}% of envelope voxels have no competing body nearby, not shown)', fontsize=10)
    ax[2].legend(fontsize=7)
    for a_ in ax[:2]:
        a_.set_xlabel(f'distance from the nearest atom surface of chain {c} (Å)')
    ax[2].set_xlabel('margin (Å), voxels inside the envelope')
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'solvent_contrast.png', dpi=150); plt.show()
print("negative-contrast fraction inside the envelope: " + ", ".join(f"chain {c} {v:.1%}" for c, v in _neg.items())
      + " (the excluded-volume shell)")

## Coupling Vector $G(\mathbf q)$ for Eight Bodies

For one body, a small rigid displacement $(\boldsymbol\Omega,\mathbf v)$ about its
centre of mass changes its transform by $\delta F=\mathbf g\cdot(\boldsymbol\Omega,\mathbf v)$
with

$$\mathbf g=\begin{pmatrix}i\,\mathbf q\times L(\mathbf q)\\-i\,\mathbf q\,F(\mathbf q)\end{pmatrix},\qquad
F=\int\rho_{\rm eff}e^{-i\mathbf q\cdot\mathbf r}d^3r,\quad
L=\int(\mathbf r-\mathbf r_{\rm cm})\rho_{\rm eff}e^{-i\mathbf q\cdot\mathbf r}d^3r,$$

exactly as in the P1 notebook, with $\mathbf r$ the absolute (unwrapped) position in
cell 0. The coupling vector of the cell is the $6N_{\rm body}$-vector
$G=(G_0,\dots,G_7)$, $G_b=\mathbf g_b^*$, and

$$I(\mathbf q)=G^\dagger D(\mathbf q)^{-1}G,$$

with $D$ the $48\times48$ dynamical matrix of the next sections.

**Symmetry.** Body $b=(s,c)$ has density
$\rho_{{\rm eff},b}(\mathbf r)=\rho_{{\rm eff},c}\bigl(R_b^{\mathsf T}(\mathbf r-\mathbf t_b)\bigr)$, so

$$F_b(\mathbf q)=e^{-i\mathbf q\cdot\mathbf t_b}F_c(R_b^{\mathsf T}\mathbf q),\qquad
L_b(\mathbf q)=e^{-i\mathbf q\cdot\mathbf t_b}R_bL_c(R_b^{\mathsf T}\mathbf q),\qquad
G_b(\mathbf q)=e^{+i\mathbf q\cdot\mathbf t_b}\,S_b\,G_c(R_b^{\mathsf T}\mathbf q),$$

with $S_b=\mathrm{diag}(R_b,R_b)$. In Miller indices $R_b^{\mathsf T}\mathbf q$ is
$\mathbf h\,W_s$ and $\mathbf q\cdot\mathbf t_b=2\pi\,\mathbf h\cdot(\mathbf w_s+\mathbf m_b)$.
Only the two asymmetric-unit chains are ever transformed: `G_at_hkl_batch` evaluates
chain $c$ once at the rotated indices of all of its images.

**Checks.**

1. The NUFFT against direct summation over the chain's voxels at fractional $hkl$.
2. At integer $hkl$, $\sum_bF_b$ (from the two chain transforms and the symmetry
   relation) against the FFT of the whole-cell contrast $(\rho-\rho_{\rm sol})M$.
   This tests the partition, the unwrapping and the symmetry bookkeeping at once.
3. $|F_{\rm contrast}|/|F_{\rm meas}|$ by resolution: near 1 at high resolution,
   below 1 at low resolution where the solvent subtraction removes amplitude.

In [ ]:
def F_L_chain(c, hkl, direct=False, chunk=None):
    """F and L of asymmetric-unit chain c at fractional indices hkl (N, 3), in the chain's
    own frame. Type-2 NUFFT of the chain's box (direct summation if direct=True)."""
    hkl = np.atleast_2d(np.asarray(hkl, float))
    B = BOX[c]
    if direct or not HAVE_FINUFFT:
        out = np.zeros((4, len(hkl)), complex)
        frac = B['I']/N_GRID
        chunk = chunk or max(1, int(2e8/(16*len(frac))))
        for s0 in range(0, len(hkl), chunk):
            ph = np.exp(-2j*np.pi*(hkl[s0:s0 + chunk] @ frac.T))          # (n, n_vox)
            out[0, s0:s0 + chunk] = ph @ B['w']
            out[1:, s0:s0 + chunk] = (ph @ (B['w'][:, None]*B['lever'])).T
    else:
        x = 2*np.pi*hkl/N_GRID
        x = (x + np.pi) % (2*np.pi) - np.pi
        out = finufft.nufft3d2(*[np.ascontiguousarray(x[:, k]) for k in range(3)], B["W4"], isign=-1,
                               eps=TRANSFORM_EPS, maxbatchsize=1)
        out = out*np.exp(-2j*np.pi*(hkl @ (B['Ic']/N_GRID)))[None, :]
    return out[0], out[1:].T

def G_at_hkl_batch(h_arr, k_arr, l_arr):
    """G(q) at fractional Miller indices: (N, 6*N_BODY) complex, G_b = conj(g_b)."""
    H = np.column_stack([np.asarray(h_arr, float), np.asarray(k_arr, float), np.asarray(l_arr, float)])
    N = len(H)
    G = np.empty((N, N_DOF), complex)
    for c in BODY_CHAINS:
        bs = [b for b in range(N_BODY) if BODIES[b]['chain'] == c]
        Hc = np.vstack([H @ BODIES[b]['W'] for b in bs])               # R_b^T q, in indices
        F, L = F_L_chain(c, Hc)
        qc = Hc @ B_recip.T
        g = np.concatenate([np.cross(1j*qc, L), -1j*qc*F[:, None]], axis=1)
        for k, b in enumerate(bs):
            ph = np.exp(-2j*np.pi*(H @ BODIES[b]['tau']))                # e^{-i q.t_b}
            G[:, 6*b:6*b + 6] = np.conj(ph[:, None]*(g[k*N:(k + 1)*N] @ S_of(BODIES[b]['R']).T))
    return G

def G_at_hkl(h, k, l):
    return G_at_hkl_batch([h], [k], [l])[0]

def F_total_hkl(H):
    """Contrast transform of the whole cell, sum_b F_b, at indices H (e^{-iq.r} convention)."""
    H = np.atleast_2d(np.asarray(H, float)); out = np.zeros(len(H), complex)
    N = len(H)
    for c in BODY_CHAINS:
        bs = [b for b in range(N_BODY) if BODIES[b]['chain'] == c]
        F, _ = F_L_chain(c, np.vstack([H @ BODIES[b]['W'] for b in bs]))
        for k, b in enumerate(bs):
            out += np.exp(-2j*np.pi*(H @ BODIES[b]['tau']))*F[k*N:(k + 1)*N]
    return out

# --- 1. NUFFT vs direct summation, fractional hkl ----------------------------------------------
_hf = np.array([[1 + 3/11, -2 + 5/11, 0.5], [7.31, 2.77, -1.13], [-15.2, 9.4, 3.71], [0.5, 0.25, 0.125]])
for c in BODY_CHAINS:
    Fn, Ln = F_L_chain(c, _hf); Fd, Ld = F_L_chain(c, _hf, direct=True)
    err = max(np.abs(Fn - Fd).max()/np.abs(Fd).max(), np.abs(Ln - Ld).max()/np.abs(Ld).max())
    print(f"chain {c}: NUFFT vs direct summation at fractional hkl, max rel. error {err:.1e}")
    assert err < 1e-6, "NUFFT transform disagrees with direct summation"

# --- 2. integer hkl: sum over bodies vs the whole-cell contrast ------------------------------
_rng_v = np.random.default_rng(0)
_sel = _rng_v.choice(len(mil_u), size=min(3000, len(mil_u)), replace=False)
_Hs = mil_u[_sel]
_Fcell = np.fft.fftn(rho_eff.reshape(nu, nv, nw))[_Hs[:, 0] % nu, _Hs[:, 1] % nv, _Hs[:, 2] % nw]*dV
_Fsum = F_total_hkl(_Hs)
_err2 = np.abs(_Fsum - _Fcell).max()/np.abs(_Fcell).max()
print(f"\nsum over the {N_BODY} bodies vs. FFT of the whole-cell contrast, {len(_Hs)} reflections: "
      f"max rel. error {_err2:.1e}")
assert _err2 < 1e-6, "the bodies' transforms do not add up to the cell's contrast transform"

# --- 3. contrast vs measured amplitudes by resolution -----------------------------------------
_ratio = np.abs(_Fsum)/Fm_u[_sel]
_dres = d_u[_sel]
_e = np.quantile(_dres, np.linspace(0, 1, 7))
print(f"\n|F_contrast| / |F_meas| at integer hkl by resolution:")
for _b in range(6):
    _m = (_dres >= _e[_b]) & (_dres <= _e[_b + 1])
    print(f"   {_e[_b]:6.2f}-{_e[_b + 1]:6.2f} Å ({_m.sum():5d}):  median {np.median(_ratio[_m]):.3f}")

_Gt = G_at_hkl(1 + 5/11, -2 + 3/11, 4/13)
assert np.abs(_Gt.imag).max() > 1e-6*np.abs(_Gt).max(), "G must be complex"
print(f"\nG at a fractional hkl: {N_DOF} complex components, |G| per body "
      f"{np.linalg.norm(_Gt.reshape(N_BODY, 6), axis=1).round(0)}")
print("OK: transform checks pass.")

## Rigid-Body Contacts and Their Symmetry Orbits

**Bonds.** Two bodies are in contact, exactly as in the P1 notebook, if at least
`MIN_CONTACTS` atom pairs lie within `CONTACT_CUTOFF` (hydrogens excluded, every
alternate conformation included). The search covers body $a$ in cell 0 against
every body $b$ in every cell $\mathbf n$ with $|n_i|\le$ `MAX_IMAGE`, and a second
search one shell further out confirms that nothing was missed. A bond is the
unordered pair $\{(a,\mathbf 0),(b,\mathbf n)\}$ up to a common lattice translation,
so each physical contact of the crystal is listed once per cell. A body touching its
own lattice translate, $(a,\mathbf 0)$–$(a,\mathbf n)$, is one bond, not two.

**Orbits.** An operation $g$ maps body $(s,c)$ in cell $\mathbf n$ to body
$(s',c)$ in cell $\mathbf m$, with $W_{s'}=W_gW_s$ and
$\mathbf m=W_g(\boldsymbol\tau_b+\mathbf n)+\mathbf w_g-\boldsymbol\tau_{b'}$, so it maps
bonds onto bonds. Bonds related in this way are one physical spring in different
frames. The bonds therefore fall into orbits, and each orbit gets **one** stiffness
matrix. Its representative is a bond with one end in the asymmetric unit: these are
the springs "from the asymmetric unit to other unit cells and asymmetric units", and
they hold every parameter of the model.

**Members.** Applying the four operations to the representative gives the members
of its orbit, each counted once per cell. Member $g$ joins body $g\cdot a$ to body
$g\cdot b$, has its contact point at $R_g\mathbf p+A\mathbf w_g$ (shifted by a lattice
vector so that its first body is in cell 0), and carries

$$K_g=S_g\,K\,S_g^{\mathsf T},\qquad S_g=\mathrm{diag}(R_g,R_g).$$

For P4₃ the four members of an orbit are distinct (the stabilizer is trivial), so an
orbit has $4$ members. If an operation mapped a bond onto itself, which needs a pure
rotation axis that P4₃ lacks, $K$ would be averaged over that stabilizer. The code
handles that case for other space groups.

**Checks.** Every image of a detected bond must itself be detected. The members of
an orbit must have the same number of atom pairs. For each member, the contact
point and the geometric prior built from its *own* atom pairs must equal the
transformed ones of the representative.

For each representative the code stores the midpoints of its atom pairs (weighted
by the product of occupancies), their centroid (the contact point $\mathbf p$), and
their gyration radius $\rho_g$. These define the contact frame and the geometric
prior, as in the P1 notebook.

In [ ]:
BODY_POS  = [body_positions(b) for b in range(N_BODY)]
BODY_OCC  = [CHAINS[BODIES[b]['chain']]['occ'] for b in range(N_BODY)]
BODY_TREE = [cKDTree(P) for P in BODY_POS]
BODY_RAD  = [float(np.linalg.norm(P - BODIES[b]['com'], axis=1).max()) for b, P in enumerate(BODY_POS)]

def canonical(n):
    n = tuple(int(x) for x in n)
    return min(n, tuple(-x for x in n))

def bond_key(a, b, n):
    """Canonical label of the bond (a, cell 0) -- (b, cell n), up to lattice translation."""
    n = tuple(int(x) for x in n)
    if a < b:
        return (a, b, n)
    if a > b:
        return (b, a, tuple(-x for x in n))
    return (a, a, canonical(n))

def _find_bonds(max_image):
    found = {}
    for a in range(N_BODY):
        for b in range(a, N_BODY):
            for n in product(range(-max_image, max_image + 1), repeat=3):
                if a == b and (n == (0, 0, 0) or canonical(n) != n):
                    continue
                Rn = A_orth @ np.array(n, float)
                if (np.linalg.norm(BODIES[b]['com'] + Rn - BODIES[a]['com'])
                        > BODY_RAD[a] + BODY_RAD[b] + CONTACT_CUTOFF):
                    continue
                lists = BODY_TREE[a].query_ball_point(BODY_POS[b] + Rn, CONTACT_CUTOFF)
                pairs = sorted((i, j) for j, l in enumerate(lists) for i in l)
                if len(pairs) >= MIN_CONTACTS:
                    pairs = np.array(pairs)
                    found[(a, b, n)] = dict(i=pairs[:, 0], j=pairs[:, 1])
    return found

BONDS = _find_bonds(MAX_IMAGE)
_outer = {k: v for k, v in _find_bonds(MAX_IMAGE + 1).items() if max(abs(x) for x in k[2]) > MAX_IMAGE}
if _outer:
    print(f"WARNING: {len(_outer)} bond(s) beyond MAX_IMAGE = {MAX_IMAGE}: {sorted(_outer)} -- increase MAX_IMAGE")
else:
    print(f"Checked shell {MAX_IMAGE + 1}: no additional contacts, MAX_IMAGE = {MAX_IMAGE} is sufficient.")

def bond_geometry(a, b, n, i, j):
    """Atom-pair midpoints (lab frame, body a in cell 0) and occupancy weights."""
    Rn = A_orth @ np.array(n, float)
    mid = 0.5*(BODY_POS[a][i] + BODY_POS[b][j] + Rn)
    return mid, BODY_OCC[a][i]*BODY_OCC[b][j]

# --- How an operation acts on bodies and bonds -----------------------------------------
_BODY_OF = {(B['chain'], tuple(np.round(B['W']).astype(int).ravel())): k for k, B in enumerate(BODIES)}

def map_body(g, b, n):
    """Operation g applied to body b in cell n: returns (body b', cell m)."""
    W, w = SYM_OPS[g]; B = BODIES[b]
    b2 = _BODY_OF[(B['chain'], tuple(np.round(W @ B['W']).astype(int).ravel()))]
    m = W @ (B['tau'] + np.asarray(n, float)) + w - BODIES[b2]['tau']
    mr = np.round(m)
    assert np.allclose(m, mr, atol=1e-6), "symmetry operation does not map bodies onto bodies"
    return b2, mr.astype(int)

def map_bond(g, a, b, n):
    """Image of the oriented bond (a, 0) -> (b, n): (a', b', n') with a' in cell 0, and the
    cell m1 into which a was moved."""
    a2, m1 = map_body(g, a, (0, 0, 0)); b2, m2 = map_body(g, b, n)
    return a2, b2, tuple(int(x) for x in (m2 - m1)), m1

# --- Orbits (union-find over the operations) --------------------------------------------
_parent = {k: k for k in BONDS}
def _root(k):
    while _parent[k] != k:
        _parent[k] = _parent[_parent[k]]; k = _parent[k]
    return k
for key in BONDS:
    for g in range(N_OPS):
        a2, b2, n2, _ = map_bond(g, *key)
        k2 = bond_key(a2, b2, n2)
        assert k2 in BONDS, f"image {k2} of bond {key} under operation {g} was not detected"
        assert len(BONDS[k2]['i']) == len(BONDS[key]['i']), "symmetry-related bonds differ in atom pairs"
        _parent[_root(k2)] = _root(key)
_orbits = {}
for key in BONDS:
    _orbits.setdefault(_root(key), []).append(key)

def _representative(keys):
    """Oriented bond (a, b, n) with a in the asymmetric unit; prefer an asymmetric-unit
    partner, then the smallest cell offset."""
    cand = []
    for (a, b, n) in keys:
        for (x, y, m) in ((a, b, n), (b, a, tuple(-v for v in n))):
            if x < N_CHAIN:
                cand.append((x, 0 if y < N_CHAIN else 1, sum(abs(v) for v in m), y, m))
    x, _, _, y, m = min(cand)
    return x, y, m

def _pairs_oriented(a, b, n):
    """Atom pairs of the oriented bond (a, 0) -> (b, n): indices into a's and b's atoms."""
    key = bond_key(a, b, n); P = BONDS[key]
    return (P['i'], P['j']) if key == (a, b, tuple(n)) else (P['j'], P['i'])

def contact_patch_stiffness(mid, wts, p, k_per_pair):
    """n isotropic point springs at the pair midpoints, referenced at p (lab axes)."""
    d = mid - p
    K = np.zeros((6, 6))
    K[3:6, 3:6] = k_per_pair*wts.sum()*np.eye(3)
    K[0:3, 0:3] = k_per_pair*(((d**2).sum(1)*wts).sum()*np.eye(3) - (d*wts[:, None]).T @ d)
    return K

unique, MEMBERS = {}, []
for o, keys in enumerate(sorted(_orbits.values(), key=lambda ks: min(ks))):
    a, b, n = _representative(keys)
    i, j = _pairs_oriented(a, b, n)
    mid, wts = bond_geometry(a, b, n, i, j)
    p = (mid*wts[:, None]).sum(0)/wts.sum()
    Rn = A_orth @ np.array(n, float)
    info = dict(a=a, b=b, n=n, R_n=BODIES[b]['com'] + Rn - BODIES[a]['com'], midpoints=mid, wts=wts,
                contact_pt=p, n_contacts=len(i), n_weighted=float(wts.sum()),
                rho_g=float(np.sqrt(((((mid - p)**2).sum(1))*wts).sum()/wts.sum())))
    info['label'] = (f"{BODIES[a]['chain']}–{BODIES[b]['chain']}"
                     f"{'' if BODIES[b]['op'] == 0 else '[s=' + str(BODIES[b]['op']) + ']'}"
                     f"{'' if n == (0, 0, 0) else str(n)}")
    seen, stab = {}, []
    for g in range(N_OPS):
        a2, b2, n2, m1 = map_bond(g, a, b, n)
        k2 = bond_key(a2, b2, n2)
        p2 = SYM_ROT_CART[g] @ p + A_orth @ (SYM_OPS[g][1] - m1)
        if k2 == bond_key(a, b, n):
            stab.append(g)
            assert np.allclose(p2, p, atol=1e-6) or np.allclose(
                (A_inv @ (p2 - p)) - np.round(A_inv @ (p2 - p)), 0, atol=1e-6), "stabilizer moves the contact point"
        if k2 in seen:
            continue
        seen[k2] = len(MEMBERS)
        MEMBERS.append(dict(orbit=o, op=g, a=a2, b=b2, n=n2, key=k2, p=p2,
                            Rn=A_orth @ np.array(n2, float)))
    info['stabilizer'] = stab
    info['members'] = [m for m in range(len(MEMBERS)) if MEMBERS[m]['orbit'] == o]
    unique[o] = info
shell_order = sorted(unique)
N_SHELL = len(shell_order)
assert len(MEMBERS) == len(BONDS), "every detected bond must be exactly one member of one orbit"

# --- Equivariance check: each member's own atom pairs reproduce the transformed geometry --
_kchk = []
for M in MEMBERS:
    info = unique[M['orbit']]
    i, j = _pairs_oriented(M['a'], M['b'], M['n'])
    mid, wts = bond_geometry(M['a'], M['b'], M['n'], i, j)
    p_own = (mid*wts[:, None]).sum(0)/wts.sum()
    S = S_of(SYM_ROT_CART[M['op']])
    K_own = contact_patch_stiffness(mid, wts, p_own, 1.0)
    K_rot = S @ contact_patch_stiffness(info['midpoints'], info['wts'], info['contact_pt'], 1.0) @ S.T
    _kchk.append(max(np.linalg.norm(p_own - M['p']), np.abs(K_own - K_rot).max()/np.abs(K_rot).max()))
print(f"member geometry vs. transformed representative: max deviation {max(_kchk):.1e}")
assert max(_kchk) < 1e-6, "a member's own contact geometry disagrees with its symmetry image"

print(f"\n{len(BONDS)} bonds per cell in {N_SHELL} symmetry orbits "
      f"(stabilizers: {sorted(set(len(unique[o]['stabilizer']) for o in shell_order))})")
print(f"{'orbit':>5s} {'representative':>18s} {'|R| Å':>7s} {'pairs':>6s} {'occ-wtd':>8s} "
      f"{'members':>8s} {'ρ_g Å':>6s}  type")
for o in shell_order:
    u = unique[o]
    kind = ('within the asymmetric unit' if (u['b'] < N_CHAIN and u['n'] == (0, 0, 0)) else
            'lattice translate of itself' if u['b'] == u['a'] else
            'translate of the other chain' if u['b'] < N_CHAIN else
            'symmetry mate')
    print(f"{o:5d} {u['label']:>18s} {np.linalg.norm(u['R_n']):7.2f} {u['n_contacts']:6d} "
          f"{u['n_weighted']:8.1f} {len(u['members']):8d} {u['rho_g']:6.2f}  {kind}")
_deg = np.bincount(np.concatenate([[M['a'], M['b']] for M in MEMBERS]), minlength=N_BODY)
print(f"\ncontacts per body (each bond counted at both ends): {_deg.tolist()}")

In [ ]:
# --- Contact network: the eight bodies and every member bond, coloured by orbit ---------
_cols = plt.cm.tab20(np.linspace(0, 1, max(N_SHELL, 2)))
fig, ax = plt.subplots(1, 2, figsize=(14, 6.5))
for k, (i0, i1, lab) in enumerate([(0, 1, ('a', 'b')), (0, 2, ('a', 'c'))]):
    corners = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], float)
    sq = np.zeros((5, 3)); sq[:, i0] = corners[:, 0]; sq[:, i1] = corners[:, 1]
    cc = sq @ A_orth.T
    ax[k].plot(cc[:, i0], cc[:, i1], color='0.5', lw=1)
    for M in MEMBERS:
        pa = BODIES[M['a']]['com']; pb = BODIES[M['b']]['com'] + M['Rn']
        ax[k].plot([pa[i0], M['p'][i0], pb[i0]], [pa[i1], M['p'][i1], pb[i1]], '-',
                   color=_cols[M['orbit']], lw=1.4, alpha=0.8)
    for b, B in enumerate(BODIES):
        mk = 'o' if B['chain'] == BODY_CHAINS[0] else 's'
        ax[k].scatter(B['com'][i0], B['com'][i1], s=120, marker=mk,
                      color='#2a78d6' if B['op'] == 0 else '0.75', edgecolor='k', zorder=5)
        ax[k].annotate(f"{B['chain']}{B['op']}", (B['com'][i0], B['com'][i1]), textcoords='offset points',
                       xytext=(6, 5), fontsize=8)
    ax[k].set_aspect('equal'); ax[k].set_xlabel(f'{"xyz"[i0]} (Å)'); ax[k].set_ylabel(f'{"xyz"[i1]} (Å)')
    ax[k].set_xlim(-25, A_orth[i0].sum() + 25); ax[k].set_ylim(-25, A_orth[i1].sum() + 25)
ax[0].set_title('Bodies (blue: asymmetric unit) and contact bonds, projected on ab')
ax[1].set_title('Projected on ac')
ax[1].legend(handles=[Line2D([0], [0], color=_cols[o], lw=2, label=f"{o}: {unique[o]['label']}")
                      for o in shell_order], fontsize=7, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'contact_network.png', dpi=150); plt.show()
print("Each bond is drawn from the first body's centre of mass through the contact point to the")
print("partner's; bonds of one colour are one orbit, i.e. one stiffness matrix seen in four frames.")

## Reduced Units

Stiffnesses are in units of $k_BT$, with rotational entries in $k_BT/\mathrm{rad}^2$,
translational entries in $k_BT/\mathrm{Å}^2$ and cross entries in
$k_BT/(\mathrm{rad}\cdot\mathrm{Å})$. In the classical limit
$\langle u_{\mathbf q}u_{\mathbf q}^\dagger\rangle=k_BT\,D(\mathbf q)^{-1}$ does not
involve the masses, so $M$ affects only the plotted frequencies.

In [ ]:
for c in BODY_CHAINS:
    print(f"chain {c}: mass {CHAINS[c]['m']:.0f} amu, inertia eigenvalues "
          f"{np.linalg.eigvalsh(CHAINS[c]['J']).round(0)} amu·Å²")
hbar_over_kT_THz = const.hbar*2*np.pi*1e12/(const.k*TEMPERATURE)
print(f"Classical-limit check: ħω/k_BT = {hbar_over_kT_THz:.4f} × (ω in THz); "
      f"{0.4*hbar_over_kT_THz:.3f} at 0.4 THz, {2*hbar_over_kT_THz:.3f} at 2 THz")

## Contact Frames, the Geometric Prior, and the Stiffness Parameterization

Each orbit's stiffness is parameterized at its representative bond, exactly as one
contact of the P1 notebook: $K_{\rm local}=LL^{\mathsf T}$ (21 free entries) in a frame
with its origin at the contact point $\mathbf p$ and its $z$ axis along the
centre-of-mass separation $\mathbf R_{ab}=\mathbf r_{{\rm cm},b}+\mathbf R_{\mathbf n}-\mathbf r_{{\rm cm},a}$.
In lab axes, still at $\mathbf p$, $K=\Gamma K_{\rm local}\Gamma^{\mathsf T}$, and member
$g$ of the orbit carries $K_g=S_gKS_g^{\mathsf T}$ at its own contact point. The
parameter vector therefore has $21N_{\rm orbit}$ entries, one block per orbit, and
every $\theta$ gives a crystal with the full P4₃ symmetry.

**The prior** for each orbit is the P1 one: $n$ isotropic point springs of stiffness
$k$ at the atom-pair midpoints $\mathbf m_i$, here weighted by the product of the two
atoms' occupancies $w_i$, with $\mathbf d_i=\mathbf m_i-\mathbf p$:

$$K_{TT}=k\Bigl(\sum_iw_i\Bigr)I_3,\qquad
K_{RR}=k\sum_iw_i\bigl(|\mathbf d_i|^2I-\mathbf d_i\mathbf d_i^{\mathsf T}\bigr),\qquad K_{TR}=0$$

($K_{TR}$ vanishes because $\mathbf p$ is the weighted centroid), plus a ridge of
$10^{-3}$ of the mean diagonal. Because the members' own atom pairs are images of the
representative's, the prior has the crystal symmetry as well (checked above).

**The regularizer** is the affine-invariant geodesic distance to the prior, summed
over orbits:
$\lambda\sum_o\|\log(K_{o,\rm prior}^{-1/2}K_oK_{o,\rm prior}^{-1/2})\|_F^2$. It is
invariant under $K\to TKT^{\mathsf T}$, so it does not matter in which frame or at
which member of the orbit it is evaluated. An orbit appears once in the penalty,
even though it appears four times in $D$.

In [ ]:
def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def ad_translation(R):
    """Moves a rigid displacement (Omega, v) referenced at x to the reference point x + R."""
    A = np.eye(6); A[3:6, 0:3] = -skew(R); return A

def contact_frame(R_n):
    """Orthonormal frame with z along R_n."""
    ez = R_n/np.linalg.norm(R_n)
    perp = np.array([1., 0., 0.]) if abs(ez[0]) < 0.9 else np.array([0., 1., 0.])
    ex = np.cross(perp, ez); ex /= np.linalg.norm(ex)
    return np.column_stack([ex, np.cross(ez, ex), ez])

for o in shell_order:
    info = unique[o]
    info['R_frame'] = contact_frame(info['R_n'])
    info['Gamma'] = S_of(info['R_frame'])
    # stabilizer average: K_rep(lab) = mean_h S_h Gamma K_local Gamma^T S_h^T
    info['Q_rep'] = [S_of(SYM_ROT_CART[h]) @ info['Gamma'] for h in info['stabilizer']]
for M in MEMBERS:
    info = unique[M['orbit']]
    Sg = S_of(SYM_ROT_CART[M['op']])
    M['Q'] = [Sg @ Q for Q in info['Q_rep']]                 # K_m = mean_Q Q K_local Q^T
    M['Ta'] = ad_translation(M['p'] - BODIES[M['a']]['com'])
    M['Tb'] = ad_translation(M['p'] - (BODIES[M['b']]['com'] + M['Rn']))

TRIL_I, TRIL_J = np.tril_indices(6)
N_TRIL   = len(TRIL_I)
N_PARAMS = N_SHELL*N_TRIL

def theta_to_Lmap(theta):
    Lmap, idx = {}, 0
    for c in shell_order:
        L = np.zeros((6, 6)); L[TRIL_I, TRIL_J] = theta[idx:idx + N_TRIL]
        Lmap[c] = L; idx += N_TRIL
    return Lmap

def Lmap_to_theta(Lmap):
    theta, idx = np.zeros(N_PARAMS), 0
    for c in shell_order:
        theta[idx:idx + N_TRIL] = Lmap[c][TRIL_I, TRIL_J]; idx += N_TRIL
    return theta

def build_K(Lmap):
    """Cholesky factors -> (K in each orbit's local frame, {orbit: K}; K of every MEMBER
    in lab axes at its own contact point, a list in MEMBERS order -- what D(q) uses)."""
    K_local = {c: Lmap[c] @ Lmap[c].T for c in shell_order}
    K_lab = [sum(Q @ K_local[M['orbit']] @ Q.T for Q in M['Q'])/len(M['Q']) for M in MEMBERS]
    return K_local, K_lab

def prior_Lmap(k_per_pair=PRIOR_K_PER_PAIR, ridge_frac=1e-3):
    Lmap, K_prior_local = {}, {}
    for c in shell_order:
        info = unique[c]; Gm = info['Gamma']
        Kl = Gm.T @ contact_patch_stiffness(info['midpoints'], info['wts'], info['contact_pt'], k_per_pair) @ Gm
        Kl = 0.5*(Kl + Kl.T); Kl += ridge_frac*np.trace(Kl)/6*np.eye(6)
        K_prior_local[c] = Kl; Lmap[c] = np.linalg.cholesky(Kl)
    return Lmap, K_prior_local

L_PRIOR, K_PRIOR_LOCAL = prior_Lmap()
_, K_LAB_PRIOR = build_K(L_PRIOR)

# --- Regularizer: geodesic distance to the geometric prior, per orbit -------------------
def _sqrtm_inv_sym(A):
    w, V = np.linalg.eigh(0.5*(A + A.T))
    w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
    return (V*w**-0.5) @ V.T

REG_PISQRT = {c: _sqrtm_inv_sym(K_PRIOR_LOCAL[c]) for c in shell_order}

def _affine_core(K, Minv):
    P = Minv @ K @ Minv; P = 0.5*(P + P.T)
    w, U = np.linalg.eigh(P)
    w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
    return w, U, np.log(w)

# lam is looked up at CALL time (see the P1 notebook: the lambda scan rebinds LAMBDA_PRIOR)
def regularizer_prior(K_local, lam=None):
    lam = LAMBDA_PRIOR if lam is None else lam
    return lam*sum(np.sum(_affine_core(K_local[c], REG_PISQRT[c])[2]**2) for c in K_local)

def regularizer_prior_dK(K_local, lam=None):
    lam = LAMBDA_PRIOR if lam is None else lam
    out = {}
    for c in K_local:
        Minv = REG_PISQRT[c]
        w, U, lg = _affine_core(K_local[c], Minv)
        out[c] = 2*lam*(Minv @ ((U*(lg/w)) @ U.T) @ Minv)
    return out

_c0 = shell_order[0]; _Kp0 = K_PRIOR_LOCAL[_c0]
_pR = _Kp0.copy(); _pR[0:3, 0:3] *= 1.10
_pT = _Kp0.copy(); _pT[3:6, 3:6] *= 1.10
_aR, _aT = regularizer_prior({_c0: _pR}, lam=1.0), regularizer_prior({_c0: _pT}, lam=1.0)
print(f"Geodesic penalty for a 10% change of each block of orbit {_c0}: "
      f"rotational {_aR:.4e}, translational {_aT:.4e}, ratio {_aR/_aT:.3f}")
print(f"Free parameters: {N_SHELL} orbits × {N_TRIL} Cholesky entries = {N_PARAMS}  "
      f"(for {len(MEMBERS)} bonds per cell)\n")
print(f"Geometric prior at k_per_pair = {PRIOR_K_PER_PAIR} k_BT/Å²:")
for c in shell_order:
    Kl = K_PRIOR_LOCAL[c]
    kt, kr = np.trace(Kl[3:6, 3:6])/3, np.trace(Kl[0:3, 0:3])/3
    print(f"  {c:3d} {unique[c]['label']:>18s}  κ_T={kt:8.2f} k_BT/Å²   κ_R={kr:9.1f} k_BT/rad²   "
          f"ratio={kr/kt:6.1f} Å²")

## Dynamical Matrix

Member $m$ joins body $a$ (cell 0) to body $b$ (cell $\mathbf n$) at contact point
$\mathbf p_m$. Both bodies' displacements are referred to $\mathbf p_m$ with
$T_a=A(\mathbf p_m-\mathbf r_{{\rm cm},a})$ and
$T_b=A(\mathbf p_m-\mathbf r_{{\rm cm},b}-\mathbf R_{\mathbf n})$, where
$A(\mathbf R)=\begin{pmatrix}I&0\\-[\mathbf R]_\times&I\end{pmatrix}$. A Bloch wave
$u_{b,\mathbf n}=e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}u_b$ stretches the spring by

$$\Delta_m=T_au_a-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}T_bu_b,$$

so member $m$ adds $T_a^{\mathsf T}K_mT_a$ to block $(a,a)$, $T_b^{\mathsf T}K_mT_b$ to
$(b,b)$, $-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}T_a^{\mathsf T}K_mT_b$ to $(a,b)$ and its
Hermitian conjugate to $(b,a)$. With one body and $\mathbf p$ at the partner's centre
of mass this is the P1 form $M_{\mathbf n}^\dagger K_{\mathbf n}M_{\mathbf n}$. The sum runs
over all members of all orbits, i.e. over every bond of the cell once.

**Checks.**

* Exactly 3 zero modes at Γ (uniform translation of every body), Hermiticity and
  positive semidefiniteness at a generic $\mathbf q$.
* **Supercell.** On a $3\times3\times3$ periodic supercell the real-space Hessian is
  built bond by bond, and its pseudo-inverse is the exact displacement covariance of
  that finite crystal. The diffuse intensity computed from it at the commensurate
  $\mathbf q$ must equal $G^\dagger D^{-1}G$. The thirds of a reciprocal-lattice
  vector give complex phases, so this tests the phase conventions of $D$ and of $G$
  together, which a real-valued check at the zone boundary would not.

In [ ]:
def dynamical_matrix_batch(q_batch, unique_unused, K_lab):
    """D(q) (N, 6 N_BODY, 6 N_BODY) complex Hermitian, summed over every member bond.
    (The second argument is kept for the P1 call signature.)"""
    q_batch = np.atleast_2d(np.asarray(q_batch, float))
    N = len(q_batch)
    C0 = np.zeros((N_DOF, N_DOF))
    D = np.zeros((N, N_DOF, N_DOF), complex)
    for M, K in zip(MEMBERS, K_lab):
        a, b, Ta, Tb = M['a'], M['b'], M['Ta'], M['Tb']
        sa, sb = slice(6*a, 6*a + 6), slice(6*b, 6*b + 6)
        C0[sa, sa] += Ta.T @ K @ Ta
        C0[sb, sb] += Tb.T @ K @ Tb
        Bab = Ta.T @ K @ Tb
        ph = np.exp(1j*(q_batch @ M['Rn']))
        D[:, sa, sb] -= ph[:, None, None]*Bab
        D[:, sb, sa] -= np.conj(ph)[:, None, None]*Bab.T
    D += 0.5*(C0 + C0.T)[None]
    return D

def dynamical_matrix(q_cart, unique_unused, K_lab):
    return dynamical_matrix_batch(np.asarray(q_cart, float)[None, :], None, K_lab)[0]

def mass_weighted_eigs(D_batch):
    """Eigenvalues of the pencil (D, M), via the Cholesky congruence (complex Hermitian)."""
    Dw = Msq_inv[None] @ D_batch @ Msq_inv.T[None]
    return np.linalg.eigvalsh(Dw)

D0  = dynamical_matrix(np.zeros(3), unique, K_LAB_PRIOR)
ev0 = mass_weighted_eigs(D0[None])[0]
print(f"D(q=0): {np.sum(np.abs(ev0) < 1e-8*abs(ev0).max())} zero eigenvalues of {N_DOF}; "
      f"smallest non-zero {np.sort(np.abs(ev0))[3]:.3e}, largest {ev0.max():.3e}")
assert np.sum(np.abs(ev0) < 1e-8*abs(ev0).max()) == 3, "expected exactly 3 zero modes at Gamma"
_qc = B_recip @ np.array([0.31, -0.17, 0.43])
_Dc = dynamical_matrix(_qc, unique, K_LAB_PRIOR)
print(f"generic q: Hermitian {np.allclose(_Dc, _Dc.conj().T)}, min eigenvalue {np.linalg.eigvalsh(_Dc).min():.3e} "
      f"(must be >= 0), ||Im D||/||Re D|| = {np.linalg.norm(_Dc.imag)/np.linalg.norm(_Dc.real):.3f}")
assert np.linalg.eigvalsh(_Dc).min() > -1e-9*np.abs(_Dc).max()

def supercell_hessian(K_lab, L=3):
    """Real-space Hessian of an L x L x L periodic supercell (cells x bodies x 6)."""
    cells = list(product(range(L), repeat=3)); idx = {c: k for k, c in enumerate(cells)}
    H = np.zeros((len(cells)*N_DOF, len(cells)*N_DOF))
    for M, K in zip(MEMBERS, K_lab):
        Ta, Tb = M['Ta'], M['Tb']
        Kaa, Kbb, Kab = Ta.T @ K @ Ta, Tb.T @ K @ Tb, Ta.T @ K @ Tb
        for c in cells:
            c2 = tuple((np.array(c) + np.array(M['n'])) % L)
            ia = idx[c]*N_DOF + 6*M['a']; ib = idx[c2]*N_DOF + 6*M['b']
            H[ia:ia + 6, ia:ia + 6] += Kaa; H[ib:ib + 6, ib:ib + 6] += Kbb
            H[ia:ia + 6, ib:ib + 6] -= Kab; H[ib:ib + 6, ia:ia + 6] -= Kab.T
    return H, cells

## Phonon Band Structure

The 48 branches along Γ–X–M–Γ–Z–R–A–Z of the tetragonal zone
(X $=(\tfrac12,0,0)$, M $=(\tfrac12,\tfrac12,0)$, Z $=(0,0,\tfrac12)$,
R $=(\tfrac12,0,\tfrac12)$, A $=(\tfrac12,\tfrac12,\tfrac12)$), from the mass-weighted
dynamical matrix at the geometric prior. Three acoustic branches start at zero;
the other 45 are optic: librations, and the relative translations of the eight
bodies.

In [ ]:
HSP = {'Γ': np.array([0., 0., 0.]), 'X': np.array([.5, 0., 0.]), 'M': np.array([.5, .5, 0.]),
       'Z': np.array([0., 0., .5]), 'R': np.array([.5, 0., .5]), 'A': np.array([.5, .5, .5])}
path_labels = ['Γ', 'X', 'M', 'Γ', 'Z', 'R', 'A', 'Z']
N_seg = 40
q_frac, tick_idx = [], [0]
for seg in range(len(path_labels) - 1):
    p0, p1 = HSP[path_labels[seg]], HSP[path_labels[seg + 1]]
    last = (seg == len(path_labels) - 2)
    for t in np.linspace(0, 1, N_seg, endpoint=last):
        q_frac.append(p0*(1 - t) + p1*t)
    if not last: tick_idx.append(len(q_frac))
tick_idx.append(len(q_frac) - 1)
q_frac = np.array(q_frac)
q_cart = (B_recip @ q_frac.T).T
x = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(q_cart, axis=0), axis=1))])

def bandstructure_freqs(K_lab, qs=None):
    qs = q_cart if qs is None else qs
    ev = mass_weighted_eigs(dynamical_matrix_batch(qs, unique, K_lab))
    return np.sqrt(np.maximum(ev, 0))*freq_unit

freqs_bs = bandstructure_freqs(K_LAB_PRIOR)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, freqs_bs[:, 3:], color='steelblue', lw=0.8)
ax.plot(x, freqs_bs[:, :3], color='crimson', lw=1.4)
for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0], x[-1]); ax.set_ylim(0)
ax.set_title(f'Rigid-body phonons — {PDB_ID} ({SG.hm}, {N_BODY} bodies), geometric prior '
             f'(red: three lowest branches)')
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'band_structure.png', dpi=150); plt.show()
print(f"Γ: 3 acoustic zeros; lowest optic frequencies {freqs_bs[0][3:9].round(4)} THz; "
      f"highest {freqs_bs.max():.3f} THz")

## Diffuse Scattering Maps

$I(\mathbf q)=G^\dagger D^{-1}G$ on the $(h,k)$ planes at $l=0$ and $l=0.5$, sampled
every $1/$`NPLOT_HK` in $h$ and $k$ and cut off at $|\mathbf q|<2\pi/$`D_MIN_DIFFUSE`.
$D$ is periodic in the reciprocal lattice, so at every Bragg position it has 3 zero
eigenvalues and `diffuse_intensity_batch` uses a pseudo-inverse there. Just off a
Bragg position the acoustic eigenvalues grow as $|\delta\mathbf q|^2$, giving the halos.
Their brightness follows $|\mathbf q\,F_{\rm cell}(\mathbf G)|^2$ with
$F_{\rm cell}=\sum_bF_b$, so the systematically absent $00l$ ($l\neq4n$) have no halo.

Before the maps, the cell checks:

* the eigendecomposition path against the plain linear solve used by the objective;
* **Laue symmetry**: $I(\mathbf hW_s)=I(\mathbf h)$ for every operation and
  $I(-\mathbf h)=I(\mathbf h)$, at random fractional $\mathbf h$, which tests the
  symmetry of $K$ (the members) and of $G$ together;
* the **supercell** check described above.

In [ ]:
BRAGG_EIGH_RADIUS = 1e-4   # Å⁻¹: closer than this to a Bragg point, use the pseudo-inverse

def diffuse_intensity_batch(K_lab, h_arr, k_arr, l_arr, chunk=None, min_eig_frac=1e-6, G_arr=None):
    """I(q) = G^dag D(q)^+ G, vectorized, full complex G and D.

    D is singular exactly at reciprocal-lattice points, so points within
    BRAGG_EIGH_RADIUS of one use an eigendecomposition with a pseudo-inverse; every other
    point uses a plain linear solve, which is about ten times faster for a 48 x 48 D."""
    chunk = chunk or CHUNK_EVAL
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float); l_arr = np.asarray(l_arr, float)
    N = len(h_arr); I_out = np.zeros(N)
    for s in range(0, N, chunk):
        sl = slice(s, min(s + chunk, N))
        hb = np.column_stack([h_arr[sl], k_arr[sl], l_arr[sl]])
        qb = hb @ B_recip.T
        Db = dynamical_matrix_batch(qb, unique, K_lab)
        Gb = G_arr[sl] if G_arr is not None else G_at_hkl_batch(*hb.T)
        near = np.linalg.norm((hb - np.round(hb)) @ B_recip.T, axis=1) < BRAGG_EIGH_RADIUS
        out = np.zeros(len(hb))
        if (~near).any():
            X = np.linalg.solve(Db[~near], Gb[~near][:, :, None])[:, :, 0]
            out[~near] = np.real(np.sum(np.conj(Gb[~near])*X, axis=1))
        if near.any():
            ev, evec = np.linalg.eigh(Db[near])
            emax = np.clip(ev.max(axis=1, keepdims=True), 1e-15, None)
            pos = ev > (min_eig_frac*emax)
            ev_inv = np.where(pos, 1.0/np.where(pos, ev, 1.0), 0.0)
            proj = np.einsum('nji,nj->ni', evec.conj(), Gb[near])
            out[near] = np.einsum('ni,ni->n', np.abs(proj)**2, ev_inv)
        I_out[sl] = out
    return I_out

def forward_I(K_lab, h, k, l, G=None):
    """Single-point I(q) by a plain solve (away from Bragg points)."""
    q = B_recip @ np.array([h, k, l], float)
    G = G_at_hkl(h, k, l) if G is None else G
    return float(np.real(np.conj(G) @ np.linalg.solve(dynamical_matrix(q, unique, K_lab), G)))

_hq = np.array([1.37, -2.13, 0.61, 3.05, -1.44, 7.21])
_kq = np.array([0.22, 1.81, -2.4, 0.13, 2.77, -4.6])
_lq = np.array([0.45, -0.33, 1.12, -2.06, 0.88, 0.3])
# (G is evaluated once per batch of points here: each transform call has a fixed cost)
_Gq = G_at_hkl_batch(_hq, _kq, _lq)
_Ia = diffuse_intensity_batch(K_LAB_PRIOR, _hq, _kq, _lq, G_arr=_Gq)
_saved_r, BRAGG_EIGH_RADIUS = BRAGG_EIGH_RADIUS, np.inf          # force the eigendecomposition path
_Ie = diffuse_intensity_batch(K_LAB_PRIOR, _hq, _kq, _lq, G_arr=_Gq)
BRAGG_EIGH_RADIUS = _saved_r
_Ib = np.array([forward_I(K_LAB_PRIOR, h, k, l, G=g) for h, k, l, g in zip(_hq, _kq, _lq, _Gq)])
_rel_e = max(np.max(np.abs(_Ia - _Ib)/np.abs(_Ib)), np.max(np.abs(_Ie - _Ib)/np.abs(_Ib)))
print(f"batched solve, batched eigendecomposition and single-point solve: max rel. difference {_rel_e:.2e}")
assert _rel_e < 1e-8

# --- Laue symmetry of the predicted intensity -------------------------------------------
_hr = np.random.default_rng(4).uniform(-12, 12, (12, 3)); _hr[:, 2] /= 3
_Himg = np.vstack([sg*(_hr @ W) for W, _ in SYM_OPS for sg in (1, -1)])     # first block: _hr itself
_Iimg = diffuse_intensity_batch(K_LAB_PRIOR, *_Himg.T).reshape(2*N_OPS, len(_hr))
_I0 = _Iimg[0]
_lerr = float(np.max(np.abs(_Iimg - _I0[None])/_I0[None]))
print(f"Laue symmetry: max |I(±h W_s) - I(h)|/I over {N_OPS} operations and Friedel mates: {_lerr:.1e}")
assert _lerr < 1e-7, "the predicted diffuse intensity does not have the Laue symmetry"

# --- Supercell check ------------------------------------------------------------------------
_L = 3
_Hsc, _cells = supercell_hessian(K_LAB_PRIOR, _L)
_w, _V = np.linalg.eigh(_Hsc)
_nz = int(np.sum(_w < 1e-9*_w.max()))
_Csc = (_V*np.where(_w > 1e-9*_w.max(), 1.0/np.where(_w > 1e-9*_w.max(), _w, 1.0), 0.0)) @ _V.T
_Rc = np.array(_cells, float) @ A_orth.T
_hsc = np.array([np.array(_G0, float) + np.array(_k)/_L for _G0 in ([2, -1, 1], [5, 3, -2], [-7, 4, 0])
                 for _k in [(1, 0, 0), (1, 2, 0), (2, 1, 1), (0, 1, 2), (1, 1, 1)]])
_Gsc = G_at_hkl_batch(*_hsc.T)
_Ibl = diffuse_intensity_batch(K_LAB_PRIOR, *_hsc.T, G_arr=_Gsc)
_errs_sc = []
for _h, _Gv, _I_bl in zip(_hsc, _Gsc, _Ibl):
    _qv = B_recip @ _h
    _v = np.concatenate([np.exp(-1j*(_qv @ R))*np.conj(_Gv) for R in _Rc])
    _I_sc = np.real(_v @ _Csc @ np.conj(_v))/len(_cells)
    _errs_sc.append(abs(_I_sc - _I_bl)/_I_bl)
print(f"{_L}x{_L}x{_L} periodic supercell ({len(_cells)*N_DOF} degrees of freedom, {_nz} zero modes): "
      f"real-space covariance vs. Bloch G^†D^-1G at {len(_errs_sc)} commensurate q, max rel. error "
      f"{max(_errs_sc):.1e}")
assert _nz == 3 and max(_errs_sc) < 1e-7, "supercell and Bloch-wave intensities disagree"
del _Hsc, _V, _Csc; gc.collect()
print("OK: both intensity paths agree, the intensity has the Laue symmetry, and the Bloch")
print("formula reproduces the exact covariance of a finite periodic crystal.")

In [ ]:
P = NPLOT_HK

def compute_diffuse_map(K_lab, L_layer=0, G_all=None):
    """I(h, k, L_layer) on the 1/NPLOT_HK grid inside the resolution sphere. Returns the
    grid, the map and the G of the in-sphere points (reusable for other K)."""
    h_max = int(np.ceil(cell.a/D_MIN_DIFFUSE)) + 1
    k_max = int(np.ceil(cell.b/D_MIN_DIFFUSE)) + 1
    q_cut = 2*np.pi/D_MIN_DIFFUSE
    h_pts = np.arange(-h_max*P, h_max*P + 1)/P
    k_pts = np.arange(-k_max*P, k_max*P + 1)/P
    HH, KK = np.meshgrid(h_pts, k_pts, indexing='ij')
    hf, kf = HH.ravel(), KK.ravel(); lf = np.full_like(hf, float(L_layer))
    qn = np.linalg.norm(np.column_stack([hf, kf, lf]) @ B_recip.T, axis=1)
    keep = (qn > 1e-3) & (qn < q_cut)
    if G_all is None:
        G_all = G_at_hkl_batch(hf[keep], kf[keep], lf[keep])
    I_flat = np.zeros(len(hf))
    I_flat[keep] = diffuse_intensity_batch(K_lab, hf[keep], kf[keep], lf[keep], G_arr=G_all)
    return h_pts, k_pts, I_flat.reshape(HH.shape), h_max, k_max, G_all

def plot_diffuse(h_pts, k_pts, I_map, h_max, k_max, L_layer, fname):
    pos_vals = I_map[I_map > 0]
    vmax = np.percentile(pos_vals, 97)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow((np.log1p(np.clip(I_map, 0, vmax))/np.log(10)).T, origin='lower', cmap='inferno',
                   extent=[h_pts[0], h_pts[-1], k_pts[0], k_pts[-1]], aspect='equal')
    plt.colorbar(im, ax=ax, label=r'$\log_{10}(1+I)$  [$k_BT$ units]')
    if float(L_layer).is_integer():
        _g = np.array([(hi, ki) for hi in range(-h_max, h_max + 1) for ki in range(-k_max, k_max + 1)])
        _g = _g[np.linalg.norm(np.column_stack([_g, np.full(len(_g), float(L_layer))]) @ B_recip.T, axis=1)
                < 2*np.pi/D_MIN_DIFFUSE]
        ax.scatter(_g[:, 0], _g[:, 1], marker='+', s=5, color='white', alpha=0.25, linewidths=0.4)
    ax.set_xlabel('h'); ax.set_ylabel('k')
    ax.set_title(f'One-phonon diffuse scattering I(h,k,{L_layer}), log scale\n'
                 f'{PDB_ID}, {N_BODY} rigid bodies, d_min = {D_MIN_DIFFUSE} Å')
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

h_pts0, k_pts0, I_map0, h_max0, k_max0, G_PLANE0 = compute_diffuse_map(K_LAB_PRIOR, L_layer=0)
plot_diffuse(h_pts0, k_pts0, I_map0, h_max0, k_max0, 0, OUT_PREFIX + 'diffuse_map_hk0.png')
h_pts5, k_pts5, I_map5, h_max5, k_max5, G_PLANE5 = compute_diffuse_map(K_LAB_PRIOR, L_layer=0.5)
plot_diffuse(h_pts5, k_pts5, I_map5, h_max5, k_max5, 0.5, OUT_PREFIX + 'diffuse_map_hk05.png')
# the hk0 plane of P4_3 has 4-fold symmetry: I(h, k, 0) = I(-k, h, 0)
_c4 = np.max(np.abs(I_map0 - np.rot90(I_map0)))/I_map0.max()
print(f"4-fold symmetry of the hk0 map: max |I(h,k) - I(-k,h)| / max I = {_c4:.1e}")

## Reading the mdx2 Map

### File layout and geometry

`mdx2.io.saveobj` stores each object as a group under `/entry`, tagged with the
attributes `mdx2_module` and `mdx2_class`:

* `hkl_table` (`HKLTable`): fields `h`, `k`, `l` (fractional Miller indices, multiples
  of $1/$`ndiv`), `ndiv`, and the data columns;
* `crystal` (`Crystal`): `unit_cell`, `ub_matrix` ($\mathbf s=UB\,\mathbf h$ in Å⁻¹,
  without $2\pi$), `space_group`;
* `symmetry` (`Symmetry`): `laue_group_operators` (real-space rotations $W$;
  indices transform as $\mathbf h\to W^{\mathsf T}\mathbf h$), the ASU expression, and
  the space-group operators.

`mdx2.merge` itself writes only the `hkl_table`. The geometry objects come from
`mdx2.import_geometry`. The cell looks for them in `MDX2_FILE`, then `MDX2_GEOMETRY`,
then in `geometry*.nxs` files next to the data or one directory up. If none is
found it falls back to the 7tx0 cell and the Laue group of P4₃, and says so.

**Checks.**

* The file's Laue operators must be the Laue group of the structure's space group.
* Every operator must respect `ndiv`: an operator that mixes $h$ and $k$ needs equal
  subdivisions along them.
* The data's cell is compared with 7tx0's. The model is evaluated at the data's
  Miller indices with the 7tx0 cell, so $|\mathbf q|$ is off by the relative cell
  difference. Below about 1% this shifts a halo by a small fraction of a voxel.

### Reading

The table is read in chunks of `READ_CHUNK` rows, and only the shell
`D_READ_MIN`–`D_READ_MAX` is kept in memory. A voxel is a **Bragg voxel** if all
three indices are integers and the reflection is allowed by the space group (the
`is_reflection` flag of `mdx2.report`, plus the P4₃ screw absences). Bragg voxels hold
what is left of the Bragg peak and are never fitted. In the same pass the cell
accumulates, in 40 resolution bins over the whole table:

* mean $I$, mean $\sigma$ and the median of $I/\sigma$ (1% subsample);
* the fraction of finite rows and of negative intensities;
* when the merge was split, the correlation of the two half-datasets, with and
  without the Bragg voxels. $CC_{1/2}$ gives
  $CC^*=\sqrt{2CC_{1/2}/(1+CC_{1/2})}$, the correlation of the merged data with the
  noise-free signal (Karplus & Diederichs 2012), which bounds what any model can
  reach. These are the curves of the project's `compare_map_correlations`
  notebook, computed in the same bins of $s=1/d$.

In [ ]:
READ_CHUNK = 4_000_000     # rows per read

def _as_str(x):
    v = np.asarray(x)
    v = v.ravel()[0] if v.size else ''
    return v.decode() if isinstance(v, (bytes, np.bytes_)) else str(v)

def _mdx2_objects(path):
    """mdx2 version and {name: class} of the mdx2 objects in /entry of a NeXus file."""
    out = {}
    with h5py.File(path, 'r') as f:
        ver = _as_str(f.attrs['mdx2_version']) if 'mdx2_version' in f.attrs else None
        if 'entry' in f:
            for k, g in f['entry'].items():
                if isinstance(g, h5py.Group) and 'mdx2_class' in g.attrs:
                    out[k] = _as_str(g.attrs['mdx2_class'])
    return ver, out

def _find_object(cls, paths):
    for p in paths:
        p = pathlib.Path(p)
        if not p.exists():
            continue
        try:
            _, objs = _mdx2_objects(p)
        except OSError:
            continue
        for name, c in objs.items():
            if c == cls:
                return p, name
    return None, None

_ver, _objs = _mdx2_objects(MDX2_FILE)
print(f"{MDX2_FILE}: mdx2 version {_ver}, objects {_objs}")
_tab_name = 'hkl_table' if _objs.get('hkl_table') == 'HKLTable' else \
            next((k for k, c in _objs.items() if c == 'HKLTable'), None)
assert _tab_name is not None, f"no HKLTable in {MDX2_FILE}"

_dir = pathlib.Path(MDX2_FILE).resolve().parent
_geom_paths = [MDX2_FILE] + ([MDX2_GEOMETRY] if MDX2_GEOMETRY else []) + \
              sorted(_dir.glob('geometry*.nxs')) + sorted(_dir.parent.glob('geometry*.nxs')) + \
              sorted(_dir.parent.glob('*/geometry*.nxs'))
_cr_file, _cr_name = _find_object('Crystal', _geom_paths)
_sy_file, _sy_name = _find_object('Symmetry', _geom_paths)

DATA_CELL, UB_DATA, DATA_SG = None, None, None
if _cr_file is not None:
    with h5py.File(_cr_file, 'r') as f:
        g = f[f'entry/{_cr_name}']
        DATA_CELL = np.asarray(g['unit_cell'][()], float).ravel()
        UB_DATA = np.asarray(g['ub_matrix'][()], float).reshape(3, 3)
        DATA_SG = _as_str(g['space_group'][()]) if 'space_group' in g else None
    print(f"crystal from {_cr_file}: cell {np.round(DATA_CELL, 3)}, space group {DATA_SG}")
else:
    print("WARNING: no mdx2 'crystal' object found; using the 7tx0 cell for the data.")

# reference Laue group of the structure: the rotations of the space group and their inverses
_laue_ref = {tuple(np.round(s*W).astype(int).ravel()) for W, _ in SYM_OPS for s in (1, -1)}
if _sy_file is not None:
    with h5py.File(_sy_file, 'r') as f:
        g = f[f'entry/{_sy_name}']
        _lops = np.asarray(g['laue_group_operators'][()], float).reshape(-1, 3, 3)
        DATA_LAUE = _as_str(g['laue_group_symbol'][()]) if 'laue_group_symbol' in g else '?'
        DATA_SG = DATA_SG or (_as_str(g['space_group_symbol'][()]) if 'space_group_symbol' in g else None)
    LAUE_OPS = [np.round(W).astype(int) for W in _lops]
    print(f"symmetry from {_sy_file}: Laue group {DATA_LAUE}, {len(LAUE_OPS)} operators")
    _file_set = {tuple(W.ravel()) for W in LAUE_OPS}
    assert _file_set == _laue_ref, (f"the file's Laue operators are not the Laue group of {SG.hm}: "
                                    f"{len(_file_set)} vs {len(_laue_ref)} operators")
else:
    LAUE_OPS = [np.array(t).reshape(3, 3) for t in sorted(_laue_ref)]
    print(f"WARNING: no mdx2 'symmetry' object found; using the Laue group of {SG.hm} "
          f"({len(LAUE_OPS)} operators).")
if DATA_SG and gemmi.find_spacegroup_by_name(DATA_SG) is not None and \
        gemmi.find_spacegroup_by_name(DATA_SG).number != SG.number:
    print(f"WARNING: the data were processed in {DATA_SG}, the structure is {SG.hm}.")

with h5py.File(MDX2_FILE, 'r') as f:
    _tab = f[f'entry/{_tab_name}']
    TAB_COLUMNS = [k for k in _tab.keys() if k not in ('h', 'k', 'l', 'ndiv')]
    N_ROWS = _tab['h'].shape[0]
    NDIV = tuple(int(round(v)) for v in np.asarray(_tab['ndiv'][()]).ravel()) if 'ndiv' in _tab else (1, 1, 1)
print(f"hkl_table '{_tab_name}': {N_ROWS} rows, ndiv = {NDIV}, columns {TAB_COLUMNS}")
# tables written by mdx2.reintegrate hold counts instead of intensities (as in mdx2.report)
REINTEGRATED = DATA_COLUMN not in TAB_COLUMNS and {'counts', 'background_counts', 'scale'} <= set(TAB_COLUMNS)
ERR_COLUMN = DATA_COLUMN + '_error' if DATA_COLUMN + '_error' in TAB_COLUMNS else 'intensity_error'
if REINTEGRATED:
    print("table from mdx2.reintegrate: intensity = (counts - background_counts)/scale, error = sqrt(counts)/scale")
else:
    assert DATA_COLUMN in TAB_COLUMNS and ERR_COLUMN in TAB_COLUMNS, \
        f"columns {DATA_COLUMN!r} / {ERR_COLUMN!r} not in the table {TAB_COLUMNS}"
HAVE_HALVES = 'group_0_intensity' in TAB_COLUMNS and 'group_1_intensity' in TAB_COLUMNS
print(f"data column {DATA_COLUMN!r} ± {ERR_COLUMN!r}; half-datasets: {HAVE_HALVES}")
for W in LAUE_OPS:
    for i in range(3):
        for j in range(3):
            assert W[i, j] == 0 or NDIV[i] == NDIV[j], \
                f"Laue operator {W.tolist()} mixes axes {i}, {j} with ndiv {NDIV}"
grid_delta = tuple(1.0/n for n in NDIV)
P1, P2, P3 = NDIV
print(f"voxel size: Δh = 1/{P1}, Δk = 1/{P2}, Δl = 1/{P3}  "
      f"({np.linalg.norm(B_recip[:, 0])/P1:.4f}, {np.linalg.norm(B_recip[:, 1])/P2:.4f}, "
      f"{np.linalg.norm(B_recip[:, 2])/P3:.4f} Å⁻¹)")

# --- the data's cell against the structure's ---------------------------------------
if DATA_CELL is not None:
    print(f"\n{'':6s} {'7tx0':>9s} {'data':>9s} {'rel. diff':>10s}")
    for nm, dep, dat in zip(['a', 'b', 'c', 'alpha', 'beta', 'gamma'],
                            [cell.a, cell.b, cell.c, cell.alpha, cell.beta, cell.gamma], DATA_CELL):
        rd = (dat - dep)/dep
        print(f"{nm:6s} {dep:9.3f} {dat:9.3f} {rd:+10.2%}{'  <-- above 1%' if abs(rd) > 0.01 else ''}")
    _hs = np.random.default_rng(0).uniform(-30, 30, (2000, 3)); _hs[:, 2] /= 2.2
    _r = np.linalg.norm(_hs @ UB_DATA.T, axis=1)/(np.linalg.norm(_hs @ B_recip.T, axis=1)/(2*np.pi))
    print(f"|q|(data UB) / |q|(7tx0 cell) over random indices: {_r.mean():.4f} ± {_r.std():.4f}")
    _shift = np.abs(_r.mean() - 1)*(2*np.pi/D_MIN_HALO)
    print(f"at the halo limit {D_MIN_HALO} Å this is a radial shift of {_shift:.4f} Å⁻¹ "
          f"({_shift/(np.linalg.norm(B_recip[:, 0])/P1):.2f} of the smallest voxel)")

In [ ]:
# --- One pass over the table: diagnostics, half-dataset CC, and the kept shell ---------
def _allowed(Hint):
    """True for integer indices of reflections allowed by the space group."""
    ok = np.ones(len(Hint), bool)
    for W, w in SYM_OPS[1:]:
        fixed = np.all(Hint @ W == Hint, axis=1)
        ph = Hint @ w
        ok &= ~(fixed & (np.abs(ph - np.round(ph)) > 1e-6))
    return ok

N_RES_BINS = 40
S_MAX_DIAG = 0.8                               # 1/d of the last diagnostic bin (1.25 Å)
_s_edges = np.linspace(0.0, S_MAX_DIAG, N_RES_BINS + 1)
_acc = {k: np.zeros(N_RES_BINS) for k in ('n', 'nfin', 'sI', 'sS', 'nneg')}
_cc = {k: {m: np.zeros(N_RES_BINS) for m in ('n', 'a', 'b', 'aa', 'bb', 'ab')} for k in ('all', 'excl')}
_snr = [[] for _ in range(N_RES_BINS)]
_q_lo, _q_hi = 2*np.pi/D_READ_MAX, 2*np.pi/D_READ_MIN
_kept = {k: [] for k in ('H', 'I', 'S', 'g0', 'g1', 'bragg')}
_rng_d = np.random.default_rng(0)
_cols = (['counts', 'background_counts', 'scale'] if REINTEGRATED else [DATA_COLUMN, ERR_COLUMN]) + \
        (['group_0_intensity', 'group_1_intensity'] if HAVE_HALVES else [])
with h5py.File(MDX2_FILE, 'r') as f:
    _tab = f[f'entry/{_tab_name}']
    for r0 in range(0, N_ROWS, READ_CHUNK):
        sl = slice(r0, min(r0 + READ_CHUNK, N_ROWS))
        hkl = np.column_stack([np.asarray(_tab[a][sl], float) for a in 'hkl'])
        Hint = np.round(hkl*np.array(NDIV)).astype(np.int32)
        assert np.abs(hkl*np.array(NDIV) - Hint).max() < 1e-3, "indices are not multiples of 1/ndiv"
        cols = {c: np.asarray(_tab[c][sl], np.float64) for c in _cols}
        if REINTEGRATED:
            I = (cols['counts'] - cols['background_counts'])/cols['scale']
            S = np.sqrt(np.maximum(cols['counts'], 0))/cols['scale']
        else:
            I, S = cols[DATA_COLUMN], cols[ERR_COLUMN]
        qn = np.linalg.norm((Hint/np.array(NDIV)) @ B_recip.T, axis=1)
        sbin = np.clip(np.digitize(qn/(2*np.pi), _s_edges) - 1, 0, N_RES_BINS - 1)
        inrange = qn/(2*np.pi) < S_MAX_DIAG
        fin = np.isfinite(I) & np.isfinite(S) & (S > 0)
        integer = np.all(Hint % np.array(NDIV) == 0, axis=1)
        bragg = np.zeros(len(Hint), bool)
        if integer.any():
            bragg[integer] = _allowed(Hint[integer]//np.array(NDIV))
        np.add.at(_acc['n'], sbin[inrange], 1)
        m = fin & inrange
        np.add.at(_acc['nfin'], sbin[m], 1); np.add.at(_acc['sI'], sbin[m], I[m])
        np.add.at(_acc['sS'], sbin[m], S[m]); np.add.at(_acc['nneg'], sbin[m & (I < 0)], 1)
        thin = m & (_rng_d.random(len(I)) < 0.01)
        for b_, v_ in zip(sbin[thin], I[thin]/S[thin]):
            _snr[b_].append(float(v_))
        if HAVE_HALVES:
            a_, b2 = cols['group_0_intensity'], cols['group_1_intensity']
            mh = inrange & np.isfinite(a_) & np.isfinite(b2)
            for key, mm in (('all', mh), ('excl', mh & ~bragg)):
                for nm, v in (('n', np.ones(len(a_))), ('a', a_), ('b', b2), ('aa', a_*a_), ('bb', b2*b2), ('ab', a_*b2)):
                    np.add.at(_cc[key][nm], sbin[mm], v[mm])
        keep = fin & (qn >= _q_lo) & (qn <= _q_hi)
        _kept['H'].append(Hint[keep]); _kept['I'].append(I[keep].astype(np.float32))
        _kept['S'].append(S[keep].astype(np.float32)); _kept['bragg'].append(bragg[keep])
        if HAVE_HALVES:
            _kept['g0'].append(cols['group_0_intensity'][keep].astype(np.float32))
            _kept['g1'].append(cols['group_1_intensity'][keep].astype(np.float32))
        del hkl, Hint, cols, I, S, qn
HKL_INT_DATA = np.concatenate(_kept['H']); I_ALL = np.concatenate(_kept['I'])
S_ALL = np.concatenate(_kept['S']); BRAGG_ALL = np.concatenate(_kept['bragg'])
G0_ALL = np.concatenate(_kept['g0']) if HAVE_HALVES else None
G1_ALL = np.concatenate(_kept['g1']) if HAVE_HALVES else None
del _kept; gc.collect()

with np.errstate(invalid='ignore', divide='ignore'):
    s_mid = 0.5*(_s_edges[:-1] + _s_edges[1:]); d_mid = 1/s_mid
    cnt_finite = _acc['nfin']
    snr_proxy = (_acc['sI']/cnt_finite)/(_acc['sS']/cnt_finite)
    med_snr = np.array([np.median(v) if len(v) > 20 else np.nan for v in _snr])
    finite_frac = cnt_finite/np.maximum(_acc['n'], 1)
    neg_frac_res = _acc['nneg']/cnt_finite
    def _pearson(c):
        n = np.maximum(c['n'], 1)
        cov = c['ab']/n - c['a']*c['b']/n**2
        va, vb = c['aa']/n - (c['a']/n)**2, c['bb']/n - (c['b']/n)**2
        return np.where(c['n'] >= 20, cov/np.sqrt(va*vb), np.nan)
    CC_HALF = {k: _pearson(_cc[k]) for k in _cc} if HAVE_HALVES else None
    CC_STAR = {k: np.sqrt(np.clip(2*v/(1 + v), 0, None)) for k, v in CC_HALF.items()} if HAVE_HALVES else None
print(f"kept {len(I_ALL)} voxels in {D_READ_MAX}–{D_READ_MIN} Å "
      f"({BRAGG_ALL.sum()} of them Bragg voxels) of {N_ROWS} rows; "
      f"{I_ALL.nbytes*(4 + 2*HAVE_HALVES)/1e6 + HKL_INT_DATA.nbytes/1e6:.0f} MB in memory")

In [ ]:
fig, axes = plt.subplots(1, 4 if HAVE_HALVES else 3, figsize=(20 if HAVE_HALVES else 15, 4))
finite_d = np.isfinite(d_mid) & (cnt_finite > 0)
axes[0].plot(d_mid[finite_d], snr_proxy[finite_d], 'o-', ms=3, label=r'$\overline{I}/\overline{\sigma}$')
axes[0].plot(d_mid[finite_d], med_snr[finite_d], 's-', ms=3, color='darkgreen', label=r'median $I/\sigma$')
axes[0].axhline(1.0, color='r', ls='--', lw=1)
axes[0].set_ylabel('signal-to-noise'); axes[0].set_title('Signal-to-noise vs. resolution')
axes[1].plot(d_mid[finite_d], finite_frac[finite_d], 'o-', ms=3, color='darkorange')
axes[1].set_ylabel('finite fraction of the rows'); axes[1].set_title('Finite rows vs. resolution')
axes[2].plot(d_mid[finite_d], neg_frac_res[finite_d], 'o-', ms=3, color='crimson')
axes[2].axhline(0.5, color='k', ls=':', lw=1)
axes[2].set_ylabel(r'fraction with $I<0$'); axes[2].set_title('Negative intensities vs. resolution')
if HAVE_HALVES:
    axes[3].plot(d_mid, CC_STAR['excl'], '-', color='tab:purple', label='$CC^*$, Bragg voxels excluded')
    axes[3].plot(d_mid, CC_STAR['all'], '--', color='tab:purple', alpha=0.6, label='$CC^*$, all voxels')
    axes[3].plot(d_mid, CC_HALF['excl'], '-', color='0.5', lw=1, label='$CC_{1/2}$, Bragg excluded')
    axes[3].set_ylim(0, 1.05); axes[3].set_ylabel('correlation'); axes[3].legend(fontsize=7)
    axes[3].set_title('Half-dataset correlation')
from matplotlib.ticker import FixedLocator, NullLocator, FormatStrFormatter
def _d_axis(a, lo, hi, ticks=(1.5, 2, 2.5, 3, 4, 6, 10, 20)):
    """Logarithmic resolution axis, high resolution on the right, plain tick labels."""
    a.set_xscale('log'); a.set_xlim(hi, lo)
    a.xaxis.set_major_locator(FixedLocator([t for t in ticks if lo <= t <= hi]))
    a.xaxis.set_minor_locator(NullLocator()); a.xaxis.set_major_formatter(FormatStrFormatter('%g'))
    a.set_xlabel('resolution $d$ (Å)')
for a in axes:
    a.axvspan(D_FIT_MIN, D_FIT_MAX, color='steelblue', alpha=0.15)
    _d_axis(a, 1/S_MAX_DIAG, 25)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'resolution_diagnostics.png', dpi=150); plt.show()
_in = finite_d & (d_mid >= D_FIT_MIN) & (d_mid <= D_FIT_MAX)
print(f"fitting range {D_FIT_MIN}–{D_FIT_MAX} Å: median I/σ {np.nanmin(med_snr[_in]):.2f}–{np.nanmax(med_snr[_in]):.2f}, "
      f"negative fraction {np.nanmin(neg_frac_res[_in]):.1%}–{np.nanmax(neg_frac_res[_in]):.1%}")
if HAVE_HALVES:
    print(f"  CC* (Bragg voxels excluded) {np.nanmin(CC_STAR['excl'][_in]):.3f}–{np.nanmax(CC_STAR['excl'][_in]):.3f}")
    print(f"  {'d (Å)':>12s} {'CC1/2':>7s} {'CC*':>7s} {'CC1/2 all':>10s}")
    for b in range(0, N_RES_BINS, 4):
        if np.isfinite(CC_HALF['excl'][b]):
            print(f"  {1/_s_edges[b + 1]:5.2f}–{min(1/max(_s_edges[b], 1e-9), 99):5.1f} {CC_HALF['excl'][b]:7.3f} "
                  f"{CC_STAR['excl'][b]:7.3f} {CC_HALF['all'][b]:10.3f}")

### Which Indexing Do the Data Use?

The lattice of a tetragonal crystal has symmetry 4/mmm, but P4₃ only 4/m. The
operations of the lattice that are not in the Laue group (for this lattice,
$(h,k,l)\to(k,h,-l)$ and its products with 4/m) map the reciprocal lattice onto itself
and preserve every $|\mathbf q|$, but they do not map the crystal's intensities onto
themselves. A crystal can be indexed either way, so the data and 7tx0 may disagree.
A wrong choice keeps every halo at a reciprocal-lattice point and every resolution
right, but pairs each halo with the wrong $|F(\mathbf G)|$.

The candidate operators are found from the cell: every integer matrix $M$ with
entries in $\{-1,0,1\}$, $\det M=1$, that preserves the metric to 0.5%, taken
modulo the Laue group. For each, the cell correlates the mean intensity of the
inner halo voxels around each reciprocal-lattice point (between `BRAGG_EXCL_CART`
and `INDEX_TEST_R`) with $|F_{\rm cell}(M\mathbf G)|^2$ from the structure. It uses
a Spearman rank correlation within resolution shells, so neither the overall
fall-off nor a smooth background drives it. The correct indexing gives a clearly
higher correlation. With `REINDEX = 'auto'` the best operator is applied to the data
indices, after which everything is in the 7tx0 indexing.

In [ ]:
from scipy.stats import spearmanr

INDEX_TEST_R      = 0.04    # Å⁻¹: inner-halo radius
INDEX_TEST_POINTS = 2000    # reciprocal-lattice points used (random subset)

def _metric_preserving_cosets():
    Gm = A_orth.T @ A_orth
    mats = []
    for e in product((-1, 0, 1), repeat=9):
        M = np.array(e).reshape(3, 3)
        if round(np.linalg.det(M)) != 1:
            continue
        if np.allclose(M.T @ Gm @ M, Gm, rtol=5e-3, atol=5e-3*np.abs(Gm).max()):
            mats.append(M)
    # reciprocal action h -> M h; cosets modulo the Laue group {W^T}
    laue_T = [W.T for W in LAUE_OPS]
    reps = []
    for M in sorted(mats, key=lambda M: (not np.array_equal(M, np.eye(3, dtype=int)), M.tolist())):
        if not any(any(np.array_equal(M, R @ L) for L in laue_T) for R in reps):
            reps.append(M)
    return reps

def _shell_spearman(x, y, qn, n_shell=8):
    edges = np.quantile(qn, np.linspace(0, 1, n_shell + 1))
    b = np.clip(np.digitize(qn, edges) - 1, 0, n_shell - 1)
    rs, ws = [], []
    for s_ in range(n_shell):
        m = b == s_
        if m.sum() >= 20:
            rs.append(spearmanr(x[m], y[m]).correlation); ws.append(m.sum())
    return float(np.average(rs, weights=ws)) if rs else np.nan

_hkl_d = HKL_INT_DATA/np.array(NDIV)
_G_int = np.round(_hkl_d).astype(int)
_bd = np.linalg.norm((_hkl_d - _G_int) @ B_recip.T, axis=1)
_qn_d = np.linalg.norm(_hkl_d @ B_recip.T, axis=1)
_m = (_bd > BRAGG_EXCL_CART) & (_bd < INDEX_TEST_R) & (_qn_d >= 2*np.pi/D_FIT_MAX) & (_qn_d <= 2*np.pi/D_FIT_MIN)
_keys, _inv, _cnt = np.unique(_G_int[_m], axis=0, return_inverse=True, return_counts=True)
_inv = np.asarray(_inv).ravel()
_meanI = np.bincount(_inv, weights=I_ALL[_m].astype(float))/_cnt
_ok = (_cnt >= 3) & _allowed(_keys)
_keys, _meanI = _keys[_ok], _meanI[_ok]
if len(_keys) > INDEX_TEST_POINTS:
    _pick = np.random.default_rng(0).choice(len(_keys), INDEX_TEST_POINTS, replace=False)
    _keys, _meanI = _keys[_pick], _meanI[_pick]
_qG = np.linalg.norm(_keys @ B_recip.T, axis=1)
REINDEX_CANDIDATES = _metric_preserving_cosets()
print(f"indexing test: {len(_keys)} reciprocal-lattice points with >= 3 inner halo voxels; "
      f"{len(REINDEX_CANDIDATES)} candidate indexing(s)")
_index_rows = []
for M in REINDEX_CANDIDATES:
    F2 = np.abs(F_total_hkl(_keys @ M.T))**2
    _index_rows.append((_shell_spearman(F2, _meanI, _qG), M))
for rho, M in sorted(_index_rows, key=lambda r: -r[0]):
    lab = ', '.join(''.join(f"{'+' if c > 0 else '-'}{'hkl'[j]}" for j, c in enumerate(row) if c).lstrip('+')
                    for row in M)
    print(f"   h_model = ({lab}):  ρ = {rho:+.3f}{'   (identity)' if np.array_equal(M, np.eye(3)) else ''}")
_best = max(_index_rows, key=lambda r: r[0])
_ident = [r for r in _index_rows if np.array_equal(r[1], np.eye(3))][0]
if isinstance(REINDEX, str) and REINDEX == 'none':
    M_REINDEX = np.eye(3, dtype=int)
elif isinstance(REINDEX, str) and REINDEX == 'auto':
    M_REINDEX = _best[1] if (_best[0] - _ident[0] > 0.1) else np.eye(3, dtype=int)
    if not np.array_equal(M_REINDEX, np.eye(3)) and REINDEX_STRICT:
        raise RuntimeError(f"the data are in a different indexing: set REINDEX = {M_REINDEX.tolist()}")
else:
    M_REINDEX = np.asarray(REINDEX, int)
if (_best[0] - sorted(r[0] for r in _index_rows)[-2] < 0.1) if len(_index_rows) > 1 else False:
    print("WARNING: the indexing test does not separate the candidates clearly (Δρ < 0.1).")
print(f"\nusing h_model = M h_data with M = {M_REINDEX.tolist()}"
      f"{'  (the data are reindexed)' if not np.array_equal(M_REINDEX, np.eye(3)) else ''}")
if not np.array_equal(M_REINDEX, np.eye(3)):
    for i in range(3):
        for j in range(3):
            assert M_REINDEX[i, j] == 0 or NDIV[i] == NDIV[j], "the reindexing operator does not respect ndiv"
HKL_INT = HKL_INT_DATA @ M_REINDEX.T           # model indexing, integer grid units
del HKL_INT_DATA, _hkl_d, _G_int, _bd, _qn_d; gc.collect()

### Looking Up the Map at Any Index

The table holds one asymmetric unit. `lookup_map(hkl, field)` returns the value at
any fractional index on the data grid by mapping both the table and the query onto
a canonical representative of their Laue orbit (the largest of the images
$\mathbf hW$, packed into one integer), so the reader does not depend on which
asymmetric unit mdx2 used. Missing voxels come back as NaN.

In [ ]:
_NDIV = np.array(NDIV)
_PACK = 1 << 20
def _canon_key(Hint):
    """One int64 per Laue orbit of integer grid indices (largest packed image)."""
    Hint = np.asarray(Hint, np.int64)
    best = None
    for W in LAUE_OPS:
        h = Hint @ W
        k = ((h[:, 0] + _PACK//2)*_PACK + (h[:, 1] + _PACK//2))*_PACK + (h[:, 2] + _PACK//2)
        best = k if best is None else np.maximum(best, k)
    return best

_KEY_ALL = _canon_key(HKL_INT)
_ORD = np.argsort(_KEY_ALL); _KEY_SORTED = _KEY_ALL[_ORD]
_dup = np.count_nonzero(np.diff(_KEY_SORTED) == 0)
print(f"lookup table: {len(_KEY_ALL)} voxels, {_dup} duplicate Laue orbits"
      f"{' (the table is not one asymmetric unit!)' if _dup else ''}")

def lookup_map(hkl, field='I'):
    """Map value at fractional indices hkl (model indexing), NaN where absent."""
    hkl = np.atleast_2d(np.asarray(hkl, float))
    Hq = np.round(hkl*_NDIV).astype(np.int64)
    on_grid = np.all(np.abs(hkl*_NDIV - Hq) < 1e-6, axis=1)
    k = _canon_key(Hq)
    pos = np.clip(np.searchsorted(_KEY_SORTED, k), 0, len(_KEY_SORTED) - 1)
    hit = (_KEY_SORTED[pos] == k) & on_grid
    src = {'I': I_ALL, 'sigma': S_ALL, 'g0': G0_ALL, 'g1': G1_ALL, 'bragg': BRAGG_ALL}[field]
    out = np.full(len(hkl), np.nan)
    out[hit] = src[_ORD[pos[hit]]]
    return out

# self-check: every stored voxel is found at itself and at a random Laue image
_t = np.random.default_rng(1).choice(len(HKL_INT), min(5000, len(HKL_INT)), replace=False)
_Wr = LAUE_OPS[3 % len(LAUE_OPS)]
assert np.allclose(lookup_map(HKL_INT[_t]/_NDIV), I_ALL[_t]) and \
       np.allclose(lookup_map((HKL_INT[_t] @ _Wr)/_NDIV), I_ALL[_t]), "lookup_map is inconsistent"
print("lookup_map: every tested voxel is found at itself and at its Laue images")

### The Candidate Pool

Every finite, non-Bragg voxel in `D_FIT_MIN`–`D_FIT_MAX` at least `BRAGG_EXCL_CART`
from its nearest reciprocal-lattice point (rounding each index; Cartesian distance)
is a candidate. Which of them are fitted is decided by the halo selection below.

In [ ]:
def _bragg_offsets(hkl):
    dq = (hkl - np.round(hkl)) @ B_recip.T
    return np.linalg.norm(dq, axis=1)

_hkl_all = HKL_INT/_NDIV
_qn_all = np.linalg.norm(_hkl_all @ B_recip.T, axis=1)
_bd_all = _bragg_offsets(_hkl_all)
_pool = (~BRAGG_ALL) & (_qn_all >= 2*np.pi/D_FIT_MAX) & (_qn_all <= 2*np.pi/D_FIT_MIN) & (_bd_all >= BRAGG_EXCL_CART)
hkl_pool = _hkl_all[_pool]
I_pool = I_ALL[_pool].astype(float); sigma_pool = S_ALL[_pool].astype(float)
bragg_dist_pool = _bd_all[_pool]
POOL_INDEX = np.where(_pool)[0]                # rows of the kept arrays
del _hkl_all, _qn_all, _bd_all; gc.collect()
print(f"{len(hkl_pool)} candidate voxels in {D_FIT_MIN}–{D_FIT_MAX} Å; negative fraction {(I_pool < 0).mean():.1%}")

## Negative Intensities

Background-subtracted voxels can be negative. They are kept throughout: if
$I_{\rm obs}=I_{\rm true}+\varepsilon$ with $\mathbb E[\varepsilon]=0$, weighted least squares
against the unfiltered data is unbiased, while discarding voxels by the sign of
$I_{\rm obs}$ would bias the sample upward. A model that predicts nearly nothing
shows $R\to1$ together with $CC\to0$, so both are reported.

### Halo and Stratified Mid-Zone Sampling

With `FIT_MODE = 'halo'` (the default) only halo voxels are fitted, following
Meisburger et al. (2020), who refined against about 400 bright halos between 2 and
2.5 Å. With `HALO_EXTENT = 'cell'` every voxel whose nearest reciprocal-lattice point
is a selected $\mathbf G$ is eligible, from `BRAGG_EXCL_CART` out to the cell
boundary. `N_HALO_FIT_ONLY` of them are drawn at random (`HALO_SAMPLING =
'uniform'`), or in equal quotas per Bragg-distance shell (`'shells'`). Whole halos
are held out. With `FIT_MODE = 'both'` a stratified mid-zone sample (`N_RES_STRATA`
resolution × `N_HALO_STRATA` Bragg-distance strata) is added to halo voxels within
`HALO_R_MAX`, and the halo block's total weight is `HALO_WEIGHT` times the mid-zone's.

### Choosing Which Halos to Fit

Near $\mathbf G$ the acoustic branches soften as $\omega\propto|\delta\mathbf q|$ and

$$I(\mathbf G+\delta\mathbf q)\approx\frac{|\mathbf G\,F_{\rm cell}(\mathbf G)\cdot\hat e_{\rm ac}|^2}{\omega_{\rm ac}^2(\delta\mathbf q)},$$

so a halo's brightness scales with $|\mathbf q\,F_{\rm cell}(\mathbf q)|^2$ at its own
Bragg position, with $F_{\rm cell}=\sum_bF_b$ the contrast transform of the whole
cell. The systematically absent $00l$ ($l\neq4n$) have none. The score is computed
from the structure and the reported $\sigma$, never from the intensities being fitted:

* `'coupling'` (default): $|\mathbf qF_{\rm cell}|^2$, the predicted halo brightness;
* `'information'`: $|\mathbf qF_{\rm cell}|^2/\sigma^2$, with $\sigma^2$ the median
  variance of that point's halo voxels;
* `'resolution'`: every point in the window.

The map holds one asymmetric unit, so one physical halo is stored under several
symmetry mates of $\mathbf G$ (each holds the part of the reciprocal cell that falls
in the asymmetric unit). Scores, selection and the cross-validation groups are
therefore per **Laue orbit** of $\mathbf G$. Only orbits with centres in
`D_MAX_HALO`–`D_MIN_HALO` and at least 10 measured halo voxels are eligible, and the
`N_HALO_SELECT` highest-scoring are kept.

In [ ]:
HALO_SELECT   = 'coupling'
HALO_TOP_FRAC = 0.35         # fraction kept when N_HALO_SELECT is None

_hl = [int(np.ceil(L/D_MIN_HALO)) + 1 for L in (cell.a, cell.b, cell.c)]
_HH, _KK, _LL = np.meshgrid(*[np.arange(-h, h + 1) for h in _hl], indexing='ij')
_hall = np.column_stack([_HH.ravel(), _KK.ravel(), _LL.ravel()])
_qall = np.linalg.norm(_hall @ B_recip.T, axis=1)
_ok = (_qall > 1e-6) & (_qall <= 2*np.pi/D_MIN_HALO) & ~systematically_absent(_hall)
# The map holds one asymmetric unit, so the voxels of one physical halo are stored under
# several symmetry mates of G. Everything here works per Laue ORBIT of G: one score, one
# selection decision and one cross-validation group per orbit.
ORBIT_KEYS, _uidx, _mult = np.unique(laue_orbit_key(_hall[_ok]), return_index=True, return_counts=True)
_hkl = _hall[_ok][_uidx].astype(float)                 # one representative per orbit
_qn = _qall[_ok][_uidx]
_F = F_total_hkl(_hkl)
COUPLING2 = (_qn*np.abs(_F))**2
_d_hkl = 2*np.pi/_qn

_edges = np.quantile(_d_hkl, np.linspace(0, 1, 25))
_cen = 0.5*(_edges[:-1] + _edges[1:])
_bi = np.clip(np.digitize(_d_hkl, _edges) - 1, 0, len(_cen) - 1)
_prof = np.array([np.mean(COUPLING2[_bi == b]) if (_bi == b).any() else np.nan for b in range(len(_cen))])
_Fprof = np.array([np.mean(np.abs(_F)[_bi == b]) if (_bi == b).any() else np.nan for b in range(len(_cen))])
_peak = _cen[np.nanargmax(_prof)]
print(f"{_ok.sum()} allowed reciprocal-lattice points to {D_MIN_HALO} Å in {len(ORBIT_KEYS)} Laue orbits")
print(f"mean |qF_cell|² peaks at d = {_peak:.2f} Å")

# pool voxels -> orbit of their nearest reciprocal-lattice point
pool_orbit = laue_orbit_key(np.round(hkl_pool).astype(int))
_pos = np.clip(np.searchsorted(ORBIT_KEYS, pool_orbit), 0, len(ORBIT_KEYS) - 1)
_idx_of = np.where(ORBIT_KEYS[_pos] == pool_orbit, _pos, -1)
_near = (bragg_dist_pool < HALO_R_MAX) if (FIT_MODE == 'both' or HALO_EXTENT != 'cell') else \
        np.ones(len(bragg_dist_pool), bool)          # the halo is the whole reciprocal cell
_sig2 = np.full(len(_hkl), np.nan); _negf = np.full(len(_hkl), np.nan); _nvox = np.zeros(len(_hkl), int)
_hsel = np.where(_near & (_idx_of >= 0))[0]
_hsel = _hsel[np.argsort(_idx_of[_hsel], kind='stable')]
_grp, _g0, _gn = np.unique(_idx_of[_hsel], return_index=True, return_counts=True)
for i, s0, n in zip(_grp, _g0, _gn):
    _nvox[i] = n
    if n >= 10:
        m = _hsel[s0:s0 + n]
        _sig2[i] = np.median(sigma_pool[m]**2)
        _negf[i] = (I_pool[m] < 0).mean()
_have = np.isfinite(_sig2)
INFORMATION = np.where(_have, COUPLING2/np.maximum(_sig2, 1e-30), 0.0)
print(f"{_have.sum()} of {len(_hkl)} orbits have >= 10 measured halo voxels "
      f"(median {np.median(_nvox[_have]) if _have.any() else 0:.0f} voxels per orbit)")

if HALO_SELECT == 'information':
    _score = INFORMATION
elif HALO_SELECT == 'coupling':
    _score = np.where(_have, COUPLING2, 0.0)
else:
    _score = np.where(_have, 1.0, 0.0)
_inwin = (_d_hkl >= D_MIN_HALO) & (_d_hkl <= D_MAX_HALO)
_score = np.where(_inwin, _score, 0.0)
if N_HALO_SELECT:
    _order = np.argsort(_score)[::-1]
    _keep = np.zeros(len(_score), bool)
    _keep[_order[:min(N_HALO_SELECT, int((_score > 0).sum()))]] = True
else:
    _cut = np.quantile(_score[_score > 0], 1.0 - HALO_TOP_FRAC) if (_score > 0).any() else 0
    _keep = _score >= max(_cut, 1e-300)
HKL_KEEP_KEYS = ORBIT_KEYS[_keep]
print(f"selection '{HALO_SELECT}': {_keep.sum()} orbits with centres in {D_MAX_HALO}–{D_MIN_HALO} Å "
      f"(of {int((_inwin & _have).sum())} candidates)")
if _keep.any():
    print(f"  resolution span {_d_hkl[_keep].min():.2f}–{_d_hkl[_keep].max():.2f} Å; they carry "
          f"{100*COUPLING2[_keep].sum()/max(COUPLING2[_inwin & _have].sum(), 1e-300):.0f}% of |qF|² in the window")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(_cen, _Fprof, 'o-', color='steelblue')
ax[0].set_xlabel('resolution $d$ (Å)'); ax[0].set_ylabel(r'$\langle|F_{\rm cell}|\rangle$')
ax[0].invert_xaxis(); ax[0].set_title('Contrast transform of the cell')
ax[1].plot(_cen, _prof, 'o-', color='darkorange')
ax[1].axvspan(D_MIN_HALO, D_MAX_HALO, color='0.85', zorder=0, label=f'halo window')
ax[1].set_xlabel('resolution $d$ (Å)'); ax[1].set_ylabel(r'$\langle|qF_{\rm cell}|^2\rangle$')
ax[1].invert_xaxis(); ax[1].set_title('Halo coupling strength'); ax[1].legend(fontsize=8)
ax[2].scatter(_d_hkl[~_keep & _have], COUPLING2[~_keep & _have], s=3, alpha=0.25, color='0.6', label='measured, dropped')
ax[2].scatter(_d_hkl[_keep], COUPLING2[_keep], s=4, alpha=0.6, color='crimson', label='kept')
ax[2].set_yscale('log'); ax[2].invert_xaxis()
ax[2].set_xlabel('resolution $d$ (Å)'); ax[2].set_ylabel(r'$|qF_{\rm cell}|^2$ per reflection')
ax[2].set_title('Per-reflection spread'); ax[2].legend(fontsize=8, markerscale=3)
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'halo_coupling.png', dpi=150); plt.show()

In [ ]:
def effective_sample_size(w):
    w = np.asarray(w, float)
    return w.sum()**2/np.sum(w**2)

def make_weights(sigma):
    """Inverse-variance weights with a percentile floor on sigma."""
    s = np.asarray(sigma, float)
    floor = np.percentile(s[np.isfinite(s) & (s > 0)], SIGMA_FLOOR_PCT)
    return 1.0/np.maximum(s, floor)**2

def sample_halo_voxels(hkl, I, sigma, bragg_d, n_total=N_HALO_FIT,
                       r_max=HALO_R_MAX, n_shell=N_HALO_SHELLS, seed=3, sampling='shells'):
    """Individual voxels from the halos of the selected reciprocal-lattice points
    (between BRAGG_EXCL_CART and r_max; r_max=None: the whole reciprocal cell)."""
    rng_h = np.random.default_rng(seed)
    r_hi = (float(bragg_d.max()) + 1e-9) if r_max is None else r_max
    sel = np.where((bragg_d >= BRAGG_EXCL_CART) & (bragg_d < r_hi))[0]
    edges = np.linspace(BRAGG_EXCL_CART, r_hi, n_shell + 1)
    sh = np.clip(np.digitize(bragg_d[sel], edges) - 1, 0, n_shell - 1)
    if sampling == 'uniform':
        idx = sel if (n_total is None or n_total >= len(sel)) else \
              np.sort(rng_h.choice(sel, size=n_total, replace=False))
    else:
        per = len(sel) if n_total is None else max(1, n_total//n_shell)
        picked = [rng_h.choice(sel[sh == s], size=min(per, int((sh == s).sum())), replace=False)
                  for s in range(n_shell) if (sh == s).any()]
        idx = np.concatenate(picked) if picked else np.zeros(0, int)
    shell = np.clip(np.digitize(bragg_d[idx], edges) - 1, 0, n_shell - 1)
    return hkl[idx], I[idx], sigma[idx], shell, edges

_sel_halo = np.isin(pool_orbit, HKL_KEEP_KEYS)
print(f"halo pool: {_sel_halo.sum()} of {len(hkl_pool)} voxels around a selected orbit "
      f"(~{_sel_halo.sum()/max(_keep.sum(), 1):.0f} per halo)")
_halo_mode = (FIT_MODE == 'halo')
hkl_halo, I_halo, sig_halo, shell_halo, HALO_SHELL_EDGES = sample_halo_voxels(
    hkl_pool[_sel_halo], I_pool[_sel_halo], sigma_pool[_sel_halo], bragg_dist_pool[_sel_halo],
    n_total=(N_HALO_FIT_ONLY if _halo_mode else N_HALO_FIT),
    r_max=(None if (_halo_mode and HALO_EXTENT == 'cell') else HALO_R_MAX),
    sampling=(HALO_SAMPLING if _halo_mode else 'shells'))
cnt_halo = np.ones(len(hkl_halo), int)
refl_halo = laue_orbit_key(np.round(hkl_halo).astype(int))   # CV group: the halo's orbit
_edges_h = HALO_SHELL_EDGES
print(f"\nHalo sample: {len(hkl_halo)} voxels around {len(np.unique(refl_halo))} reciprocal-lattice points")
print(f"  {'shell (Å⁻¹)':>16s} {'voxels':>7s} {'mean I':>10s} {'median I/σ':>11s} {'I<0':>6s}")
for s in range(N_HALO_SHELLS):
    m = shell_halo == s
    if m.any():
        print(f"  {_edges_h[s]:7.4f}–{_edges_h[s + 1]:7.4f} {m.sum():7d} {I_halo[m].mean():10.4g} "
              f"{np.median(I_halo[m]/sig_halo[m]):11.2f} {(I_halo[m] < 0).mean():6.1%}")

# --- Stratified mid-zone sample (used when FIT_MODE = 'both') --------------------------
rng = np.random.default_rng(7)
q_norm_pool = np.linalg.norm(hkl_pool @ B_recip.T, axis=1)
idx_all = np.where(bragg_dist_pool >= HALO_R_MAX)[0]
q_bins = np.quantile(q_norm_pool[idx_all], np.linspace(0, 1, N_RES_STRATA + 1))
b_bins = np.quantile(bragg_dist_pool[idx_all], np.linspace(0, 1, N_HALO_STRATA + 1))
qi = np.clip(np.digitize(q_norm_pool[idx_all], q_bins) - 1, 0, N_RES_STRATA - 1)
bi = np.clip(np.digitize(bragg_dist_pool[idx_all], b_bins) - 1, 0, N_HALO_STRATA - 1)
strata_key = qi*N_HALO_STRATA + bi
per = max(1, N_FIT_MAX//(N_RES_STRATA*N_HALO_STRATA))
picked = [rng.choice(idx_all[strata_key == s_], size=min(per, int((strata_key == s_).sum())), replace=False)
          for s_ in range(N_RES_STRATA*N_HALO_STRATA) if (strata_key == s_).any()]
sub_idx = np.concatenate(picked)
hkl_mid, I_mid, sigma_mid = hkl_pool[sub_idx], I_pool[sub_idx], sigma_pool[sub_idx]
train_mask = rng.random(len(sub_idx)) > HOLDOUT_FRAC
q_train, I_train, sigma_train = hkl_mid[train_mask], I_mid[train_mask], sigma_mid[train_mask]
q_test, I_test, sigma_test = hkl_mid[~train_mask], I_mid[~train_mask], sigma_mid[~train_mask]
w_train_mid = make_weights(sigma_train)
if not _halo_mode:
    print(f"\nMid-zone sample: {len(sub_idx)} points; training {train_mask.sum()} "
          f"(effective {effective_sample_size(w_train_mid):.0f}), held out {(~train_mask).sum()}")

In [ ]:
def well_conditioned_batch(K_lab, hkl, min_eig_frac=1e-9):
    """True where lambda_min(D)/lambda_max(D) > min_eig_frac (removes only points that are
    numerically at a Bragg position)."""
    if len(hkl) == 0:
        return np.zeros(0, bool)
    out = []
    for s in range(0, len(hkl), CHUNK_EVAL):
        ev = np.linalg.eigvalsh(dynamical_matrix_batch(hkl[s:s + CHUNK_EVAL] @ B_recip.T, unique, K_lab))
        out.append((ev[:, -1] > 0) & (ev[:, 0]/np.where(ev[:, -1] > 0, ev[:, -1], 1.0) > min_eig_frac))
    return np.concatenate(out)

keep_tr = well_conditioned_batch(K_LAB_PRIOR, q_train)
keep_te = well_conditioned_batch(K_LAB_PRIOR, q_test)
keep_ha = well_conditioned_batch(K_LAB_PRIOR, hkl_halo)
print(f"conditioning screen: mid-zone training {keep_tr.sum()}/{len(keep_tr)}, held out "
      f"{keep_te.sum()}/{len(keep_te)}, halo {keep_ha.sum()}/{len(keep_ha)}")
q_train, I_train, sigma_train = q_train[keep_tr], I_train[keep_tr], sigma_train[keep_tr]
q_test, I_test, sigma_test = q_test[keep_te], I_test[keep_te], sigma_test[keep_te]
hkl_halo, I_halo, sig_halo, cnt_halo, shell_halo, refl_halo = (
    hkl_halo[keep_ha], I_halo[keep_ha], sig_halo[keep_ha], cnt_halo[keep_ha], shell_halo[keep_ha], refl_halo[keep_ha])
w_train_mid = make_weights(sigma_train)

### Assembling the Objective

**Halo-only mode.** The objective is the halo voxels alone with natural $1/\sigma^2$
weights (σ floored at its `SIGMA_FLOOR_PCT`-th percentile), so $\chi^2/N$ is a true
reduced $\chi^2$ against the quoted errors. A fraction `HALO_HOLDOUT_FRAC` of the
selected reciprocal-lattice points is held out whole. Anything specific to one halo
(an error in $|F(\mathbf G)|$, Bragg-tail contamination, a mosaic width) is shared by
all its voxels, and a voxel-wise split would let the fit learn it from the training
voxels and "predict" it in the test voxels.

**Combined mode.** Halo and mid-zone voxels in one weighted sum, the halo weights
rescaled to `HALO_WEIGHT` times the mid-zone total.

$G(\mathbf q)$ depends only on the structure and $\mathbf q$, so it is computed once for
the fitting and held-out points.

In [ ]:
if FIT_MODE == 'halo':
    _rk = np.unique(refl_halo)
    _held = np.random.default_rng(13).choice(_rk, size=max(1, int(round(HALO_HOLDOUT_FRAC*len(_rk)))), replace=False)
    _te = np.isin(refl_halo, _held)
    q_test, I_test, sigma_test, shell_test = hkl_halo[_te], I_halo[_te], sig_halo[_te], shell_halo[_te]
    hkl_halo, I_halo, sig_halo, cnt_halo, shell_halo, refl_halo = (
        hkl_halo[~_te], I_halo[~_te], sig_halo[~_te], cnt_halo[~_te], shell_halo[~_te], refl_halo[~_te])
    w_halo = make_weights(sig_halo); HALO_W_RESCALE = 1.0
    q_fit, I_fit, w_fit = hkl_halo, I_halo, w_halo
    is_halo = np.ones(len(q_fit), bool)
    refl_fit = refl_halo
    print(f"FIT_MODE = 'halo': {len(q_fit)} voxels around {len(np.unique(refl_fit))} reciprocal-lattice "
          f"points for fitting; {len(q_test)} voxels around {len(_held)} held-out points for testing")
elif len(hkl_halo):
    w_halo = make_weights(sig_halo)
    HALO_W_RESCALE = HALO_WEIGHT*(w_train_mid.sum()/max(w_halo.sum(), 1e-30))
    w_halo = HALO_W_RESCALE*w_halo
    q_fit = np.vstack([q_train, hkl_halo]); I_fit = np.concatenate([I_train, I_halo])
    w_fit = np.concatenate([w_train_mid, w_halo])
    is_halo = np.concatenate([np.zeros(len(q_train), bool), np.ones(len(hkl_halo), bool)])
    shell_test = np.full(len(q_test), -1)
else:
    w_halo = np.zeros(0); HALO_W_RESCALE = 1.0
    q_fit, I_fit, w_fit = q_train, I_train, w_train_mid
    is_halo = np.zeros(len(q_train), bool); shell_test = np.full(len(q_test), -1)
if FIT_MODE != 'halo':
    refl_fit = np.concatenate([-1 - np.arange(len(q_fit) - len(hkl_halo)), refl_halo])
ess = effective_sample_size(w_fit)
print(f"Objective: {len(q_fit)} observations ({int((~is_halo).sum())} mid-zone + {int(is_halo.sum())} halo)")
print(f"  effective sample size {ess:.0f}; free parameters {N_PARAMS}")
if ess < 5*N_PARAMS:
    print("*** WARNING: fewer than 5 effective observations per parameter. ***")

G_fit = G_at_hkl_batch(q_fit[:, 0], q_fit[:, 1], q_fit[:, 2])
G_test = G_at_hkl_batch(q_test[:, 0], q_test[:, 1], q_test[:, 2]) if len(q_test) else np.zeros((0, N_DOF), complex)
print(f"cached G(q) for {len(G_fit)} fitting and {len(G_test)} held-out points ({G_fit.nbytes/1e6:.0f} MB)")

## Where the Fitted q-Points Sit

Left: the fitting and held-out voxels in Cartesian $\mathbf q$. Right: the measured map
on the grid layers $l=0$ and $l=\tfrac12$ (or the nearest layers of the data grid),
expanded from the asymmetric unit with the Laue group, with the voxels within one
grid step of each layer overlaid. Bragg voxels are blanked.

In [ ]:
def read_hk_plane(l_target, field='I', d_min=None):
    """Measured map on the data-grid layer nearest l_target, expanded over the Laue group.
    Returns h_pts, k_pts, plane (n_h, n_k; NaN where unmeasured or Bragg) and the l value."""
    d_min = D_READ_MIN if d_min is None else d_min
    l_val = np.round(l_target*P3)/P3
    hm = int(np.ceil(cell.a/d_min)); km = int(np.ceil(cell.b/d_min))
    h_pts = np.arange(-hm*P1, hm*P1 + 1)/P1; k_pts = np.arange(-km*P2, km*P2 + 1)/P2
    HH, KK = np.meshgrid(h_pts, k_pts, indexing='ij')
    hkl = np.column_stack([HH.ravel(), KK.ravel(), np.full(HH.size, l_val)])
    val = lookup_map(hkl, field)
    brg = lookup_map(hkl, 'bragg')
    val[brg == 1] = np.nan
    qn = np.linalg.norm(hkl @ B_recip.T, axis=1)
    val[(qn > 2*np.pi/d_min) | (qn < 2*np.pi/D_READ_MAX)] = np.nan
    return h_pts, k_pts, val.reshape(HH.shape), float(l_val)

h_pts_bg0, k_pts_bg0, I_bg0, l_layer_0 = read_hk_plane(0.0)
h_pts_bg05, k_pts_bg05, I_bg05, l_layer_05 = read_hk_plane(0.5)
SLAB = 1.0*grid_delta[2] + 1e-9
qc_fit, qc_test = q_fit @ B_recip.T, q_test @ B_recip.T

fig = plt.figure(figsize=(17, 5.2))
ax3d = fig.add_subplot(1, 3, 1, projection='3d')
_sub = np.random.default_rng(0).choice(len(qc_fit), min(len(qc_fit), 8000), replace=False)
ax3d.scatter(*qc_fit[_sub].T, s=2, alpha=0.3, color='steelblue', label='fitting')
if len(qc_test):
    ax3d.scatter(*qc_test[:3000].T, s=2, alpha=0.5, color='orangered', label='held out')
ax3d.set_xlabel('$q_x$'); ax3d.set_ylabel('$q_y$'); ax3d.set_zlabel('$q_z$')
ax3d.set_title('Sampled q-points (Å$^{-1}$)'); ax3d.legend(fontsize=8)

def overlay_panel(ax, h_pts, k_pts, I_bg, l_layer):
    pos = I_bg[np.isfinite(I_bg) & (I_bg > 0)]
    lo, hi = (np.percentile(pos, 2), np.percentile(pos, 98)) if len(pos) else (0, 1)
    ax.imshow(np.clip(I_bg, lo, hi).T, origin='lower', cmap='gray',
              extent=[h_pts[0], h_pts[-1], k_pts[0], k_pts[-1]], aspect='equal', alpha=0.9)
    for pts, color, label in [(q_fit, 'steelblue', 'fitting'), (q_test, 'orangered', 'held out')]:
        sel = np.abs(pts[:, 2] - l_layer) < SLAB
        ax.scatter(pts[sel, 0], pts[sel, 1], s=3, color=color, edgecolor='none', alpha=0.6,
                   label=f'{label} ({sel.sum()})')
    ax.set_xlabel('h'); ax.set_ylabel('k')
    ax.set_title(f'measured map, l = {l_layer:g} (points within ±{SLAB:.2f})')
    ax.legend(fontsize=7, loc='upper right')

overlay_panel(fig.add_subplot(1, 3, 2), h_pts_bg0, k_pts_bg0, I_bg0, l_layer_0)
overlay_panel(fig.add_subplot(1, 3, 3), h_pts_bg05, k_pts_bg05, I_bg05, l_layer_05)
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'sampled_q_points.png', dpi=150); plt.show()
_c4m = np.nanmax(np.abs(I_bg0 - np.rot90(I_bg0)))
print(f"4-fold check of the expanded l = 0 layer: max |I(h,k) - I(-k,h)| = {_c4m:.3g} (0: expansion consistent)")

## Refining $K$ to the Experimental Data

$$\mathcal L(\theta)=\sum_{\mathbf q}w(\mathbf q)\bigl[I_{\rm TDS}(\mathbf q;\theta)+b(|\mathbf q|)-I_{\rm obs}(\mathbf q)\bigr]^2
+\lambda\sum_o\bigl\|\log\bigl(K_{o,\rm prior}^{-1/2}K_oK_{o,\rm prior}^{-1/2}\bigr)\bigr\|_F^2,$$

the sum over orbits in the penalty, $\lambda$ times the squared geodesic distance to
the geometric prior.

**Isotropic background.** $b$ is piecewise linear in $|\mathbf q|$ on `N_BG_KNOTS`
knots spanning the $|\mathbf q|$ range of the fitting points (constant beyond them)
and may be negative. It enters linearly, so for every $\theta$ it is solved exactly
by weighted least squares and profiled out: the optimizer sees only the 147 stiffness
parameters, and the Jacobian of the profiled residual is $(I-UU^{\mathsf T})J$ with
$U$ an orthonormal basis of $\sqrt w\times$ the background basis. mdx2 maps are total
scattering, so $b$ absorbs the solvent and Compton scattering and any liquid-like
motion: everything isotropic that is not lattice TDS.

**Scale.** There is no separate scale factor. $D$ is linear in $K$, so $K\to K/s$
multiplies every $I_{\rm TDS}$ by $s$, and the overall magnitude of $K$ is fitted
against the map's own units. The prior's magnitude is calibrated to the data first.

**Gradient.** As in the synthetic notebook: with $x=D^{-1}G$, member $m$ of orbit $o$
gives $\partial I/\partial K_m=-\mathrm{Re}(\Delta x_m\Delta x_m^\dagger)$, summed over the
orbit's members in the local frame and chained through $K=LL^{\mathsf T}$. Because the
background is optimal for every $\theta$, the envelope theorem removes its derivative
from the gradient. Both are checked against finite differences.

**Optimizer.** Trust-region least squares (`OPTIMIZER = 'lsq'`) on the residual vector
(data residuals after the background, and $\sqrt\lambda\,\mathrm{vec}\log P_o$ for the
penalty) with the analytic Jacobian, as in the synthetic notebook.

**Order of operations.** Prior calibration, then selection of `LAMBDA_PRIOR` by
cross-validation, then the main refinement.

In [ ]:
# --- Smooth isotropic background b(|q|), profiled out ------------------------------
_q_bg_fit = np.linalg.norm(np.asarray(q_fit, float) @ B_recip.T, axis=1)
_BG_KNOTS = np.linspace(_q_bg_fit.min(), _q_bg_fit.max(), N_BG_KNOTS)
N_BG = N_BG_KNOTS if USE_BACKGROUND else 0

def q_magnitude(q_list):
    return np.linalg.norm(np.asarray(q_list, float) @ B_recip.T, axis=1)

def bg_basis(q_list):
    """(N, N_BG_KNOTS) piecewise-linear hat basis in |q|, constant beyond the end knots."""
    x = np.clip(q_magnitude(q_list), _BG_KNOTS[0], _BG_KNOTS[-1])
    j = np.clip(np.searchsorted(_BG_KNOTS, x, side='right') - 1, 0, len(_BG_KNOTS) - 2)
    t = (x - _BG_KNOTS[j])/(_BG_KNOTS[j + 1] - _BG_KNOTS[j])
    Bm = np.zeros((len(x), len(_BG_KNOTS))); ar = np.arange(len(x))
    Bm[ar, j] = 1.0 - t; Bm[ar, j + 1] += t
    return Bm

def _bg_svd(q_list, weights):
    A = np.sqrt(np.asarray(weights, float))[:, None]*bg_basis(q_list)
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    keep = s > 1e-10*s.max()
    return U[:, keep], s[keep], Vt[keep]

def bg_profile(resid, q_list, weights):
    """resid = I_TDS - I_obs. Returns sqrt(w)*(I_TDS + b - I_obs) for the optimal b, the
    coefficients of b, and U (None, None when USE_BACKGROUND is off)."""
    rd = np.sqrt(np.asarray(weights, float))*np.asarray(resid, float)
    if not USE_BACKGROUND:
        return rd, None, None
    U, s, Vt = _bg_svd(q_list, weights)
    c = U.T @ rd
    return rd - U @ c, -Vt.T @ (c/s), U

def background_at(q_list, beta):
    return np.zeros(len(q_list)) if beta is None else bg_basis(q_list) @ beta

def fit_background(K_lab, q_arr, G_arr, I_obs, weights):
    if not USE_BACKGROUND:
        return None
    q_arr = np.asarray(q_arr, float)
    It = diffuse_intensity_batch(K_lab, q_arr[:, 0], q_arr[:, 1], q_arr[:, 2], G_arr=np.asarray(G_arr))
    return bg_profile(It - np.asarray(I_obs), q_arr, weights)[1]

# --- Forward model, per-member derivatives (multi-body, as in the synthetic notebook) --
def _forward(theta, q_list, G_arr):
    Lmap = theta_to_Lmap(theta)
    K_local, K_lab = build_K(Lmap)
    qb = np.asarray(q_list, float) @ B_recip.T
    V = np.empty_like(G_arr)
    for s in range(0, len(qb), CHUNK_EVAL):
        sl = slice(s, s + CHUNK_EVAL)
        V[sl] = np.linalg.solve(dynamical_matrix_batch(qb[sl], unique, K_lab), G_arr[sl][:, :, None])[:, :, 0]
    I = np.real(np.sum(np.conj(G_arr)*V, axis=1))
    return Lmap, K_local, K_lab, qb, V, I

def _member_dx(V, qb, M):
    a, b = M['a'], M['b']
    ph = np.exp(1j*(qb @ M['Rn']))
    return V[:, 6*a:6*a + 6] @ M['Ta'].T - ph[:, None]*(V[:, 6*b:6*b + 6] @ M['Tb'].T)

def dI_dKlocal(V, qb, coeff=None):
    """dI/dK_local per orbit: per point (N, N_SHELL, 6, 6) if coeff is None, else summed
    with weights coeff (N_SHELL, 6, 6)."""
    S = np.zeros((N_SHELL, 6, 6)) if coeff is not None else np.zeros((len(qb), N_SHELL, 6, 6))
    for M in MEMBERS:
        W = _member_dx(V, qb, M)
        o = M['orbit']; nQ = len(M['Q'])
        if coeff is not None:
            S_lab = -np.real((W*coeff[:, None]).T @ W.conj())
            for Q in M['Q']:
                S[o] += Q.T @ S_lab @ Q/nQ
        else:
            S_lab = -np.real(W[:, :, None]*W.conj()[:, None, :])
            for Q in M['Q']:
                S[:, o] += (Q.T[None] @ S_lab @ Q[None])/nQ
    return S

def loss_and_grad(theta, q_list, G_list, I_obs, weights):
    G_arr = np.asarray(G_list); q_list = np.asarray(q_list, float)
    Lmap, K_local, K_lab, qb, V, Imodel = _forward(theta, q_list, G_arr)
    weights = np.asarray(weights, float)
    resid = Imodel - np.asarray(I_obs)
    if USE_BACKGROUND:
        rd, _, _ = bg_profile(resid, q_list, weights)
        resid = np.where(weights > 0, rd/np.sqrt(np.maximum(weights, 1e-300)), 0.0)
    data_loss = float(np.sum(weights*resid**2))
    S = np.zeros((N_SHELL, 6, 6))
    for s in range(0, len(qb), CHUNK_EVAL):
        sl = slice(s, s + CHUNK_EVAL)
        S += dI_dKlocal(V[sl], qb[sl], coeff=2.0*weights[sl]*resid[sl])
    dRdK = regularizer_prior_dK(K_local)
    grad_L = {}
    for ci, c in enumerate(shell_order):
        grad_L[c] = np.tril((S[ci] + S[ci].T) @ Lmap[c] + (dRdK[c] + dRdK[c].T) @ Lmap[c])
    return data_loss + regularizer_prior(K_local), Lmap_to_theta(grad_L)

def dI_dtheta(theta, q_list, G_list):
    """Exact Jacobian of the model TDS intensities, (N, N_PARAMS), and the intensities."""
    G_arr = np.asarray(G_list)
    Lmap, _, _, qb, V, I = _forward(theta, q_list, G_arr)
    J = np.zeros((len(qb), N_PARAMS))
    for s in range(0, len(qb), CHUNK_EVAL):
        sl = slice(s, s + CHUNK_EVAL)
        S = dI_dKlocal(V[sl], qb[sl])
        for ci, c in enumerate(shell_order):
            dL = 2.0*(S[:, ci] @ Lmap[c][None])
            J[sl, ci*N_TRIL:(ci + 1)*N_TRIL] = dL[:, TRIL_I, TRIL_J]
    return J, I

_TRIU_I, _TRIU_J = np.triu_indices(6, 1)

def _reg_residuals(K_local, Lmap, lam):
    """Regularizer residuals and their Jacobian w.r.t. theta (Daleckii-Krein)."""
    r = np.zeros(N_SHELL*N_TRIL); Jr = np.zeros((N_SHELL*N_TRIL, N_PARAMS))
    s = np.sqrt(lam)
    for ci, c in enumerate(shell_order):
        Minv = REG_PISQRT[c]
        P = Minv @ K_local[c] @ Minv; P = 0.5*(P + P.T)
        w, U = np.linalg.eigh(P)
        w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
        lw = np.log(w); logP = (U*lw) @ U.T
        rows = slice(ci*N_TRIL, (ci + 1)*N_TRIL)
        r[rows] = s*np.r_[np.diag(logP), np.sqrt(2)*logP[_TRIU_I, _TRIU_J]]
        dw = w[:, None] - w[None, :]
        same = np.abs(dw) <= 1e-12*np.maximum(w[:, None], w[None, :])
        Phi = np.where(same, 1.0/w[:, None], (lw[:, None] - lw[None, :])/np.where(same, 1.0, dw))
        L = Lmap[c]
        for a in range(N_TRIL):
            E = np.zeros((6, 6)); E[TRIL_I[a], TRIL_J[a]] = 1.0
            dP = Minv @ (E @ L.T + L @ E.T) @ Minv
            dlog = U @ (Phi*(U.T @ dP @ U)) @ U.T
            Jr[rows, ci*N_TRIL + a] = s*np.r_[np.diag(dlog), np.sqrt(2)*dlog[_TRIU_I, _TRIU_J]]
    return r, Jr

def residuals_and_jac(theta, q_list, G_list, I_obs, weights, lam=None, need_jac=True):
    """r(theta) with ||r||^2 = loss (background profiled out), and dr/dtheta."""
    lam = LAMBDA_PRIOR if lam is None else lam
    q_list = np.asarray(q_list, float)
    if need_jac:
        J, Imodel = dI_dtheta(theta, q_list, G_list)
    else:
        Imodel = _forward(theta, q_list, np.asarray(G_list))[5]
    r_data, _, U_bg = bg_profile(Imodel - np.asarray(I_obs), q_list, weights)
    Lmap = theta_to_Lmap(theta); K_local, _ = build_K(Lmap)
    r_reg, J_reg = _reg_residuals(K_local, Lmap, lam)
    if not need_jac:
        return np.r_[r_data, r_reg], None
    J_data = np.sqrt(np.asarray(weights, float))[:, None]*J
    if U_bg is not None:
        J_data -= U_bg @ (U_bg.T @ J_data)       # exact: the basis does not depend on theta
    return np.r_[r_data, r_reg], np.vstack([J_data, J_reg])

In [ ]:
# Gradient and Jacobian checks against central finite differences (background profiled).
rng_chk = np.random.default_rng(42)
theta_chk = Lmap_to_theta(L_PRIOR)
theta_chk = theta_chk + 0.05*np.abs(theta_chk).mean()*rng_chk.standard_normal(N_PARAMS)
n_chk = min(40, len(q_fit))
q_chk, G_chk, I_chk, w_chk = q_fit[:n_chk], G_fit[:n_chk], I_fit[:n_chk], w_fit[:n_chk]
loss0, grad0 = loss_and_grad(theta_chk, q_chk, G_chk, I_chk, w_chk)
_gscale = float(np.abs(grad0).max()); _tscale = float(np.abs(theta_chk).mean())
def _grad_err(idx, rel):
    h = rel*max(abs(theta_chk[idx]), _tscale)
    tp = theta_chk.copy(); tp[idx] += h; tm = theta_chk.copy(); tm[idx] -= h
    fd = (loss_and_grad(tp, q_chk, G_chk, I_chk, w_chk)[0] - loss_and_grad(tm, q_chk, G_chk, I_chk, w_chk)[0])/(2*h)
    return abs(fd - grad0[idx])/max(abs(fd), 1e-3*_gscale, 1e-300)
errs = [min(_grad_err(idx, r) for r in (1e-4, 1e-5, 1e-6, 1e-7)) for idx in rng_chk.choice(N_PARAMS, size=8, replace=False)]
print(f"Gradient check, max relative error over 8 random directions: {max(errs):.2e}")
assert max(errs) < 1e-3
_r0, _J0 = residuals_and_jac(theta_chk, q_chk, G_chk, I_chk, w_chk)
assert abs(_r0 @ _r0 - loss0) <= 1e-9*abs(loss0), "||r||^2 differs from the loss"
def _jac_err(idx, rel):
    h = rel*max(abs(theta_chk[idx]), _tscale)
    tp = theta_chk.copy(); tp[idx] += h; tm = theta_chk.copy(); tm[idx] -= h
    fd = (residuals_and_jac(tp, q_chk, G_chk, I_chk, w_chk, need_jac=False)[0]
          - residuals_and_jac(tm, q_chk, G_chk, I_chk, w_chk, need_jac=False)[0])/(2*h)
    return np.linalg.norm(fd - _J0[:, idx])/max(np.linalg.norm(fd), 1e-12)
_jerr = [min(_jac_err(idx, r) for r in (1e-4, 1e-5, 1e-6)) for idx in rng_chk.choice(N_PARAMS, size=6, replace=False)]
print(f"Residual Jacobian check, max relative error: {max(_jerr):.2e}"
      f"{'  (background profiled out)' if USE_BACKGROUND else ''}")
assert max(_jerr) < 1e-4

In [ ]:
# --- Convergence criteria and optimizers (as in the synthetic notebook) ----------------
def geodesic_step(theta_a, theta_b):
    Ka, _ = build_K(theta_to_Lmap(theta_a)); Kb, _ = build_K(theta_to_Lmap(theta_b))
    tot = 0.0
    for c in shell_order:
        Minv = _sqrtm_inv_sym(Ka[c]); P = Minv @ Kb[c] @ Minv
        w = np.linalg.eigvalsh(0.5*(P + P.T)); w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
        tot += float(np.sum(np.log(w)**2))
    return float(np.sqrt(tot))

BLOCK, MAX_BLOCK, LOSS_TOL, LOSS_WIN, GEO_TOL = 500, 80, 1.0, 5, 1e-2

def refine(theta_start, args, block=BLOCK, max_block=MAX_BLOCK,
           loss_tol=LOSS_TOL, loss_win=LOSS_WIN, geo_tol=GEO_TOL, verbose=True):
    """L-BFGS-B in warm-started segments (OPTIMIZER = 'lbfgs'); 'lsq' calls refine_lsq."""
    if OPTIMIZER == 'lsq':
        return refine_lsq(theta_start, args, verbose=verbose)
    theta = np.asarray(theta_start, float).copy(); hist = [theta.copy()]
    nit = nfev = 0; rows, why = [], "max_block"
    for b in range(max_block):
        res = minimize(loss_and_grad, theta, args=args, jac=True, method='L-BFGS-B',
                       options={'maxiter': block, 'maxfun': 4*block, 'ftol': 0.0, 'gtol': 0.0},
                       callback=lambda xk: hist.append(xk.copy()))
        step = geodesic_step(theta, res.x); theta = res.x; nit += res.nit; nfev += res.nfev
        K_local, _ = build_K(theta_to_Lmap(theta)); pen = regularizer_prior(K_local)
        rows.append((nit, res.fun, res.fun - pen, pen, step))
        if verbose:
            print(f"   {b + 1:3d} {nit:7d} {res.fun - pen:12.6g} {pen:9.4f} {step:10.3e}")
        if step < geo_tol:
            why = "geodesic"; break
        if len(rows) > loss_win and (rows[-1 - loss_win][1] - rows[-1][1]) < loss_tol:
            why = "loss"; break
    return theta, hist, nit, nfev, rows, why

from scipy.optimize import least_squares
OPTIMIZER    = 'lsq'
LSQ_MAX_NFEV = 300
LSQ_FTOL     = 1e-8

def refine_lsq(theta_start, args, max_nfev=None, verbose=True):
    """Minimize ||r||^2 = chi^2 + lambda sum d^2 with scipy's 'trf' solver."""
    q_list, G_list, I_obs, weights = args
    hist, cache, losses = [], {}, []
    def fun(th):
        key = th.tobytes()
        if key not in cache:
            cache.clear()
            cache[key] = residuals_and_jac(th, q_list, G_list, I_obs, weights, need_jac=False)[0]
        return cache[key]
    def jac(th):
        hist.append(th.copy()); r = fun(th); losses.append(float(r @ r))
        return residuals_and_jac(th, q_list, G_list, I_obs, weights)[1]
    res = least_squares(fun, np.asarray(theta_start, float), jac=jac, method='trf',
                        x_scale='jac', ftol=LSQ_FTOL, xtol=1e-12, gtol=1e-12,
                        max_nfev=max_nfev or LSQ_MAX_NFEV)
    theta = res.x
    hist = [np.asarray(theta_start, float).copy()] + hist[1:]
    if not np.array_equal(hist[-1], theta):
        hist.append(theta.copy())
    K_local, _ = build_K(theta_to_Lmap(theta)); pen = regularizer_prior(K_local)
    loss = 2.0*res.cost
    step = geodesic_step(hist[-2], theta) if len(hist) > 1 else 0.0
    nan = float('nan')
    rows = [(i, losses[i], nan, nan, geodesic_step(hist[i - 1], hist[i]) if i else 0.0)
            for i in range(min(len(losses), len(hist)))]
    rows.append((res.njev, loss, loss - pen, pen, step))
    if verbose:
        print(f"   trust-region least squares: {res.njev} iterations, {res.nfev} evaluations")
        print(f"   data loss {loss - pen:.6g}   penalty {pen:.4f}   last step {step:.3e}")
    return theta, hist, res.njev, res.nfev, rows, f"lsq: {res.message}"

def fit_stats(K_lab, q_arr, G_arr, I_arr, scale=1.0, bg=None):
    """R and CC of TDS (+ background bg) against the data, and the model intensities."""
    q_arr = np.asarray(q_arr, float)
    Imodel = scale*diffuse_intensity_batch(K_lab, q_arr[:, 0], q_arr[:, 1], q_arr[:, 2],
                                           G_arr=np.asarray(G_arr)) + background_at(q_arr, bg)
    R = np.sum(np.abs(Imodel - I_arr))/np.sum(np.abs(I_arr))
    CC = np.corrcoef(Imodel, I_arr)[0, 1] if len(I_arr) > 2 else np.nan
    return R, CC, Imodel

def geodesic_per_contact(theta_a, theta_b):
    Ka, _ = build_K(theta_to_Lmap(theta_a)); Kb, _ = build_K(theta_to_Lmap(theta_b))
    out = {}
    for c in shell_order:
        Minv = _sqrtm_inv_sym(Ka[c]); P = Minv @ Kb[c] @ Minv
        w = np.linalg.eigvalsh(0.5*(P + P.T)); w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
        out[c] = float(np.sqrt(np.sum(np.log(w)**2)))
    return out

def geodesic_total(theta_a, theta_b):
    return float(np.sqrt(sum(v**2 for v in geodesic_per_contact(theta_a, theta_b).values())))

THETA_PRIOR = Lmap_to_theta(L_PRIOR)

### Putting the Prior on the Data's Scale

$K\to K/s$ sends $I\to s\,I$, so the magnitude of $K$ is the magnitude of the
intensities, and the map is in mdx2's own units. The cell computes the weighted
least-squares scale $s$ of $s\,I_{\rm TDS}$ against the fitting data (jointly with the
background), divides `PRIOR_K_PER_PAIR` by $s$, and rebuilds the prior, its
regularizer factors and the starting point. The prior's shape is unchanged. The
background found here, `BG_PRIOR`, is used wherever the prior is compared with the
data.

**Units.** mdx2 scales the data to a reference but not to absolute $e^2$ per unit
cell, so the calibrated per-pair stiffness is in the map's units, not in
$k_BT/\mathrm{Å}^2$. Relative stiffnesses, the shape of the dispersion, and the ratios of
predicted ADPs are meaningful. Absolute stiffnesses and absolute ADPs are not, until
the map is put on an absolute scale.

In [ ]:
def _scale_of(K_lab, q_arr, G_arr, I_obs, weights):
    Im = diffuse_intensity_batch(K_lab, q_arr[:, 0], q_arr[:, 1], q_arr[:, 2], G_arr=np.asarray(G_arr))
    sw = np.sqrt(np.asarray(weights, float))
    cols = np.hstack([Im[:, None]] + ([bg_basis(q_arr)] if USE_BACKGROUND else []))
    x = np.linalg.lstsq(sw[:, None]*cols, sw*np.asarray(I_obs), rcond=None)[0]
    return float(x[0]), (x[1:] if USE_BACKGROUND else None)

_s_cal, _ = _scale_of(K_LAB_PRIOR, q_fit, G_fit, I_fit, w_fit)
assert _s_cal > 0, "the TDS model anti-correlates with the data after background removal"
print(f"prior predicts intensities {1/_s_cal:.4g}x the data at PRIOR_K_PER_PAIR = {PRIOR_K_PER_PAIR:g}"
      f"{f' (jointly with a {N_BG_KNOTS}-knot background)' if USE_BACKGROUND else ''}")
PRIOR_K_PER_PAIR = PRIOR_K_PER_PAIR/_s_cal
L_PRIOR, K_PRIOR_LOCAL = prior_Lmap(PRIOR_K_PER_PAIR)
_, K_LAB_PRIOR = build_K(L_PRIOR)
REG_PISQRT = {c: _sqrtm_inv_sym(K_PRIOR_LOCAL[c]) for c in shell_order}
THETA_PRIOR = Lmap_to_theta(L_PRIOR)
_s_chk, BG_PRIOR = _scale_of(K_LAB_PRIOR, q_fit, G_fit, I_fit, w_fit)
print(f"calibrated PRIOR_K_PER_PAIR = {PRIOR_K_PER_PAIR:.4g} (map units); residual scale {_s_chk:.4f} (must be ~1)")
_R0, _CC0, _ = fit_stats(K_LAB_PRIOR, q_fit, G_fit, I_fit, bg=BG_PRIOR)
print(f"prior on the fitting set after calibration: R = {_R0:.4f}, CC = {_CC0:.5f}")

### Choosing `LAMBDA_PRIOR`

For each $\lambda$ in `LAMBDA_SCAN`, walked from large to small with warm starts:
`N_FOLDS`-fold cross-validation on $R$ over a random subset of `N_CV_MAX` fitting
voxels, the folds grouped by halo (Laue orbit of $\mathbf G$) in the halo-only mode,
each fold's background taken from its training voxels; and a full fit to the
subset, for $\chi^2/\mathrm{dof}$ and the distance from the prior. The calibrated prior
itself ($\lambda\to\infty$) is scored on the same folds. The selected value is the
largest $\lambda$ within one standard error of the best $R_{\rm cv}$ (`LAMBDA_RULE =
'1se'`) or the best itself (`'min'`). The scan is cached in
`OUT_PREFIX + 'lambda_scan_cache.npz'`, keyed on the fitting points, weights, prior,
grid and settings. The observed intensities are not part of the key, so set
`FORCE_RESCAN = True` after any change that alters them alone.

In [ ]:
import hashlib
CV_CACHE_FILE = OUT_PREFIX + 'lambda_scan_cache.npz'
FORCE_RESCAN  = False
CACHE_SIG     = 8

def _quant(a):
    a = np.asarray(a, float)
    if a.size == 0:
        return a
    m = float(np.max(np.abs(a)))
    return np.round(a/(m if m > 0 else 1.0), CACHE_SIG)

def _digest(a):
    h = hashlib.sha1(np.ascontiguousarray(_quant(a)).tobytes())
    h.update(repr(np.asarray(a).shape).encode())
    return h.hexdigest()[:16]

def _scan_key(**components):
    def _numeric(v):
        if not isinstance(v, (np.ndarray, list, tuple)):
            return False
        try:
            return np.asarray(v).dtype.kind in 'fiu'
        except ValueError:
            return False
    d = {k: (_digest(v) if _numeric(v) else hashlib.sha1(repr(v).encode()).hexdigest()[:16])
         for k, v in sorted(components.items())}
    d['__all__'] = hashlib.sha1(repr(sorted(d.items())).encode()).hexdigest()[:16]
    return d

def _load_scan(key):
    if FORCE_RESCAN or not os.path.exists(CV_CACHE_FILE):
        return None
    try:
        z = np.load(CV_CACHE_FILE, allow_pickle=False)
        old = {k: str(v) for k, v in zip(z['key_names'], z['key_values'])}
    except Exception as e:
        print(f"cache unreadable ({e}) -- rescanning"); return None
    if old.get('__all__') == key['__all__']:
        return [tuple(r) for r in z['scan']]
    print("cache found but these inputs differ:", [k for k in key if k != '__all__' and old.get(k) != key[k]])
    return None

def _save_scan(key, scan):
    np.savez(CV_CACHE_FILE, key_names=np.array(list(key.keys())), key_values=np.array(list(key.values())),
             scan=np.array(scan, float))
    print(f"scan cached to {CV_CACHE_FILE}")

LAMBDA_SCAN = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0]   # walked from large to small
LAMBDA_RULE = '1se'       # '1se' or 'min'
N_FOLDS     = 3
SCAN_BLOCKS = 30
N_CV_MAX    = 6000        # voxels used in the scan (the main refinement uses all)

_saved_lambda = LAMBDA_PRIOR
_probe = {shell_order[0]: 1.05*K_PRIOR_LOCAL[shell_order[0]]}
LAMBDA_PRIOR = 1.0;  _p1 = regularizer_prior(_probe)
LAMBDA_PRIOR = 10.0; _p2 = regularizer_prior(_probe)
LAMBDA_PRIOR = _saved_lambda
assert abs(_p2/_p1 - 10.0) < 1e-9, "regularizer_prior is not picking up LAMBDA_PRIOR at call time"

_rng_cv = np.random.default_rng(11)
CV_GROUPED = (FIT_MODE == 'halo')
if CV_GROUPED and N_CV_MAX and len(q_fit) > N_CV_MAX:
    # subsample whole halos, so every fold still holds complete halos
    _ug = _rng_cv.permutation(np.unique(refl_fit))
    _csum = np.cumsum([np.sum(refl_fit == g) for g in _ug])
    _take = _ug[:max(N_FOLDS, int(np.searchsorted(_csum, N_CV_MAX)) + 1)]
    _cv_sub = np.where(np.isin(refl_fit, _take))[0]
else:
    _cv_sub = (np.sort(_rng_cv.choice(len(q_fit), N_CV_MAX, replace=False))
               if N_CV_MAX and len(q_fit) > N_CV_MAX else np.arange(len(q_fit)))
_q_cv, _G_cv, _I_cv, _w_cv = q_fit[_cv_sub], G_fit[_cv_sub], I_fit[_cv_sub], w_fit[_cv_sub]
if CV_GROUPED:
    _grp = refl_fit[_cv_sub]
    _fold = [np.where(np.isin(_grp, gs))[0] for gs in np.array_split(_rng_cv.permutation(np.unique(_grp)), max(N_FOLDS, 1))]
    print(f"folds grouped by halo: {len(np.unique(_grp))} halos, fold sizes {[len(f) for f in _fold]}")
else:
    _fold = np.array_split(_rng_cv.permutation(len(_cv_sub)), max(N_FOLDS, 1))
if len(_cv_sub)*(1 - 1/N_FOLDS) < 5*N_PARAMS:
    print(f"*** WARNING: each fold trains on ~{int(len(_cv_sub)*(1 - 1/N_FOLDS))} voxels for {N_PARAMS} parameters. ***")
_WARM = {}

def cv_score(lam):
    global LAMBDA_PRIOR
    LAMBDA_PRIOR = lam
    Rs, CCs = [], []
    for _k, f in enumerate(_fold if N_FOLDS > 1 else []):
        te = np.zeros(len(_q_cv), bool); te[f] = True
        th, _, _, _, _, _ = refine(_WARM.get(_k, THETA_PRIOR), (_q_cv[~te], _G_cv[~te], _I_cv[~te], _w_cv[~te]),
                                   max_block=SCAN_BLOCKS, verbose=False)
        _WARM[_k] = th
        _, Kl = build_K(theta_to_Lmap(th))
        _bgk = fit_background(Kl, _q_cv[~te], _G_cv[~te], _I_cv[~te], _w_cv[~te])
        R, CC, _ = fit_stats(Kl, _q_cv[te], _G_cv[te], _I_cv[te], bg=_bgk)
        Rs.append(R); CCs.append(CC)
    th, _, nit, _, rw, _ = refine(_WARM.get('full', THETA_PRIOR), (_q_cv, _G_cv, _I_cv, _w_cv),
                                  max_block=SCAN_BLOCKS, verbose=False)
    _WARM['full'] = th
    _, Klab = build_K(theta_to_Lmap(th))
    if N_FOLDS <= 1:
        R, CC, _ = fit_stats(Klab, q_test, G_test, I_test, bg=fit_background(Klab, _q_cv, _G_cv, _I_cv, _w_cv))
        Rs, CCs = [R], [CC]
    chi2 = rw[-1][2]/max(len(_q_cv) - N_PARAMS - N_BG, 1)
    return (np.mean(Rs), np.std(Rs), np.mean(CCs), chi2, geodesic_total(THETA_PRIOR, th), nit, rw[-1][4])

print(f"{N_FOLDS}-fold cross-validation over {len(_cv_sub)} of {len(q_fit)} voxels\n")
print(f"{'lambda':>8s} {'chi2/dof':>9s} {'R_cv':>8s} {'±':>7s} {'CC_cv':>9s} {'prior→fit':>10s} {'iters':>7s} {'last step':>10s}")
_KEY = _scan_key(q_points=_q_cv, weights=_w_cv, prior=THETA_PRIOR, lambdas=list(LAMBDA_SCAN),
                 bg_knots=np.asarray(_BG_KNOTS if USE_BACKGROUND else [], float),
                 settings=(N_FOLDS, SCAN_BLOCKS, OPTIMIZER, LSQ_MAX_NFEV, LSQ_FTOL, N_CV_MAX, USE_BACKGROUND,
                           N_BG_KNOTS, FIT_MODE, CV_GROUPED, str(MDX2_FILE), DATA_COLUMN, M_REINDEX.tolist()))
_scan = _load_scan(_KEY)
if _scan is not None:
    print(f"loaded cached scan for {len(_scan)} lambda values (FORCE_RESCAN = True redoes it)")
    for _r in sorted(_scan, key=lambda r: -r[0]):
        print(f"{_r[0]:8.3g} {_r[4]:9.4f} {_r[1]:8.4f} {_r[2]:7.4f} {_r[3]:9.5f} {_r[5]:10.3f} {int(_r[6]):7d} {_r[7]:10.2e}")
else:
    _scan = []
    for _lam in sorted(LAMBDA_SCAN, reverse=True):
        r = cv_score(_lam); _scan.append((_lam,) + r)
        print(f"{_lam:8.3g} {r[3]:9.4f} {r[0]:8.4f} {r[1]:7.4f} {r[2]:9.5f} {r[4]:10.3f} {r[5]:7d} {r[6]:10.2e}", flush=True)
    _save_scan(_KEY, _scan)
LAMBDA_PRIOR = _saved_lambda
_scan.sort(key=lambda row: row[0])

_Rb, _CCb = [], []
for f in (_fold if N_FOLDS > 1 else []):
    te = np.zeros(len(_q_cv), bool); te[f] = True
    _bgk = fit_background(K_LAB_PRIOR, _q_cv[~te], _G_cv[~te], _I_cv[~te], _w_cv[~te])
    R, CC, _ = fit_stats(K_LAB_PRIOR, _q_cv[te], _G_cv[te], _I_cv[te], bg=_bgk)
    _Rb.append(R); _CCb.append(CC)
R_CV_PRIOR = float(np.mean(_Rb)) if _Rb else np.nan
print(f"{'prior':>8s} {'':9s} {R_CV_PRIOR:8.4f} {np.std(_Rb) if _Rb else np.nan:7.4f} "
      f"{np.mean(_CCb) if _CCb else np.nan:9.5f}   (λ → ∞: calibrated prior, not refit)")

_Rs = np.array([r[1] for r in _scan]); _ses = np.array([r[2] for r in _scan])/max(np.sqrt(N_FOLDS), 1.0)
_i = int(np.argmin(_Rs)); _thr = _Rs[_i] + _ses[_i]
LAMBDA_PRIOR = float(max(r[0] for r in _scan if r[1] <= _thr)) if LAMBDA_RULE == '1se' else float(_scan[_i][0])
print(f"\nbest R_cv = {_Rs[_i]:.4f} at lambda = {_scan[_i][0]:g} (standard error {_ses[_i]:.4f})")
print(f"rule '{LAMBDA_RULE}' -> LAMBDA_PRIOR = {LAMBDA_PRIOR:g}")
if np.isfinite(R_CV_PRIOR) and R_CV_PRIOR <= _thr:
    print("NOTE: the calibrated prior is within one standard error of the best refit: cross-validation")
    print("does not show that refining K improves prediction of held-out halos.")
if LAMBDA_PRIOR in (max(LAMBDA_SCAN), min(LAMBDA_SCAN)):
    print(f"The selection is at the edge of LAMBDA_SCAN ({LAMBDA_PRIOR:g}); extend the grid.")

_fig, _ax = plt.subplots(1, 3, figsize=(15, 4))
_l = [r[0] for r in _scan]
_ax[0].errorbar(_l, _Rs, yerr=_ses, fmt='o-', capsize=3)
_ax[0].axvline(LAMBDA_PRIOR, color='g', lw=1.5, label='selected')
_ax[0].axhline(_thr, color='r', ls=':', lw=1, label='best + 1 s.e.')
if np.isfinite(R_CV_PRIOR):
    _ax[0].axhline(R_CV_PRIOR, color='0.4', ls='--', lw=1, label=r'calibrated prior ($\lambda\to\infty$)')
_ax[0].set_xscale('log'); _ax[0].set_xlabel(r'$\lambda$'); _ax[0].set_ylabel('cross-validated $R$')
_ax[0].set_title('Generalization'); _ax[0].legend(fontsize=8)
_ax[1].loglog(_l, [max(r[5], 1e-6) for r in _scan], 'o-'); _ax[1].axvline(LAMBDA_PRIOR, color='g', lw=1.5)
_ax[1].set_xlabel(r'$\lambda$'); _ax[1].set_ylabel('geodesic prior→fit'); _ax[1].set_title('How far the fit moves')
_ax[2].semilogx(_l, [r[4] for r in _scan], 'o-'); _ax[2].axvline(LAMBDA_PRIOR, color='g', lw=1.5)
_ax[2].axhline(1, color='0.5', ls='--', lw=1)
_ax[2].set_xlabel(r'$\lambda$'); _ax[2].set_ylabel(r'$\chi^2$/dof (scan subset)'); _ax[2].set_title('Misfit')
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'lambda_scan.png', dpi=150); plt.show()

### Refinement at the Selected $\lambda$

In [ ]:
print(f"Refining at LAMBDA_PRIOR = {LAMBDA_PRIOR:g} on {len(q_fit)} voxels")
theta_fit, theta_history, NIT, NFEV, _rows, _why = refine(THETA_PRIOR, (q_fit, G_fit, I_fit, w_fit))
_, K_LAB_FIT = build_K(theta_to_Lmap(theta_fit))
BG_FIT = fit_background(K_LAB_FIT, q_fit, G_fit, I_fit, w_fit)
_step = _rows[-1][4]
print(f"stopped after {NIT} iterations ({NFEV} evaluations): {_why}")
print(f"  K moved {_step:.3e} in the last iteration (factor {np.exp(_step):.3f} at most on any eigenvalue)")
chi2_dof = _rows[-1][2]/max(len(q_fit) - N_PARAMS - N_BG, 1)
print(f"  weighted χ²/dof (training) = {chi2_dof:.4f}  (dof counts {N_BG} background coefficients)")
if chi2_dof > 2:
    print(f"  χ²/dof ≫ 1: the residuals are dominated by model error, not the quoted σ. On errors rescaled")
    print(f"  to χ²/dof = 1, LAMBDA_PRIOR = {LAMBDA_PRIOR:g} acts like λ = {LAMBDA_PRIOR/chi2_dof:.3g}.")
if 'function evaluations' in _why and _step > 10*GEO_TOL:
    print("NOT CONVERGED: LSQ_MAX_NFEV was reached while K was still moving; raise LSQ_MAX_NFEV.")

In [ ]:
R_fit, CC_fit, I_model_fit = fit_stats(K_LAB_FIT, q_fit, G_fit, I_fit, bg=BG_FIT)
R_free, CC_free, I_model_test = fit_stats(K_LAB_FIT, q_test, G_test, I_test, bg=BG_FIT)
_Rp_fit, _CCp_fit, _ = fit_stats(K_LAB_PRIOR, q_fit, G_fit, I_fit, bg=BG_PRIOR)
_Rp_free, _CCp_free, _Ip_test = fit_stats(K_LAB_PRIOR, q_test, G_test, I_test, bg=BG_PRIOR)
print(f"{'':12s} {'fit R':>7s} {'fit CC':>7s}   {'prior R':>7s} {'prior CC':>8s}")
print(f"fitting set: {R_fit:7.4f} {CC_fit:7.4f}   {_Rp_fit:7.4f} {_CCp_fit:8.4f}")
print(f"held out:    {R_free:7.4f} {CC_free:7.4f}   {_Rp_free:7.4f} {_CCp_free:8.4f}")
if USE_BACKGROUND:
    _bt = background_at(q_test, BG_FIT)
    _cc_aniso = np.corrcoef(I_model_test - _bt, I_test - _bt)[0, 1]
    print(f"held out, fitted TDS against data minus the fitted background: CC = {_cc_aniso:.4f}")
_wt = make_weights(sigma_test)
if FIT_MODE == 'halo':
    print(f"\nHeld-out halos by Bragg-distance shell (fit | prior):")
    print(f"  {'shell (Å⁻¹)':>16s} {'voxels':>7s} {'CC':>7s} {'χ²/N':>9s} | {'CC':>7s} {'χ²/N':>9s}")
    for _s in range(N_HALO_SHELLS):
        _m = shell_test == _s
        if _m.sum() < 5:
            continue
        _c = [np.corrcoef(Im[_m], I_test[_m])[0, 1] for Im in (I_model_test, _Ip_test)]
        _x = [np.sum(_wt[_m]*(Im[_m] - I_test[_m])**2)/_m.sum() for Im in (I_model_test, _Ip_test)]
        print(f"  {HALO_SHELL_EDGES[_s]:7.4f}–{HALO_SHELL_EDGES[_s + 1]:7.4f} {_m.sum():7d} {_c[0]:7.3f} {_x[0]:9.2f} | "
              f"{_c[1]:7.3f} {_x[1]:9.2f}")
print("\nR → 1 with CC → 0 means the model predicts essentially nothing; R → 1 with CC substantial")
print("means a scale or shape mismatch.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, Im, Io, title in [(axes[0], I_model_fit, I_fit, 'fitting set'), (axes[1], I_model_test, I_test, 'held-out halos')]:
    _sub = np.random.default_rng(0).choice(len(Io), min(len(Io), 6000), replace=False)
    ax.scatter(Io[_sub], Im[_sub], s=4, alpha=0.3)
    lim = [min(Io.min(), Im.min()), max(np.percentile(Io, 99.9), Im.max())]
    ax.plot(lim, lim, 'k--', lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(r'$I_{\rm obs}$ (mdx2)'); ax.set_ylabel(r'$I_{\rm model}$ (TDS + background)'); ax.set_title(title)
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'refinement_fit_quality.png', dpi=150); plt.show()

### The Fitted Isotropic Background

Mean observed intensity, mean fitted TDS, TDS plus background, and $b$ itself for the
fit and for the calibrated prior, in shells of $|\mathbf q|$ over the fitting voxels.

In [ ]:
if USE_BACKGROUND:
    _qa = q_magnitude(q_fit)
    _Itds = I_model_fit - background_at(q_fit, BG_FIT)
    _ed = np.linspace(_BG_KNOTS[0], _BG_KNOTS[-1], 16)
    _qc, _mo, _mt, _mb = [], [], [], []
    for a, b_ in zip(_ed[:-1], _ed[1:]):
        m = (_qa >= a) & (_qa < b_)
        if m.sum() < 10:
            continue
        _qc.append(0.5*(a + b_)); _mo.append(I_fit[m].mean()); _mt.append(_Itds[m].mean()); _mb.append(I_model_fit[m].mean())
    _qg = np.linspace(_BG_KNOTS[0], _BG_KNOTS[-1], 200)
    _qg3 = np.c_[_qg, np.zeros((len(_qg), 2))] @ np.linalg.inv(B_recip).T
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(_qc, _mo, 'ko', ms=5, label=r'mean $I_{\rm obs}$')
    ax.plot(_qc, _mt, 's-', color='tab:blue', ms=4, label='mean fitted TDS')
    ax.plot(_qc, _mb, '^-', color='tab:purple', ms=4, label='mean TDS + background')
    ax.plot(_qg, background_at(_qg3, BG_FIT), '-', color='tab:orange', lw=2, label='background $b$ (fit)')
    ax.plot(_qg, background_at(_qg3, BG_PRIOR), '--', color='tab:orange', lw=1.2, label='background $b$ (prior)')
    ax.axhline(0, color='0.6', lw=0.8)
    ax.set_xlabel(r'$|\mathbf{q}|$ (Å$^{-1}$)'); ax.set_ylabel('intensity (map units)')
    ax.set_title('Isotropic background vs. lattice TDS (fitting voxels)'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_PREFIX + 'isotropic_background.png', dpi=150); plt.show()
    _var = lambda x: float(np.sum(w_fit*x**2))
    _res_tds = I_fit - _Itds
    print(f"weighted variance of the data about the TDS explained by the background: "
          f"{1 - _var(_res_tds - background_at(q_fit, BG_FIT))/_var(_res_tds):.1%}")
    print(f"background / mean intensity at the knots: {np.round(BG_FIT/np.mean(I_fit), 2)}")

### Is the Misfit a Per-Halo Amplitude?

With $K$ fixed at the fit, each halo (Laue orbit of $\mathbf G$) gets one scale
$s_{\mathbf G}$ on the TDS by weighted least squares, the background kept. $\chi^2/N$
with the quoted σ is reported as fitted, with the scale fitted on all of the halo's
voxels, and **split-half** (scale from one random half of each halo's voxels, scored
on the other), so the extra freedom cannot lower $\chi^2$ by fitting noise. A large
split-half drop means the remaining misfit is mostly a per-reflection amplitude:
the effective $|F_{\rm cell}(\mathbf G)|$ of 7tx0 against this crystal, Bragg-tail
contamination, or scaling. The scale's rank correlations with resolution, with the
7tx0 $|F_{\rm meas}(\mathbf G)|$ and with the predicted halo $I/\sigma$ point at the cause.

In [ ]:
HALO_SCALE_MIN_VOX = 4
def _rank_corr(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 5:
        return np.nan
    return float(np.corrcoef(np.argsort(np.argsort(a[m])), np.argsort(np.argsort(b[m])))[0, 1])

_Fm_lookup = dict(zip(laue_orbit_key(mil_u).tolist(), Fm_u.tolist()))

def per_halo_test(q, G, I, sig, K, bg, seed=17):
    w = make_weights(sig)
    I_tot = fit_stats(K, q, G, I, bg=bg)[2]
    b = background_at(q, bg); t, y = I_tot - b, I - b
    refl = laue_orbit_key(np.round(q).astype(int))
    keys, inv = np.unique(refl, return_inverse=True); inv = np.asarray(inv).ravel()
    nvox = np.bincount(inv)
    half = np.random.default_rng(seed).random(len(q)) < 0.5
    def _scale(mask):
        num = np.bincount(inv, weights=np.where(mask, w*t*y, 0.0), minlength=len(keys))
        den = np.bincount(inv, weights=np.where(mask, w*t*t, 0.0), minlength=len(keys))
        return np.where(den > 0, num/np.where(den > 0, den, 1.0), np.nan)
    s_all, s_A, s_B = _scale(np.ones(len(q), bool)), _scale(half), _scale(~half)
    s_x = np.where(half, s_B[inv], s_A[inv])
    ok = (nvox[inv] >= HALO_SCALE_MIN_VOX) & np.isfinite(s_x)
    chi = {'as fitted': w*(t - y)**2, 'in-sample': w*(s_all[inv]*t - y)**2, 'split-half': w*(s_x*t - y)**2}
    hkl_G = _decode_key(keys)
    d_G = 2*np.pi/np.maximum(np.linalg.norm(hkl_G @ B_recip.T, axis=1), 1e-12)
    F_G = np.array([_Fm_lookup.get(k, np.nan) for k in keys.tolist()])
    snr_G = np.bincount(inv, weights=t/sig)/np.maximum(nvox, 1)
    good_h = (nvox >= HALO_SCALE_MIN_VOX) & np.isfinite(s_all) & (s_all > 0)
    return dict(chi=chi, ok=ok, s=s_all, good=good_h, d=d_G, F=F_G, snr=snr_G, n_halo=int(good_h.sum()))

_sets = {'fitting halos': (q_fit[is_halo], G_fit[is_halo], I_fit[is_halo], sig_halo, shell_halo)}
if FIT_MODE == 'halo' and len(q_test):
    _sets['held-out halos'] = (q_test, G_test, I_test, sigma_test, shell_test)
_res = {}
for lab, (q, G, I, sg, sh) in _sets.items():
    r = per_halo_test(q, G, I, sg, K_LAB_FIT, BG_FIT); rp = per_halo_test(q, G, I, sg, K_LAB_PRIOR, BG_PRIOR)
    r['shell'] = sh; _res[lab] = r; ok = r['ok']
    c0, cx = np.mean(r['chi']['as fitted'][ok]), np.mean(r['chi']['split-half'][ok])
    p0, px = np.mean(rp['chi']['as fitted'][ok]), np.mean(rp['chi']['split-half'][ok])
    print(f"\n{lab}: {ok.sum()} voxels in {r['n_halo']} halos")
    print(f"  χ²/N as fitted {c0:.2f}, in-sample {np.mean(r['chi']['in-sample'][ok]):.2f}, split-half {cx:.2f}")
    print(f"  one scale per halo removes {100*(1 - cx/c0):.0f}% of χ² (split-half) at the fit, "
          f"{100*(1 - px/p0):.0f}% at the prior (χ²/N {p0:.1f} → {px:.1f})")
    g = r['good']; ls = np.log(r['s'][g])
    print(f"  per-halo scale: median {np.exp(np.median(ls)):.3f}, central 68% "
          f"{np.exp(np.percentile(ls, 16)):.2f}–{np.exp(np.percentile(ls, 84)):.2f}")
    print(f"  rank correlation of ln s with: d {_rank_corr(ls, r['d'][g]):+.2f}, |F_meas(7tx0)| "
          f"{_rank_corr(ls, r['F'][g]):+.2f}, predicted halo I/σ {_rank_corr(ls, r['snr'][g]):+.2f}")

_lab = 'held-out halos' if 'held-out halos' in _res else 'fitting halos'
r = _res[_lab]; g = r['good']
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
_sc = 0.5*(HALO_SHELL_EDGES[:-1] + HALO_SHELL_EDGES[1:])
for k, _sty in (('as fitted', 'o-'), ('in-sample', 's--'), ('split-half', '^-')):
    ax[0].plot(_sc, [np.mean(r['chi'][k][r['ok'] & (r['shell'] == s)]) if (r['ok'] & (r['shell'] == s)).sum() >= 5
                     else np.nan for s in range(N_HALO_SHELLS)], _sty, label=k)
ax[0].set_yscale('log'); ax[0].set_xlabel('distance to the Bragg point (Å⁻¹)')
ax[0].set_ylabel('χ²/N (quoted σ)'); ax[0].set_title(f'{_lab}: one scale per halo'); ax[0].legend(fontsize=8)
ax[1].hist(np.log2(r['s'][g]), bins=40, color='0.5'); ax[1].axvline(0, color='k', lw=0.8)
ax[1].set_xlabel('log$_2$ (per-halo scale)'); ax[1].set_ylabel('halos'); ax[1].set_title('Per-halo amplitude correction')
ax[2].scatter(r['d'][g], r['s'][g], s=8, alpha=0.5); ax[2].axhline(1, color='k', lw=0.8)
ax[2].set_yscale('log'); ax[2].invert_xaxis()
ax[2].set_xlabel('resolution of the reflection $d$ (Å)'); ax[2].set_ylabel('per-halo scale'); ax[2].set_title('Scale vs. resolution')
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'per_halo_amplitude_test.png', dpi=150); plt.show()

### Distance Travelled from the Prior

The geodesic distance $d_o$ from the calibrated prior to the fit for each orbit.
$e^{d_o}$ bounds the factor by which any stiffness eigenvalue of that orbit changed.

In [ ]:
_d_pf = geodesic_per_contact(THETA_PRIOR, theta_fit)
print(f"{'orbit':>5s} {'label':>20s} {'pairs':>6s} {'prior→fit':>10s} {'factor e^d':>11s}")
for c in shell_order:
    print(f"{c:5d} {unique[c]['label']:>20s} {unique[c]['n_contacts']:6d} {_d_pf[c]:10.3f} {np.exp(_d_pf[c]):11.1f}")
_tot_pf = float(np.sqrt(sum(v**2 for v in _d_pf.values())))
print(f"{'total':>26s} {'':6s} {_tot_pf:10.3f} {np.exp(_tot_pf):11.1f}   (sqrt of the sum of squares)")

## How Many Parameters Does the Data Determine?

### The Jacobian

With $\chi^2=\sum_i w_i\,(I_i(\theta)-y_i)^2$ and $w_i=1/\sigma_i^2$, the
standardized residual is $r_i=(I_i-y_i)/\sigma_i$ and its Jacobian is

$$J_{i\alpha}=\frac{\partial r_i}{\partial\theta_\alpha}
=\sqrt{w_i}\,\frac{\partial I_i}{\partial\theta_\alpha}.$$

$J^{\mathsf T}J=\tfrac12\partial^2\chi^2/\partial\theta^2$ (Gauss-Newton) is the
Fisher information of the Gaussian likelihood. A displacement $\delta\theta$
costs $\Delta\chi^2\approx\delta\theta^{\mathsf T}(J^{\mathsf T}J)\,\delta\theta$.
$J$ depends on the model, the sampled $\mathbf q$ and the error bars, but not on
the observed values. `weighted_jacobian` computes it exactly. With the background on,
it is projected onto the complement of the background basis, so $J^{\mathsf T}J$ is the
information about $\theta$ left after the background has absorbed what it can.

### An invariant spectrum

The eigenvalues of $J^{\mathsf T}J$ change under a reparameterization
$\theta=A\phi$ ($J^{\mathsf T}J\to A^{\mathsf T}J^{\mathsf T}JA$). Here the
parameters mix units ($\sqrt{k_BT}/\mathrm{rad}$ and $\sqrt{k_BT}/\mathrm{Å}$),
so the raw spectrum is not a property of the problem. The generalized
eigenvalues of $(J^{\mathsf T}J,M)$ for a metric $M$ with the same
transformation law are invariant.

The metric is the affine-invariant metric on positive-definite matrices,
evaluated at the point $\theta$ where the spectrum is computed:

$$M(\theta)_{ab}=\sum_{\mathbf n}\mathrm{tr}\bigl(K_{\mathbf n}^{-1}\,\partial_aK_{\mathbf n}\,
K_{\mathbf n}^{-1}\,\partial_bK_{\mathbf n}\bigr),\qquad
\partial_aK=\partial_aL\,L^{\mathsf T}+L\,\partial_aL^{\mathsf T}.$$

`metric_at` computes it in closed form. It depends on where it is evaluated
(through $K(\theta)$) but not on $\lambda$ or the prior. `parameter_metric`, half
the Hessian of $\sum_{\mathbf n}d^2(K_{\mathbf n},K_{\mathbf n}^{\rm prior})$,
equals $M$ only at the prior. It is used for the curvature of the penalty in the
posterior.

$$g_i=\text{eigenvalues of }M^{-1}J^{\mathsf T}J$$

$g_i$ is the $\chi^2$ cost of moving one geodesic unit along direction $i$ (one
geodesic unit is a factor $e$ in one stiffness eigenvalue of one orbit).
Directions with $g>1$ are counted as measurable. Near the prior the posterior
precision is $J^{\mathsf T}J+\lambda M$, so $g_i/(g_i+\lambda)$ is the share of
the precision along direction $i$ that comes from the data.

`report_sloppiness` evaluates $J$ and $M$ at the fit, and plots the raw $J^{\mathsf T}J$ spectrum (for reference), the
$g$ spectrum with the $g=1$ and $g=\lambda$ lines, and the shrinkage
$g/(g+\lambda)$.

In [ ]:
# --- Invariant sloppiness analysis -----------------------------------------
def reg_grad_theta(theta, lam=1.0):
    """dR/dtheta for the geodesic penalty, through K = L L^T."""
    Lmap = theta_to_Lmap(theta)
    K_local, _ = build_K(Lmap)
    dRdK = regularizer_prior_dK(K_local, lam=lam)
    return Lmap_to_theta({c: np.tril((dRdK[c] + dRdK[c].T) @ Lmap[c])
                          for c in shell_order})

def parameter_metric(theta, eps=1e-5):
    """(1/2) d^2/dtheta^2 of the summed squared geodesic distance to the prior,
    by finite differences of the analytic gradient. lambda times this is the
    curvature of the penalty, used for the posterior. At theta = THETA_PRIOR it
    equals the affine-invariant metric; elsewhere it also contains curvature
    terms, so the spectrum uses metric_at(theta) instead."""
    n = len(theta); H = np.zeros((n, n))
    for j in range(n):
        tp = theta.copy(); tp[j] += eps
        tm = theta.copy(); tm[j] -= eps
        H[:, j] = (reg_grad_theta(tp, lam=1.0) - reg_grad_theta(tm, lam=1.0))/(2*eps)
    return 0.25*(H + H.T)          # (1/2) * symmetrized Hessian

def metric_at(theta):
    """Affine-invariant metric on stiffness space, pulled back to theta and
    evaluated AT theta (closed form):
        M_ab = sum_n tr(K_n^-1 dK_n/dtheta_a  K_n^-1 dK_n/dtheta_b),
    with dK/dtheta_a = E_a L^T + L E_a^T for the Cholesky entry a. Each orbit's
    21 parameters only move that orbit's K, so M is block-diagonal by orbit.
    Independent of LAMBDA_PRIOR and of the prior; equals parameter_metric(theta)
    at theta = THETA_PRIOR."""
    Lmap = theta_to_Lmap(theta)
    K_local, _ = build_K(Lmap)
    M = np.zeros((N_PARAMS, N_PARAMS))
    for ci, c in enumerate(shell_order):
        L, Ki = Lmap[c], np.linalg.inv(K_local[c])
        Z = np.empty((N_TRIL, 6, 6))
        for a in range(N_TRIL):
            E = np.zeros((6, 6)); E[TRIL_I[a], TRIL_J[a]] = 1.0
            Z[a] = Ki @ (E @ L.T + L @ E.T)                 # K^-1 dK_a
        s = slice(ci*N_TRIL, (ci+1)*N_TRIL)
        M[s, s] = np.einsum('aij,bji->ab', Z, Z)
    return 0.5*(M + M.T)

def weighted_jacobian(theta, q_list, G_list, weights, scale=1.0, eps=None, project=True):
    """J = d r / d theta for the standardized residual (I - y)/sigma (exact), with the
    profiled background projected out when USE_BACKGROUND."""
    J = scale*np.sqrt(np.asarray(weights, float))[:, None]*dI_dtheta(theta, q_list, G_list)[0]
    if USE_BACKGROUND and project:
        U, _, _ = _bg_svd(q_list, weights)
        J -= U @ (U.T @ J)
    return J

def invariant_spectrum(theta, q_list, G_list, weights, scale=1.0):
    """g = eig(M^-1 J^T J): chi^2 cost per squared geodesic displacement.
    Dimensionless, reparameterization-invariant, independent of LAMBDA_PRIOR."""
    Jm = weighted_jacobian(theta, q_list, G_list, weights, scale)
    F  = Jm.T @ Jm
    M  = metric_at(theta)
    Mi = _sqrtm_inv_sym(M)
    A  = Mi @ F @ Mi
    return np.linalg.eigvalsh(0.5*(A + A.T))[::-1], F, M, Jm

def report_sloppiness(theta, q_list, G_list, weights, scale=1.0,
                      fname=OUT_PREFIX + 'sloppiness_spectrum.png'):
    g, F, M, Jm = invariant_spectrum(theta, q_list, G_list, weights, scale)
    g = np.maximum(g, 0.0)
    ev_raw = np.linalg.eigvalsh(F)[::-1]
    n_meas = int(np.sum(g > 1.0))
    shrink = g/(g + LAMBDA_PRIOR)

    print(f"invariant spectrum g = Δχ² per squared geodesic unit")
    print(f"   g_max {g[0]:.4e}   g_min {g[-1]:.4e}   range {g[0]/max(g[-1],1e-300):.2e}")
    print(f"   measurable (g > 1):            {n_meas:3d} / {len(g)}")
    print(f"   recovered >90% at λ={LAMBDA_PRIOR:g}:  "
          f"{int(np.sum(shrink>0.9)):3d};  >50%: {int(np.sum(shrink>0.5)):3d}")
    print(f"\n   for reference, the raw J^T J spectrum spans "
          f"{ev_raw[0]/max(ev_raw[-1],1e-300):.1e} -- but that number changes under")
    print( "   any reparameterization and should not be used for counting.")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].semilogy(np.arange(1, len(ev_raw)+1),
                   np.maximum(ev_raw, 1e-300)/ev_raw[0], 'o-', ms=3, color='0.5')
    ax[0].set_xlabel('index'); ax[0].set_ylabel(r'$\lambda_i/\lambda_1$')
    ax[0].set_title('Raw $J^TJ$ (not invariant)')
    ax[1].semilogy(np.arange(1, len(g)+1), np.maximum(g, 1e-300), 'o-', ms=3)
    ax[1].axhline(1.0, color='r', ls='--', lw=1, label=r'$g=1$: measurable')
    ax[1].axhline(LAMBDA_PRIOR, color='g', ls=':', lw=1.4,
                  label=rf'$g=\lambda$ = {LAMBDA_PRIOR:g}')
    ax[1].axvline(n_meas+0.5, color='k', ls=':', lw=1)
    ax[1].set_xlabel('index'); ax[1].set_ylabel(r'$g$  ($\Delta\chi^2$ per geodesic unit$^2$)')
    ax[1].set_title(f'Invariant spectrum  ({n_meas} measurable)'); ax[1].legend(fontsize=8)
    ax[2].plot(np.arange(1, len(g)+1), shrink, 'o-', ms=3)
    ax[2].axhline(0.5, color='r', ls='--', lw=1)
    ax[2].set_ylim(-0.02, 1.02); ax[2].set_xlabel('index')
    ax[2].set_ylabel(r'recovered fraction $g/(g+\lambda)$')
    ax[2].set_title(rf'Shrinkage at $\lambda$ = {LAMBDA_PRIOR:g}')
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()
    return g, n_meas

In [ ]:
gen_spec, n_eff = report_sloppiness(theta_fit, q_fit, G_fit, w_fit)
print(f"\nAt the selected LAMBDA_PRIOR = {LAMBDA_PRIOR:g}, the data determines {n_eff} of {N_PARAMS}")
print("parameter combinations more tightly than one geodesic unit.")

In [ ]:
def bz_covariance(K_lab, n_grid=BZ_NGRID, offset=True, eig_floor=None, chunk=None):
    """Sigma = <u u^T> for all bodies of a cell (6 N_BODY x 6 N_BODY), from integrating
    D(q)^-1 over the Brillouin zone on an offset (Monkhorst-Pack) grid, with an
    absolute eigenvalue floor (see the P1 notebook). Block (b, b) is body b's 6x6
    covariance about its own centre of mass."""
    chunk = chunk or CHUNK_EVAL
    g = (np.arange(n_grid) + (0.5 if offset else 0.0))/n_grid
    Ig, Jg, Kg = np.meshgrid(g, g, g, indexing='ij')
    q_batch = np.column_stack([Ig.ravel(), Jg.ravel(), Kg.ravel()]) @ B_recip.T
    N = len(q_batch)
    if eig_floor is None:
        Dref = dynamical_matrix_batch(q_batch[:min(N, 2000)], unique, K_lab)
        eig_floor = 1e-9*mass_weighted_eigs(Dref).max()
    Sigma = np.zeros((N_DOF, N_DOF))
    for s0 in range(0, N, chunk):
        sl = slice(s0, min(s0 + chunk, N))
        Db = dynamical_matrix_batch(q_batch[sl], unique, K_lab)
        Dw = Msq_inv[None] @ Db @ Msq_inv.T[None]
        ev, evec = np.linalg.eigh(Dw)
        e_lab = Msq_inv.T[None] @ evec
        coeff = np.where(ev > eig_floor, 1.0/np.where(ev > eig_floor, ev, 1.0), 0.0)
        Sigma += np.real(np.tensordot(e_lab*coeff[:, None, :], e_lab.conj(), axes=([0, 2], [0, 2])))
    return Sigma/N

def rigid_jacobians(positions, com):
    """(m, 3, 6): maps a body's (Omega, v) about com to each atom's displacement."""
    d = np.asarray(positions) - com
    Jr = np.zeros((len(d), 3, 6))
    Jr[:, 0, 1] = d[:, 2]; Jr[:, 0, 2] = -d[:, 1]
    Jr[:, 1, 0] = -d[:, 2]; Jr[:, 1, 2] = d[:, 0]
    Jr[:, 2, 0] = d[:, 1]; Jr[:, 2, 1] = -d[:, 0]
    Jr[:, 0, 3] = Jr[:, 1, 4] = Jr[:, 2, 5] = 1.0
    return Jr

def body_block(Sigma, b, b2=None):
    b2 = b if b2 is None else b2
    return Sigma[6*b:6*b + 6, 6*b2:6*b2 + 6]

def atom_adp_tensors(Sigma6, positions, com):
    """Full anisotropic displacement tensor U (Å², 3x3) per atom of one body."""
    Jr = rigid_jacobians(positions, com)
    return Jr @ Sigma6[None] @ np.swapaxes(Jr, 1, 2)

def all_adp_tensors(Sigma):
    """U for every atom of the asymmetric-unit chains (concatenated in BODY_CHAINS order)."""
    return np.concatenate([atom_adp_tensors(body_block(Sigma, ASU_BODY[c]), CHAINS[c]['pos'], CHAINS[c]['com'])
                           for c in BODY_CHAINS])

def isotropic_B(U):
    return (8*np.pi**2/3)*np.trace(U, axis1=1, axis2=2)

def mean_predicted_B(K_lab, n_grid=BZ_NGRID):
    return isotropic_B(all_adp_tensors(bz_covariance(K_lab, n_grid=n_grid))).mean()

def check_bz_convergence(K_lab, grids=(8, 12, 16, BZ_NGRID, BZ_NGRID + 6)):
    """Sigma is the headline ADP prediction -- never report it without this."""
    print("BZ-grid convergence of the predicted mean B:")
    prev = None; Bs = []
    for n in grids:
        B = mean_predicted_B(K_lab, n_grid=n); Bs.append(B)
        delta = '' if prev is None else f'   Δ = {100*(B - prev)/prev:+.2f}%'
        print(f"  n_grid={n:3d}³ ({n**3:6d} q-points):  mean B = {B:8.3f} Å²{delta}")
        prev = B
    # The acoustic branches make the integrand ~1/q^2, so the offset-grid error falls as
    # 1/n. Extrapolating the last two grids in 1/n estimates the converged value.
    n1, n2 = grids[-2], grids[-1]
    B_inf = (n2*Bs[-1] - n1*Bs[-2])/(n2 - n1)
    print(f"  1/n extrapolation of the last two grids: mean B -> {B_inf:.3f} Å² "
          f"({100*(B_inf - Bs[-1])/Bs[-1]:+.1f}% from n_grid = {n2}; ratios between K are much better converged)")
    return B_inf

## What the Data Actually Determines

At each $\mathbf q$ the data measures

$$I(\mathbf q)=G^\dagger D^{-1}G=\sum_s\frac{|G^\dagger e_s|^2}{\omega_s^2}$$

(with $e_s$ the $M$-normalized eigenvectors), a single projection of $D^{-1}$
weighted by $1/\omega^2$. Stiff branches contribute little intensity, so the
stiff directions of $K$ are poorly constrained.

The displacement covariance
$\Sigma=\langle uu^{\mathsf T}\rangle=N^{-1}\sum_{\mathbf q}D(\mathbf q)^{-1}$
has the same $1/\omega^2$ weighting, integrated over the zone. It can therefore
be well determined even where $K$ is not.

A band structure compares 48 eigenvalues of $D(\mathbf q)$ at each $\mathbf q$,
out of 2304 real numbers. Matrices related by a unitary change of basis have the
same eigenvalues, so band structures can agree closely while the $K$ differ
substantially.

The cell below draws $\theta\sim\mathcal N(\hat\theta,\mathcal C)$ with
$\mathcal C=(J^{\mathsf T}J+\tfrac12\partial^2R/\partial\theta^2)^{-1}$
(inflated by $\chi^2/\mathrm{dof}$) and propagates the samples to three outputs:

1. The dispersion, with a 95% band. The band is centred on the fit, because
   sampling $\theta$ symmetrically makes $K=LL^{\mathsf T}$ biased stiff.
2. The predicted mean $B$.
3. The spread in $K$: the RMS geodesic distance of the samples from the fit, per
   orbit ($d_o$) and in total ($d$).

In [ ]:
# --- Propagate the posterior to physical observables -----------------------
# Covariance from the Gauss-Newton Hessian of the data term plus the prior's
# curvature. With chi^2 = sum w r^2 the Fisher information is J^T J, and treating
# the penalty R as -2 ln(prior) contributes R''/2, so C = (J^T J + H_R/2)^-1.
# Inflating by chi2/dof allows for model error on top of the quoted sigmas.

N_POST = 60          # posterior samples
INFLATE_BY_CHI2 = True

def posterior_cov_sqrt(theta, q_list, G_list, weights, scale=1.0, chi2_dof=1.0):
    Jm = weighted_jacobian(theta, q_list, G_list, weights, scale)
    H  = Jm.T @ Jm + LAMBDA_PRIOR*parameter_metric(theta)
    H  = 0.5*(H + H.T)
    w, V = np.linalg.eigh(H)
    w = np.maximum(w, 1e-12*w.max())
    f = np.sqrt(chi2_dof) if INFLATE_BY_CHI2 else 1.0
    return f*(V*w**-0.5)          # C^(1/2): samples = theta + Csqrt @ randn

_Csq = posterior_cov_sqrt(theta_fit, q_fit, G_fit, w_fit, chi2_dof=chi2_dof)
_rng_p = np.random.default_rng(3)
_samples = theta_fit[None, :] + (_Csq @ _rng_p.standard_normal((N_PARAMS, N_POST))).T

# --- 1. dispersion ---------------------------------------------------------
_bands = np.stack([bandstructure_freqs(build_K(theta_to_Lmap(t))[1]) for t in _samples])
# Centre the band on the fit. Sampling theta symmetrically does NOT give samples
# of K centred on K_fit: with K = L L^T, E[K] = K_fit + E[dL dL^T] > K_fit, so the
# ensemble is systematically stiffer than the fit and every nonlinear observable
# is biased. Quoting raw percentiles would report a band offset from the value it
# is supposed to bracket. The width is taken from the samples, the centre from
# the fit.
_med = np.median(_bands, axis=0)
_lo, _hi = np.percentile(_bands, [2.5, 97.5], axis=0)
_fit_b   = bandstructure_freqs(K_LAB_FIT)
_lo, _hi = _lo - _med + _fit_b, _hi - _med + _fit_b
print(f"Cholesky sampling bias, removed by centring: median sample frequency is "
      f"{100*np.nanmedian((_med - _fit_b)/np.maximum(_fit_b,1e-12)):+.1f}% off the fit")
_prior_b = bandstructure_freqs(K_LAB_PRIOR)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for s in range(N_DOF):
    ax[0].fill_between(x, _lo[:, s], _hi[:, s], color='steelblue', alpha=0.2, lw=0)
ax[0].plot(x, _prior_b, color='0.55', lw=0.8, ls='--')
ax[0].plot(x, _fit_b,  color='steelblue', lw=0.7)
ax[0].plot([], [], color='0.55', lw=0.8, ls='--', label='calibrated prior'); ax[0].plot([], [], color='steelblue', lw=0.7, label='fit')
for ti in tick_idx: ax[0].axvline(x[ti], color='k', lw=0.6)
ax[0].set_xticks([x[ti] for ti in tick_idx]); ax[0].set_xticklabels(path_labels)
ax[0].set_ylabel('Frequency (THz)'); ax[0].set_xlim(x[0], x[-1]); ax[0].set_ylim(0)
ax[0].set_title('Dispersion, 95% posterior band'); ax[0].legend(fontsize=8)

# at Γ the acoustic frequencies are zero, so the ratio is undefined there; mask those points
_rel = np.where(_fit_b > 1e-3*_fit_b.max(), 0.5*(_hi - _lo)/np.maximum(_fit_b, 1e-12), np.nan)
for s in range(N_DOF):
    ax[1].plot(x, 100*_rel[:, s], lw=1.0 if s < 3 else 0.5, color='crimson' if s < 3 else '0.6')
ax[1].plot([], [], color='crimson', label='three lowest branches'); ax[1].plot([], [], color='0.6', label='other branches')
for ti in tick_idx: ax[1].axvline(x[ti], color='k', lw=0.6)
ax[1].set_xticks([x[ti] for ti in tick_idx]); ax[1].set_xticklabels(path_labels)
ax[1].set_ylabel('half-width / frequency  (%)'); ax[1].set_yscale('log')
ax[1].set_title('Fractional uncertainty per branch'); ax[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'posterior_bands.png', dpi=150); plt.show()

_outside = np.mean((_prior_b < _lo) | (_prior_b > _hi))
print(f"calibrated prior outside the fit's 95% band: {_outside:.1%} of (q, branch) pairs "
      f"(how much the data moved the dispersion beyond its own uncertainty)")
print(f"Median fractional uncertainty, acoustic branches 0-2: "
      f"{100*np.nanmedian(_rel[:, :3]):.2f}%")
print(f"Median fractional uncertainty, optic branches 3-{N_DOF - 1}: "
      f"{100*np.nanmedian(_rel[:, 3:]):.2f}%")

# --- 2. displacement parameters -------------------------------------------
_B = np.array([mean_predicted_B(build_K(theta_to_Lmap(t))[1], n_grid=12)
               for t in _samples[:min(N_POST, 25)]])
_B_prior = mean_predicted_B(K_LAB_PRIOR, n_grid=12)
_B_fit = mean_predicted_B(K_LAB_FIT, n_grid=12)
print(f"\nMean rigid-body B (map units, see the calibration cell): fit {_B_fit:.4g}   prior {_B_prior:.4g}")
print(f"   posterior spread ±{np.std(_B):.3f} ({100*np.std(_B)/max(_B_fit,1e-12):.1f}% of the fit)")
print(f"   sample median {np.median(_B):.4g} -- the offset from the fit is the")
print( "   Cholesky sampling bias described above, not an uncertainty.")

# --- 3. the stiffnesses themselves ----------------------------------------
_dsamp = [geodesic_per_contact(theta_fit, t) for t in _samples[:30]]
_spread = {c: float(np.sqrt(np.mean([d[c]**2 for d in _dsamp]))) for c in shell_order}
_spread_tot = float(np.sqrt(np.mean([sum(v**2 for v in d.values()) for d in _dsamp])))
print(f"\nRMS geodesic distance of posterior samples from the fit:")
print(f"{'orbit':>20s} {'RMS distance':>13s} {'factor e^d':>11s}")
for c in shell_order:
    print(f"{unique[c]['label']:>20s} {_spread[c]:13.3f} {np.exp(_spread[c]):11.2f}")
print(f"{'total':>20s} {_spread_tot:13.3f} {np.exp(_spread_tot):11.2f}")
print("\nRead the three together. The comparison that matters is the fractional")
print("uncertainty on the dispersion and on the mean B against the per-orbit")
print("spread in K: where the observables are much better pinned than the")
print("stiffnesses, the data is measuring the soft dynamics rather than the springs.")

### Reading the Stiff and Sloppy Directions

The next cell computes the eigendirections at the fit.

**Which directions the data constrains.** The cell solves the
generalized eigenproblem $(F,M)$ with $F=J^{\mathsf T}J$ and
$M=$ `metric_at(theta_fit)`, both evaluated at the fit. This gives the invariant
spectrum $g$ ($\Delta\chi^2$ per squared geodesic step) and directions that are
orthonormal in the metric at the fit. For each direction it reports $g$, the data
share of the local precision $g/(g+\lambda)$, the composition, and the effect.
The stacked bar chart (`direction_block_composition.png`) shows the $g$
spectrum, with the $g=1$ and $g=\lambda$ lines, above the $RR/TR/TT$ split of
every direction sorted by $g$.

**Composition** (`direction_anatomy(direction, theta0)`). For each orbit,
$\delta K=\delta L\,L_0^{\mathsf T}+L_0\,\delta L^{\mathsf T}$ at the base point
$\theta_0$. Then $X=W\,\delta K\,W^{\mathsf T}$, where $W$ comes from the
translation-first block factorization of $K_0=K(\theta_0)$:

$$K_0=UDU^{\mathsf T},\quad
U=\begin{pmatrix}I&K_{RT}K_{TT}^{-1}\\0&I\end{pmatrix},\quad
D=\mathrm{diag}(S_R,K_{TT}),\quad S_R=K_{RR}-K_{RT}K_{TT}^{-1}K_{TR},\quad
W=D^{-1/2}U^{-1}.$$

With this $W$, $\sum_{\mathbf n}\|X_{\mathbf n}\|_F^2=\delta\theta^{\mathsf T}M(\theta_0)\,\delta\theta$
(the squared geodesic length at $\theta_0$), and
$\|X\|_F^2=\|X_{RR}\|_F^2+2\|X_{TR}\|_F^2+\|X_{TT}\|_F^2$ exactly. The blocks
mean:

* $X_{TT}=K_{TT}^{-1/2}\,\delta K_{TT}\,K_{TT}^{-1/2}$: the relative change in the
  translational stiffness;
* $X_{RR}$ and $X_{TR}$: the rotational and coupling changes, measured against
  $S_R$ (the rotational stiffness with translations relaxed), after removing the
  part explained by $\delta K_{TT}$.

The split is unchanged by moving the reference point, rotating the axes, or
changing units. Putting translations first is what makes it independent of the
reference point. A pure translation is the same motion whatever point it is
referred to, but what counts as a pure rotation changes with that point. At the
prior ($K_{TR}=0$) the split reduces to $X=K_{\rm prior}^{-1/2}\delta K\,K_{\rm prior}^{-1/2}$.

The fractions are computed in $K$, not in $\theta$, because the Cholesky entries
do not separate by block. The three blocks have 6, 9 and 6 independent components
per orbit, so a direction spread evenly gives $RR:TR:TT=6:9:6$ (out of 21).
The directions form a complete orthonormal basis in the metric, so the average
over all `N_PARAMS` reproduces that split exactly. The cell prints the average as a
check.

**Effect** (`direction_effect(direction, theta0=theta_fit)`). Each direction is
rescaled to a geodesic displacement of `EFFECT_D` about the fit, and the cell
reports the fractional change in:

* the acoustic frequencies near Γ;
* the optic frequencies at Γ;
* all 48 branches at $(\tfrac12,\tfrac12,\tfrac12)$;
* the mean predicted $B$.

**Expected pattern.** The well-determined directions should be mostly
translational and the poorly determined ones mostly rotational, for three
reasons:

1. $I\propto1/\omega^2$, and the librational branches are the stiff ones.
2. Near any reciprocal-lattice point the acoustic eigenvectors are nearly pure
   translations, so the halos couple mainly through the translational part of
   $G$.
3. The rotational coupling $\mathbf q\times L$ is weak at low $|\mathbf q|$,
   because the contrast centroid lies close to the centre of mass.

The $TR$ block should appear in both groups.

In [ ]:
# --- composition of a direction -------------------------------------------
def _tfirst_whitener(K):
    """W_K such that X = W_K dK W_K^T has ||X||_F^2 = tr(K^-1 dK K^-1 dK).

    Uses the translation-first block factorization K = U D U^T with
    U = [[I, K_RT K_TT^-1], [0, I]] and D = diag(S_R, K_TT), where
    S_R = K_RR - K_RT K_TT^-1 K_TR is the rotational stiffness with translations
    relaxed. Then W_K = D^{-1/2} U^{-1}, and the blocks of X are:
      X_TT = K_TT^{-1/2} dK_TT K_TT^{-1/2}    (relative change of K_TT)
      X_RR, X_RT: the rotational and coupling changes measured against S_R,
      after the part explained by dK_TT is removed.
    When K_TR = 0 this reduces to X = K^{-1/2} dK K^{-1/2}. The split does not
    depend on the reference point, because moving the reference point leaves
    K_TT, S_R and these blocks unchanged."""
    K_RT, K_TT = K[:3, 3:], K[3:, 3:]
    C = K_RT @ np.linalg.inv(K_TT)
    Uinv = np.eye(6); Uinv[:3, 3:] = -C
    S_R = K[:3, :3] - C @ K_RT.T
    Dmh = np.zeros((6, 6))
    Dmh[:3, :3] = _sqrtm_inv_sym(S_R); Dmh[3:, 3:] = _sqrtm_inv_sym(K_TT)
    return Dmh @ Uinv

def direction_anatomy(direction, theta0=None):
    """Where a theta-direction lives in K, measured in the geodesic metric at the
    base point theta0 (default THETA_PRIOR).

    For each contact, dK = dL L0^T + L0 dL^T (local frame, exact) and
    X = W dK W^T with W from _tfirst_whitener(K(theta0)). sum_n ||X_n||_F^2 is the
    squared geodesic length of the step at theta0 (direction^T metric_at(theta0)
    direction), and ||X||^2 = ||X_RR||^2 + 2||X_TR||^2 + ||X_TT||^2 exactly.
    A direction spread evenly over the 21 independent components of each K gives
    RR : TR : TT = 6 : 9 : 6 (out of 21)."""
    theta0 = THETA_PRIOR if theta0 is None else theta0
    L0, dLm = theta_to_Lmap(theta0), theta_to_Lmap(direction)
    K0, _ = build_K(L0)
    X = {}
    for c in shell_order:
        W = _tfirst_whitener(K0[c])
        X[c] = W @ (dLm[c] @ L0[c].T + L0[c] @ dLm[c].T) @ W.T
    tot = sum(np.sum(X[c]**2) for c in shell_order) or 1.0
    per_contact = {c: np.sum(X[c]**2)/tot for c in shell_order}
    blocks, iso = {}, {}
    for name, sl in [('RR', slice(0,3)), ('TT', slice(3,6))]:
        b = sum(np.sum(X[c][sl, sl]**2) for c in shell_order)
        t = sum((np.trace(X[c][sl, sl])**2)/3 for c in shell_order)
        blocks[name] = b/tot; iso[name] = t/max(b, 1e-300)
    blocks['TR'] = sum(2*np.sum(X[c][0:3,3:6]**2) for c in shell_order)/tot
    return per_contact, blocks, iso

# --- what a direction does, physically ------------------------------------
# Every direction is rescaled to the SAME geodesic distance before the effect is
# measured, so the comparison is between physical perturbations of equal size.
_q_ac = np.array([[0.04,0,0],[0,0.04,0],[0,0,0.04]]) @ B_recip.T   # near Gamma
_q_zb = (B_recip @ np.array([0.5,0.5,0.5]))[None, :]
_EFF_AC, _EFF_G, _EFF_ZB = slice(0, 9), slice(9, 9 + N_DOF - 3), slice(9 + N_DOF - 3, 9 + 2*N_DOF - 3)
EFFECT_D = 0.20        # geodesic distance at which every direction is compared

def scale_to_geodesic(direction, target_d=EFFECT_D, tol=0.01, theta0=None):
    """Amplitude along `direction` giving a geodesic displacement (geodesic_total)
    of target_d from the base point theta0 (default THETA_PRIOR)."""
    theta0 = THETA_PRIOR if theta0 is None else theta0
    lo, hi = 1e-6, 1e4
    for _ in range(80):
        a = np.sqrt(lo*hi)
        try:
            d = geodesic_total(theta0, theta0 + a*direction)
        except np.linalg.LinAlgError:
            hi = a; continue
        if abs(d - target_d) < tol*target_d: return a
        lo, hi = (a, hi) if d < target_d else (lo, a)
    return a

def direction_effect(direction, target_d=EFFECT_D, theta0=None):
    """Fractional change in physical observables for a displacement of fixed
    geodesic size along `direction`, centred on theta0 (default THETA_PRIOR).
    Branches are sorted ascending, so [:3] are acoustic and [3:] optic."""
    theta0 = THETA_PRIOR if theta0 is None else theta0
    a = scale_to_geodesic(direction, target_d, theta0=theta0)
    def obs(th):
        _, Kl = build_K(theta_to_Lmap(th))
        f_ac = bandstructure_freqs(Kl, _q_ac)[:, :3].ravel()   # acoustic near Gamma
        f_G  = bandstructure_freqs(Kl, np.zeros((1,3)))[0][3:] # optic at Gamma
        f_zb = bandstructure_freqs(Kl, _q_zb)[0]               # zone boundary, all branches
        B    = mean_predicted_B(Kl, n_grid=8)
        return np.r_[f_ac, f_G, f_zb, B]
    p, m = obs(theta0 + a*direction), obs(theta0 - a*direction)
    return np.abs(p-m)/np.maximum(0.5*np.abs(p+m), 1e-30)

# ============================================================================
# Directions at the FIT: which combinations of K does the data constrain there?
# ============================================================================
# Generalized eigenproblem (F, M) with both evaluated at theta_fit:
#   F = J^T J  (data curvature),  M = metric_at(theta_fit)  (geodesic metric).
# g_i = Delta chi^2 per squared geodesic step along direction i; g > 1 is
# measurable. g/(g + lambda) is the share of the local posterior precision along
# that direction that comes from the data rather than the prior.
_Jf  = weighted_jacobian(theta_fit, q_fit, G_fit, w_fit)
_Ff  = _Jf.T @ _Jf
_Mf  = metric_at(theta_fit)
_Mfi = _sqrtm_inv_sym(_Mf)
_Af  = _Mfi @ _Ff @ _Mfi
_gF, _VF = np.linalg.eigh(0.5*(_Af + _Af.T))
_oF = np.argsort(_gF)[::-1]
_gF, _VF = np.maximum(_gF[_oF], 0.0), _VF[:, _oF]
_DIRS_FIT = _Mfi @ _VF                    # M_fit-orthonormal, in theta coordinates
_share = _gF/(_gF + LAMBDA_PRIOR)
_nm = max(int(np.sum(_gF > 1)), 1)
print(f"\nAT THE FIT: {int(np.sum(_gF > 1))} of {N_PARAMS} directions measurable (g > 1); "
      f"data share g/(g+λ) > 50% for {int(np.sum(_share > 0.5))}, > 90% for "
      f"{int(np.sum(_share > 0.9))}  (λ = {LAMBDA_PRIOR:g})")

_probe_fit = sorted(set([0, 1, 2, _nm-1, _nm, N_PARAMS-3, N_PARAMS-2, N_PARAMS-1]))
print(f"\nComposition at the fit, and the effect of a displacement of geodesic size "
      f"{EFFECT_D} about the fit:")
print(f"{'dir':>4s} {'g':>9s} {'data %':>7s} {'RR':>5s} {'TR':>5s} {'TT':>5s} "
      f"{'iso(TT)':>8s} {'top orbit':>18s} {'Δν_ac':>7s} {'Δν_opt':>7s} "
      f"{'Δν_zb':>7s} {'ΔB':>7s}")
for i in _probe_fit:
    d = _DIRS_FIT[:, i]/np.linalg.norm(_DIRS_FIT[:, i])
    pc, bl, iso = direction_anatomy(d, theta_fit)
    eff = direction_effect(d, theta0=theta_fit)
    top = max(pc, key=pc.get)
    print(f"{i:4d} {_gF[i]:9.2e} {_share[i]:7.1%} {bl['RR']:5.2f} {bl['TR']:5.2f} "
          f"{bl['TT']:5.2f} {iso['TT']:8.2f} {unique[top]['label']:>18s} "
          f"{np.mean(eff[_EFF_AC]):7.1%} {np.mean(eff[_EFF_G]):7.1%} "
          f"{np.mean(eff[_EFF_ZB]):7.1%} {eff[-1]:7.1%}")
print("   Δν_ac  acoustic branches near Γ (the sound speeds)")
print("   Δν_opt optic frequencies at Γ (librations and relative translations of the bodies)")
print("   Δν_zb  all branches at the zone boundary")

# --- Block composition of every eigendirection at the fit -------------------
_anat_fit = [direction_anatomy(_DIRS_FIT[:, i], theta_fit) for i in range(N_PARAMS)]
_comp = np.array([[a[1]['RR'], a[1]['TR'], a[1]['TT']] for a in _anat_fit])
_pcs_all = np.array([[a[0][c] for c in shell_order] for a in _anat_fit])
_isoTT = np.array([a[2]['TT'] for a in _anat_fit])

print(f"\naveraged composition at the fit ({_nm} with g > 1 vs {N_PARAMS-_nm} with g <= 1):")
for _lab, _sl in [('g > 1', slice(0, _nm)), ('g <= 1', slice(_nm, N_PARAMS))]:
    _cm = _comp[_sl].mean(0); _pm = _pcs_all[_sl].mean(0)
    print(f"   {_lab:7s} RR {_cm[0]:.2f}   TR {_cm[1]:.2f}   TT {_cm[2]:.2f}   "
          f"isotropic fraction of TT {_isoTT[_sl].mean():.2f}")
    _top3 = np.argsort(_pm)[::-1][:3]
    print(f"   {'':7s} orbits: " + "  ".join(f"{unique[shell_order[j]]['label']} {_pm[j]:.2f}" for j in _top3))
print(f"   even split (reference): RR {6/21:.2f}   TR {9/21:.2f}   TT {6/21:.2f}")
# The directions form a complete M_fit-orthonormal basis, so averaged over all
# N_PARAMS the fractions must reproduce the even split exactly -- a check on the split.
print(f"   mean over all directions: RR {_comp[:,0].mean():.3f}  TR {_comp[:,1].mean():.3f}  "
      f"TT {_comp[:,2].mean():.3f}  (must equal {6/21:.3f} / {9/21:.3f} / {6/21:.3f})")

_ix = np.arange(1, N_PARAMS + 1)
_block_cols = [('RR', '#2a78d6'), ('TR', '#eb6834'), ('TT', '#1baf7a')]
fig, ax = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True,
                       gridspec_kw={'height_ratios': [1, 2.2], 'hspace': 0.08})
ax[0].semilogy(_ix, np.maximum(_gF, 1e-12), 'o-', ms=2.5, lw=1.0, color='0.3')
ax[0].axhline(1.0, color='0.55', ls='--', lw=1)
ax[0].axhline(LAMBDA_PRIOR, color='0.55', ls=':', lw=1)
ax[0].text(N_PARAMS + 1, 1.0, '$g=1$', va=('bottom' if LAMBDA_PRIOR <= 1 else 'top'), fontsize=8, color='0.25')
ax[0].text(N_PARAMS + 1, LAMBDA_PRIOR, '$g=\\lambda$', va=('top' if LAMBDA_PRIOR <= 1 else 'bottom'), fontsize=8, color='0.25')
ax[0].axvline(_nm + 0.5, color='0.2', ls=':', lw=1)
ax[0].set_ylabel('$g$ ($\\Delta\\chi^2$ per\ngeodesic unit$^2$)')
ax[0].set_title(f'Eigendirections at the fit: invariant spectrum and block composition '
                f'({int(np.sum(_gF > 1))} with $g>1$, left of the dotted line)')
_bottom = np.zeros(N_PARAMS)
for j, (name, col) in enumerate(_block_cols):
    ax[1].bar(_ix, _comp[:, j], bottom=_bottom, width=1.0, color=col,
              edgecolor='white', linewidth=0.5, label=name)
    _bottom += _comp[:, j]
for y in (6/21, 15/21):
    ax[1].axhline(y, color='0.15', ls='--', lw=0.8)
ax[1].text(N_PARAMS + 1, 3/21, 'RR\n6/21', va='center', fontsize=8, color='0.25')
ax[1].text(N_PARAMS + 1, 10.5/21, 'TR\n9/21', va='center', fontsize=8, color='0.25')
ax[1].text(N_PARAMS + 1, 18/21, 'TT\n6/21', va='center', fontsize=8, color='0.25')
ax[1].axvline(_nm + 0.5, color='0.2', ls=':', lw=1)
ax[1].set_xlim(0.5, N_PARAMS + 0.5); ax[1].set_ylim(0, 1)
ax[1].set_xlabel('eigendirection index at the fit (sorted by $g$, best determined first)')
ax[1].set_ylabel('fraction of squared\ngeodesic length')
ax[1].legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.16), frameon=False)
plt.savefig(OUT_PREFIX + 'direction_block_composition.png', dpi=150, bbox_inches='tight'); plt.show()

### Which Blocks of $K$ the Data Constrains

This answers the question of whether couplings involving rotations are better or
worse determined than the rotation–rotation block. At the fit, each orbit's $K$
is expressed in the 21 whitened components of
$X=W\,\delta K\,W^{\mathsf T}$ used for the composition above (off-diagonal
entries scaled by $\sqrt2$). The matrix $A$ that maps a $\theta$ displacement to
these components satisfies $A^{\mathsf T}A=M(\theta_{\rm fit})$, and the cell
checks this. In these coordinates the geodesic prior has precision $\lambda I$,
and the data add the Fisher information $\tilde F=A^{-\mathsf T}J^{\mathsf T}JA^{-1}$.
The posterior-to-prior variance ratio of each component is therefore

$$\lambda\,\bigl[(\tilde F+\lambda I)^{-1}\bigr]_{cc},$$

which is 1 for a component the data does not touch and 0 for one it fixes
completely.

For each orbit and block the table and heat map give the **data share**
$1-{\rm ratio}$ in two forms:

* **mean:** averaged over the block's components;
* **best:** for the best-determined combination within the block (smallest
  eigenvalue of the block's marginal posterior covariance, in prior units).

Both are independent of the local axes, the reference point and the units.
Individual components are not, because the $x,y$ axes of the contact frame are
arbitrary.

In [ ]:
# --- Which blocks of K does the data constrain? (at the fit) ---------------------
# Whitened components of each contact's K at the base point: the 21 independent
# entries of X = W dK W^T (off-diagonal entries scaled by sqrt 2), so that the sum
# of squares of the components is the squared geodesic length of the step.
_COMP_LIST = ([('RR', ij) for ij in [(0,0),(1,1),(2,2),(0,1),(0,2),(1,2)]]
              + [('TR', (i, j)) for i in range(3) for j in range(3, 6)]
              + [('TT', ij) for ij in [(3,3),(4,4),(5,5),(3,4),(3,5),(4,5)]])
# weight 1 for diagonal entries, sqrt(2) for off-diagonal ones (each appears twice in X)
_COMP_W = np.array([1.0 if (b != 'TR' and ij[0] == ij[1]) else np.sqrt(2) for b, ij in _COMP_LIST])
_BLOCKS = ('RR', 'TR', 'TT')

def whitened_component_map(theta0):
    """A (N_PARAMS x N_PARAMS): theta displacement -> whitened K components at theta0.
    Block-diagonal by orbit, and A^T A = metric_at(theta0)."""
    L0 = theta_to_Lmap(theta0); K0, _ = build_K(L0)
    A = np.zeros((N_PARAMS, N_PARAMS))
    for ci, c in enumerate(shell_order):
        W = _tfirst_whitener(K0[c])
        for a in range(N_TRIL):
            E = np.zeros((6, 6)); E[TRIL_I[a], TRIL_J[a]] = 1.0
            X = W @ (E @ L0[c].T + L0[c] @ E.T) @ W.T
            A[ci*N_TRIL:(ci+1)*N_TRIL, ci*N_TRIL + a] = _COMP_W*np.array([X[i, j] for _, (i, j) in _COMP_LIST])
    return A

def block_constraint(fisher, theta0, lam, A=None):
    """Posterior / prior variance of each whitened K component at theta0.

    The prior precision in these coordinates is lam * I (the geodesic prior, locally),
    so ratio = lam * [(F~ + lam I)^-1] with F~ the Fisher information of the data in
    the same coordinates. ratio -> 0: fixed by the data; ratio -> 1: prior only.
    Returns the per-component ratios (n_contacts x 21) and, per contact and block,
    (mean ratio over the block's components, ratio of its best-determined combination)."""
    A = whitened_component_map(theta0) if A is None else A
    Ai = np.linalg.inv(A)
    Ft = Ai.T @ fisher @ Ai
    ratio = lam*np.linalg.inv(0.5*(Ft + Ft.T) + lam*np.eye(N_PARAMS))
    summary = {}
    for ci, c in enumerate(shell_order):
        for b in _BLOCKS:
            ix = [ci*N_TRIL + k for k, (bb, _) in enumerate(_COMP_LIST) if bb == b]
            sub = ratio[np.ix_(ix, ix)]
            summary[(c, b)] = (float(np.mean(np.diag(sub))),
                               float(np.linalg.eigvalsh(0.5*(sub + sub.T)).min()))
    return np.diag(ratio).reshape(N_SHELL, N_TRIL), summary

def plot_block_constraint(summary, title, fname):
    share = np.array([[1 - summary[(c, b)][0] for b in _BLOCKS] for c in shell_order])
    best  = np.array([[1 - summary[(c, b)][1] for b in _BLOCKS] for c in shell_order])
    fig, ax = plt.subplots(figsize=(6.2, 0.55*N_SHELL + 1.6))
    im = ax.imshow(share, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    for i in range(N_SHELL):
        for j in range(3):
            ax.text(j, i, f"{share[i, j]:.2f} / {best[i, j]:.2f}", ha='center', va='center',
                    fontsize=8, color='white' if share[i, j] > 0.6 else '0.1')
    ax.set_xticks(range(3)); ax.set_xticklabels(_BLOCKS)
    ax.set_yticks(range(N_SHELL))
    ax.set_yticklabels([f"{unique[c]['label']}  ({unique[c]['n_contacts']} pairs)" for c in shell_order], fontsize=8)
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, label='data share of the prior variance (mean over block)')
    plt.tight_layout(); plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.show()
    return share, best

_A_fit = whitened_component_map(theta_fit)
_Mchk = metric_at(theta_fit)
print(f"check: max|A^T A - metric_at(theta_fit)| / max|M| = "
      f"{np.abs(_A_fit.T @ _A_fit - _Mchk).max()/np.abs(_Mchk).max():.1e}")
_ratio_fit, _blocks_fit = block_constraint(_Jf.T @ _Jf, theta_fit, LAMBDA_PRIOR, A=_A_fit)

print(f"\nData share of the prior variance, per orbit and block, at the fit "
      f"(λ = {LAMBDA_PRIOR:g}); 'mean' over the block's components / 'best' combination:")
print(f"{'orbit':>20s} " + "".join(f"{b:>16s}" for b in _BLOCKS))
for c in shell_order:
    print(f"{unique[c]['label']:>20s} " + "".join(f"{1-_blocks_fit[(c, b)][0]:8.2f} / {1-_blocks_fit[(c, b)][1]:5.2f}" for b in _BLOCKS))
_avg = {b: np.mean([1 - _blocks_fit[(c, b)][0] for c in shell_order]) for b in _BLOCKS}
print("   average over orbits:  " + "   ".join(f"{b} {_avg[b]:.2f}" for b in _BLOCKS))
plot_block_constraint(_blocks_fit, f'What the fitted halos fix at the fit (λ = {LAMBDA_PRIOR:g})\n'
                      'mean / best-combination data share',
                      OUT_PREFIX + 'block_constraint_fit.png')

### The Elastic Tensor

$D(\mathbf q)$ is the Fourier series of the rigid-body force constants, so every halo
samples the same small-$\delta\mathbf q$ expansion of $D$. Its second-order
translational term, with all internal degrees of freedom relaxed, is the elastic
tensor.

**Computation** (`elastic_tensor`). Under a homogeneous strain $\varepsilon$ every
body's centre of mass moves affinely, $\mathbf v_b=\varepsilon\,\mathbf r_{{\rm cm},b}$ (and
$\varepsilon(\mathbf r_{{\rm cm},b}+\mathbf R_{\mathbf n})$ for its image in cell $\mathbf n$). On
top of that the internal coordinates $\xi$, the rotations $\boldsymbol\omega_b$ and
extra shifts $\delta\mathbf v_b$ of all eight bodies, are free. Member $m$ is deformed by
$\Delta_m=P_m\xi+Q_me$ (Voigt strain $e$), and minimizing
$\tfrac12\sum_m\Delta_m^{\mathsf T}K_m\Delta_m$ over $\xi$ gives

$$C=\frac1{V_{\rm cell}}\Bigl[\textstyle\sum Q^{\mathsf T}KQ-\bigl(\sum Q^{\mathsf T}KP\bigr)
\bigl(\sum P^{\mathsf T}KP\bigr)^{+}\bigl(\sum P^{\mathsf T}KQ\bigr)\Bigr].$$

The pseudo-inverse removes the uniform translation, which deforms nothing. With
eight bodies the relaxation includes the sublattice shifts as well as the
rotations, so it is a larger effect than in P1. The check is the same:
$m_{\rm cell}\,\omega^2=V_{\rm cell}\,\mathrm{eig}\,\Gamma(\mathbf q)$ for the three acoustic
branches of $D(\mathbf q)$ at small $\mathbf q$, with $m_{\rm cell}$ the mass of the cell.

**Reported quantities.** As for 6o2h: the 21 components $C_{IJ}$ in the lab frame,
the six Kelvin moduli, and the Voigt bulk and shear moduli. For P4₃ (Laue class
4/m) $C$ has 7 independent components, and the components that symmetry forces to
vanish or to be equal are reported as a check. Posterior and prior standard
deviations and data shares follow from $\partial C/\partial\theta$ as in the P1
notebook.

In [ ]:
# --- Elastic tensor of the multi-body rigid lattice ------------------------------------
_V_CELL  = (2*np.pi)**3/abs(np.linalg.det(B_recip))          # Å^3
_KT_GPA  = 1.380649e-23*TEMPERATURE/1e-30/1e9                 # 1 k_BT/Å^3 in GPa
_VOIGT   = [(0, 0), (1, 1), (2, 2), (1, 2), (0, 2), (0, 1)]
_VIDX    = np.array([[0, 5, 4], [5, 1, 3], [4, 3, 2]])        # (i,j) -> Voigt index
_VLAB    = ['11', '22', '33', '23', '13', '12']
_MANDEL  = np.diag([1, 1, 1, np.sqrt(2), np.sqrt(2), np.sqrt(2)])

def _strain_from_voigt(e):
    E = np.zeros((3, 3))
    for a, (i, j) in enumerate(_VOIGT):
        if i == j: E[i, i] = e[a]
        else:      E[i, j] = E[j, i] = 0.5*e[a]
    return E

def _member_strain_maps():
    """Delta_m = P_m xi + Q_m e: xi the internal coordinates (6 per body), e the Voigt strain."""
    maps = []
    for M in MEMBERS:
        a, b = M['a'], M['b']
        P = np.zeros((6, N_DOF)); P[:, 6*a:6*a + 6] += M['Ta']; P[:, 6*b:6*b + 6] -= M['Tb']
        ra, rb = BODIES[a]['com'], BODIES[b]['com'] + M['Rn']
        Q = np.zeros((6, 6))
        for k in range(6):
            E = _strain_from_voigt(np.eye(6)[k])
            Q[:, k] = M['Ta'] @ np.r_[0, 0, 0, E @ ra] - M['Tb'] @ np.r_[0, 0, 0, E @ rb]
        maps.append((P, Q))
    return maps
_STRAIN_MAPS = _member_strain_maps()

def elastic_tensor(K_lab, relax=True):
    """6x6 Voigt stiffness in k_BT/Å^3 (x _KT_GPA for GPa), lab frame. relax=False clamps
    every internal coordinate (no rotations, no sublattice shifts)."""
    App = np.zeros((N_DOF, N_DOF)); Apq = np.zeros((N_DOF, 6)); Aqq = np.zeros((6, 6))
    for (P, Q), K in zip(_STRAIN_MAPS, K_lab):
        App += P.T @ K @ P; Apq += P.T @ K @ Q; Aqq += Q.T @ K @ Q
    C = Aqq - Apq.T @ np.linalg.lstsq(App, Apq, rcond=1e-12)[0] if relax else Aqq
    return 0.5*(C + C.T)/_V_CELL

def christoffel(Cv, q):
    G = np.zeros((3, 3))
    for i in range(3):
        for k in range(3):
            G[i, k] = sum(Cv[_VIDX[i, j], _VIDX[k, l]]*q[j]*q[l] for j in range(3) for l in range(3))
    return G

def elastic_summary(Cv):
    comps = np.array([Cv[i, j] for i in range(6) for j in range(i, 6)])
    kelvin = np.sort(np.linalg.eigvalsh(_MANDEL @ Cv @ _MANDEL))[::-1]
    KV = (Cv[0, 0] + Cv[1, 1] + Cv[2, 2] + 2*(Cv[0, 1] + Cv[0, 2] + Cv[1, 2]))/9
    GV = ((Cv[0, 0] + Cv[1, 1] + Cv[2, 2]) - (Cv[0, 1] + Cv[0, 2] + Cv[1, 2])
          + 3*(Cv[3, 3] + Cv[4, 4] + Cv[5, 5]))/15
    return np.r_[comps, kelvin, KV, GV]
_COMP_LAB = [f'C{_VLAB[i]},{_VLAB[j]}' for i in range(6) for j in range(i, 6)]
_SUMM_LAB = _COMP_LAB + [f'Kelvin {k + 1}' for k in range(6)] + ['K_V (bulk)', 'G_V (shear)']
_N_COMP   = len(_COMP_LAB)

def elastic_of_theta(theta):
    return elastic_summary(elastic_tensor(build_K(theta_to_Lmap(theta))[1]))

def elastic_jacobian(theta, rel_eps=1e-5):
    theta = np.asarray(theta, float)
    scale = max(np.abs(theta).mean(), 1e-12)
    Jc = np.zeros((len(elastic_of_theta(theta)), len(theta)))
    for j in range(len(theta)):
        h = rel_eps*max(abs(theta[j]), 1e-3*scale)
        tp = theta.copy(); tp[j] += h
        tm = theta.copy(); tm[j] -= h
        Jc[:, j] = (elastic_of_theta(tp) - elastic_of_theta(tm))/(2*h)
    return Jc

def elastic_constraint(Jc, F, P_prec):
    Cpost = np.linalg.inv(F + P_prec); Cpr = np.linalg.inv(P_prec)
    vpost = np.einsum('ij,jk,ik->i', Jc, Cpost, Jc)
    vpr   = np.einsum('ij,jk,ik->i', Jc, Cpr, Jc)
    return np.sqrt(np.maximum(vpost, 0)), np.sqrt(np.maximum(vpr, 0)), 1 - vpost/np.maximum(vpr, 1e-300)

# --- Check against the acoustic dispersion of D(q) itself --------------------------------
_K_chk = K_LAB_PRIOR
_C_chk = elastic_tensor(_K_chk)
_errs = []
for _qh in np.random.default_rng(0).standard_normal((6, 3)):
    _qh /= np.linalg.norm(_qh); _s = 1e-4
    _D = dynamical_matrix_batch((_s*_qh)[None], unique, _K_chk)[0]
    _w2 = np.sort(np.linalg.eigvalsh(Msq_inv @ _D @ Msq_inv.T))[:3]/_s**2
    _w2c = np.sort(np.linalg.eigvalsh(christoffel(_C_chk, _qh)))*_V_CELL/m_cell
    _errs.append(np.max(np.abs(_w2 - _w2c)/_w2c))
print(f"elastic tensor vs. small-q acoustic dispersion of D(q): max relative difference "
      f"{max(_errs):.1e} over 6 directions")
assert max(_errs) < 1e-3, "elastic tensor disagrees with the acoustic branches of D(q)"
# Laue class 4/m: C11 = C22, C44 = C55, C13 = C23, C16 = -C26, and C14 = C15 = C24 = C25
# = C34 = C35 = C36 = C46 = C56 = 0 (Voigt indices 1..6 = 11, 22, 33, 23, 13, 12).
_Cg = _C_chk/np.abs(_C_chk).max()
_tet = [abs(_Cg[0, 0] - _Cg[1, 1]), abs(_Cg[3, 3] - _Cg[4, 4]), abs(_Cg[0, 2] - _Cg[1, 2]),
        abs(_Cg[0, 5] + _Cg[1, 5])] + [abs(_Cg[i, j]) for i, j in
        [(0, 3), (0, 4), (1, 3), (1, 4), (2, 3), (2, 4), (2, 5), (3, 5), (4, 5)]]
if SG.number in range(75, 89):
    print(f"tetragonal (4/m) form of C: largest violation {max(_tet):.1e} of the largest component")
    assert max(_tet) < 1e-8, "the elastic tensor does not have the Laue symmetry"

def relaxation_effect(K_lab):
    Cr, Cc = elastic_tensor(K_lab), elastic_tensor(K_lab, relax=False)
    return np.linalg.norm(_MANDEL @ (Cr - Cc) @ _MANDEL)/np.linalg.norm(_MANDEL @ Cc @ _MANDEL)


# --- The elastic tensor of the fit and of the calibrated prior, with the posterior ----
_Cv = {lab: elastic_tensor(K)*_KT_GPA for lab, K in [('prior', K_LAB_PRIOR), ('fit', K_LAB_FIT)]}
_Es = {lab: elastic_summary(C) for lab, C in _Cv.items()}
_Jc_fit = elastic_jacobian(theta_fit)*_KT_GPA
_sd_fit, _sd_pr, _share_el = elastic_constraint(_Jc_fit, _Jf.T @ _Jf, LAMBDA_PRIOR*parameter_metric(theta_fit))
_sd_fit *= np.sqrt(chi2_dof) if INFLATE_BY_CHI2 else 1.0
_nrm = lambda C: np.linalg.norm(_MANDEL @ C @ _MANDEL)
print(f"Elastic tensor (GPa-equivalent in the map's units, {TEMPERATURE:g} K, lab frame, internal coordinates relaxed)")
print(f"internal relaxation softens the fitted tensor by {100*relaxation_effect(K_LAB_FIT):.1f}%")
print(f"||C_fit - C_prior|| / ||C_prior|| = {_nrm(_Cv['fit'] - _Cv['prior'])/_nrm(_Cv['prior']):.1%}")
print(f"\n{'':14s} {'prior':>9s} {'fit':>9s} {'fit/prior':>10s} {'post. sd':>9s} {'data share':>11s}")
for i in range(_N_COMP, len(_SUMM_LAB)):
    p, f, sd = _Es['prior'][i], _Es['fit'][i], _sd_fit[i]
    print(f"{_SUMM_LAB[i]:14s} {p:9.4g} {f:9.4g} {f/p:10.3f} {100*sd/abs(f):8.1f}% {_share_el[i]:11.2f}")
print("   Kelvin moduli: eigenvalues of the Mandel form of C (frame-independent)")
print("   data share: 1 - posterior/prior variance at the fit")
_ind = [(0, 0), (2, 2), (0, 1), (0, 2), (3, 3), (5, 5), (0, 5)]          # the 7 independent 4/m components
print(f"\nindependent 4/m components (Voigt): " + "  ".join(
    f"C{i + 1}{j + 1} {_Cv['fit'][i, j]:.4g} ± {_sd_fit[_COMP_LAB.index(f'C{_VLAB[i]},{_VLAB[j]}')]:.2g}" for i, j in _ind))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
_xk = np.arange(6)
ax[0].bar(_xk - 0.2, _Es['prior'][_N_COMP:_N_COMP + 6], 0.4, color='0.6', label='calibrated prior')
ax[0].bar(_xk + 0.2, _Es['fit'][_N_COMP:_N_COMP + 6], 0.4, yerr=_sd_fit[_N_COMP:_N_COMP + 6], color='#2a78d6',
          capsize=3, label='fit (± posterior sd)')
ax[0].set_xticks(_xk); ax[0].set_xticklabels([f'K{k + 1}' for k in range(6)]); ax[0].set_yscale('log')
ax[0].set_ylabel('Kelvin modulus (map units)'); ax[0].set_title('Kelvin moduli'); ax[0].legend(fontsize=8)
ax[1].bar(np.arange(len(_SUMM_LAB) - _N_COMP), _share_el[_N_COMP:], color='#2a78d6')
ax[1].set_xticks(np.arange(len(_SUMM_LAB) - _N_COMP)); ax[1].set_xticklabels([l.replace(' (bulk)', '').replace(' (shear)', '') for l in _SUMM_LAB[_N_COMP:]], rotation=45, fontsize=8)
ax[1].set_ylim(0, 1); ax[1].set_ylabel('data share of the prior variance'); ax[1].set_title('What the halos fix')
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'elastic_tensor.png', dpi=150); plt.show()

## Watching the Band Structure Converge

The calibrated prior's band structure (thick grey) and the band structure at up to
60 evenly spaced iterates of the main refinement (blue).

In [ ]:
freqs_prior_bs = bandstructure_freqs(K_LAB_PRIOR)
ymax = max(freqs_prior_bs.max(), bandstructure_freqs(K_LAB_FIT).max())*1.15
frame_idx = np.unique(np.linspace(0, len(theta_history) - 1, min(60, len(theta_history))).astype(int))
legend_handles = [Line2D([0], [0], color='0.6', lw=3.5, label='calibrated prior'),
                  Line2D([0], [0], color='steelblue', lw=1.3, label='current fit')]
images_bs = []
for fi in frame_idx:
    freqs_fi = bandstructure_freqs(build_K(theta_to_Lmap(theta_history[fi]))[1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, freqs_prior_bs, color='0.6', lw=2.5, zorder=1)
    ax.plot(x, freqs_fi, color='steelblue', lw=0.8, zorder=2)
    for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
    ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
    ax.set_ylabel('Frequency (map units$^{1/2}$)'); ax.set_xlim(x[0], x[-1]); ax.set_ylim(0, ymax)
    ax.set_title(f'Band-structure convergence — iteration {fi}/{len(theta_history) - 1}')
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8)
    plt.tight_layout(); images_bs.append(fig_to_image(fig)); plt.close(fig)
imageio.mimsave(OUT_PREFIX + 'band_structure_convergence.gif', images_bs, fps=8, loop=0)
del images_bs; gc.collect()
print(f"Saved {OUT_PREFIX}band_structure_convergence.gif ({len(frame_idx)} frames)")
print("(frequencies scale as the square root of the map's intensity scale: only their shape is absolute)")

## Measured vs. Fitted Diffuse Scattering

The measured map on the layers $l=0$ and $l=\tfrac12$ of the data grid (expanded from
the asymmetric unit with the Laue group), next to the fitted and the calibrated-prior
models, TDS plus each model's own isotropic background, out to `D_FIT_MIN`. Bragg
voxels and voxels within `BRAGG_EXCL_CART` of a reciprocal-lattice point are blanked.
The colour scale is linear and shared. The last panel is the
anisotropic part of $(I_{\rm obs}-I_{\rm fit})/\sigma$ (each map minus its mean in thin
$|\mathbf q|$ shells), so that the background's mismatch outside the fitted shell does not
dominate it.

The background was fitted only over the $|\mathbf q|$ range of the fitting voxels and is
constant beyond it, so at low resolution the models' absolute level is not
meaningful. The plane correlations are therefore also given after subtracting each
map's mean in thin $|\mathbf q|$ shells (the *anisotropic* CC), which removes any
isotropic term, including the background.

In [ ]:
def model_on_points(hkl, K_lab, bg, G=None):
    G = G_at_hkl_batch(*hkl.T) if G is None else G
    return diffuse_intensity_batch(K_lab, *hkl.T, G_arr=G) + background_at(hkl, bg), G

def _aniso(v, qn, n_shell=40):
    """v minus its mean in equal-count |q| shells (NaNs ignored)."""
    out = np.full(len(v), np.nan); ok = np.isfinite(v)
    if ok.sum() < n_shell:
        return out
    edges = np.quantile(qn[ok], np.linspace(0, 1, n_shell + 1))
    b = np.clip(np.digitize(qn, edges) - 1, 0, n_shell - 1)
    for s_ in range(n_shell):
        m = ok & (b == s_)
        if m.any():
            out[m] = v[m] - v[m].mean()
    return out

PLANE_MAPS = {}
for l_t in (0.0, 0.5):
    hp, kp, I_obs_p, l_val = read_hk_plane(l_t, d_min=D_FIT_MIN)
    _, _, S_obs_p, _ = read_hk_plane(l_t, field='sigma', d_min=D_FIT_MIN)
    HH, KK = np.meshgrid(hp, kp, indexing='ij')
    hkl_p = np.column_stack([HH.ravel(), KK.ravel(), np.full(HH.size, l_val)])
    ok = np.isfinite(I_obs_p.ravel()) & (_bragg_offsets(hkl_p) >= BRAGG_EXCL_CART)
    I_fit_p = np.full(HH.size, np.nan); I_pri_p = np.full(HH.size, np.nan)
    I_fit_p[ok], Gp = model_on_points(hkl_p[ok], K_LAB_FIT, BG_FIT)
    I_pri_p[ok], _ = model_on_points(hkl_p[ok], K_LAB_PRIOR, BG_PRIOR, G=Gp)
    obs = np.where(ok, I_obs_p.ravel(), np.nan)
    PLANE_MAPS[l_val] = dict(h=hp, k=kp, obs=obs.reshape(HH.shape), fit=I_fit_p.reshape(HH.shape),
                             prior=I_pri_p.reshape(HH.shape), sig=S_obs_p, hkl=hkl_p, ok=ok)
    del Gp; gc.collect()

for l_val, P_ in PLANE_MAPS.items():
    ext = [P_['h'][0], P_['h'][-1], P_['k'][0], P_['k'][-1]]
    fin = np.isfinite(P_['obs'])
    lo, hi = np.percentile(P_['obs'][fin], [1, 99])
    fig, axes = plt.subplots(1, 4, figsize=(21, 5.2))
    for ax, key, title in zip(axes[:3], ['obs', 'fit', 'prior'],
                              ['measured (mdx2)', 'fit: TDS + background', 'calibrated prior: TDS + background']):
        im = ax.imshow(np.clip(P_[key], lo, hi).T, origin='lower', cmap='inferno', extent=ext, aspect='equal')
        ax.set_title(f'{title}, l = {l_val:g}'); ax.set_xlabel('h'); ax.set_ylabel('k')
    fig.colorbar(im, ax=axes[:3], fraction=0.015, pad=0.01, label='intensity (map units)')
    qn = np.linalg.norm(P_['hkl'] @ B_recip.T, axis=1)
    _sg = np.where(np.isfinite(P_['sig']), P_['sig'], np.nan).ravel()
    z = ((_aniso(P_['obs'].ravel(), qn) - _aniso(P_['fit'].ravel(), qn))/_sg).reshape(P_['obs'].shape)
    im2 = axes[3].imshow(np.clip(z, -5, 5).T, origin='lower', cmap='RdBu_r', vmin=-5, vmax=5, extent=ext, aspect='equal')
    axes[3].set_title('anisotropic residual / σ\n(shell means removed from both maps)', fontsize=10); axes[3].set_xlabel('h')
    fig.colorbar(im2, ax=axes[3], fraction=0.046)
    plt.savefig(OUT_PREFIX + f'map_measured_vs_fit_l{l_val:g}.png', dpi=130, bbox_inches='tight'); plt.show()
    o, f, p = P_['obs'].ravel(), P_['fit'].ravel(), P_['prior'].ravel()
    m = np.isfinite(o) & np.isfinite(f)
    ao, af, ap = _aniso(o, qn), _aniso(f, qn), _aniso(p, qn)
    ma = np.isfinite(ao) & np.isfinite(af)
    print(f"l = {l_val:g}: {m.sum()} voxels; CC(measured, fit) = {np.corrcoef(o[m], f[m])[0, 1]:.4f}, "
          f"(measured, prior) = {np.corrcoef(o[m], p[m])[0, 1]:.4f}; anisotropic CC: fit "
          f"{np.corrcoef(ao[ma], af[ma])[0, 1]:.4f}, prior {np.corrcoef(ao[ma], ap[ma])[0, 1]:.4f}; "
          f"median |z| {np.nanmedian(np.abs(z)):.2f}")

### Model Accuracy by Resolution Shell, Against the Data's Own Ceiling

In `N_CC_SHELLS` equal-count shells of $|\mathbf q|$ between `D_FIT_MAX` and
`D_FIT_MIN`, up to `CC_SHELL_MAX` random non-Bragg voxels per shell are compared with
the fitted and the prior TDS. Within a thin shell the correlation does not depend on
any isotropic term, so the background plays no role.

The ceiling is $CC^*$ computed on the **same voxels** from the two half-datasets
($CC^*=\sqrt{2CC_{1/2}/(1+CC_{1/2})}$, the correlation of the merged map with its
noise-free version). A model that reached $CC^*$ would explain everything the data
can resolve, so $CC_{\rm model}/CC^*$ is the fraction of the measurable anisotropic
signal the model accounts for. Voxels in the fitted halos are included (they are a
small part of every shell). The panel marks the halo window.

In [ ]:
N_CC_SHELLS  = 14
CC_SHELL_MAX = 4000
_rng_cc = np.random.default_rng(5)
_hk_k = HKL_INT/_NDIV
_qn_k = np.linalg.norm(_hk_k @ B_recip.T, axis=1)
_elig = (~BRAGG_ALL) & (_qn_k >= 2*np.pi/D_FIT_MAX) & (_qn_k <= 2*np.pi/D_FIT_MIN) & \
        (_bragg_offsets(_hk_k) >= BRAGG_EXCL_CART)
if HAVE_HALVES:
    _elig &= np.isfinite(G0_ALL) & np.isfinite(G1_ALL)
_ix_e = np.where(_elig)[0]
_cc_edges = np.quantile(_qn_k[_ix_e], np.linspace(0, 1, N_CC_SHELLS + 1))
_shell_e = np.clip(np.digitize(_qn_k[_ix_e], _cc_edges) - 1, 0, N_CC_SHELLS - 1)
CC_ROWS = []
for s_ in range(N_CC_SHELLS):
    ii = _ix_e[_shell_e == s_]
    if len(ii) > CC_SHELL_MAX:
        ii = _rng_cc.choice(ii, CC_SHELL_MAX, replace=False)
    hk = _hk_k[ii]
    G_s = G_at_hkl_batch(*hk.T)
    I_f = diffuse_intensity_batch(K_LAB_FIT, *hk.T, G_arr=G_s)
    I_p = diffuse_intensity_batch(K_LAB_PRIOR, *hk.T, G_arr=G_s)
    o = I_ALL[ii].astype(float)
    cc_f, cc_p = np.corrcoef(o, I_f)[0, 1], np.corrcoef(o, I_p)[0, 1]
    if HAVE_HALVES:
        c12 = np.corrcoef(G0_ALL[ii].astype(float), G1_ALL[ii].astype(float))[0, 1]
        ccs = np.sqrt(max(2*c12/(1 + c12), 0.0)) if c12 > -1 else np.nan
    else:
        c12 = ccs = np.nan
    d_lo, d_hi = 2*np.pi/_cc_edges[s_ + 1], 2*np.pi/_cc_edges[s_]
    CC_ROWS.append((d_hi, d_lo, len(ii), cc_f, cc_p, c12, ccs))
    del G_s
print(f"{'shell d (Å)':>14s} {'voxels':>7s} {'CC fit':>7s} {'CC prior':>9s} {'CC1/2':>7s} {'CC*':>6s} {'fit/CC*':>8s}")
for d_hi, d_lo, n, cf, cp, c12, ccs in CC_ROWS:
    print(f"{d_hi:6.2f}–{d_lo:5.2f} {n:7d} {cf:7.3f} {cp:9.3f} {c12:7.3f} {ccs:6.3f} {cf/ccs if ccs > 0 else np.nan:8.2f}")
_dm = np.array([0.5*(r[0] + r[1]) for r in CC_ROWS])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(_dm, [r[3] for r in CC_ROWS], 'o-', color='#2a78d6', label='fit')
ax.plot(_dm, [r[4] for r in CC_ROWS], 's--', color='#eb6834', label='calibrated prior')
if HAVE_HALVES:
    ax.plot(_dm, [r[6] for r in CC_ROWS], '-', color='0.3', lw=2, label='$CC^*$ (data ceiling, same voxels)')
ax.axvspan(D_MIN_HALO, D_MAX_HALO, color='0.9', zorder=0, label='halo window (fitted)')
_d_axis(ax, 0.95*D_FIT_MIN, 1.05*D_FIT_MAX); ax.set_ylim(0, 1.02)
ax.set_ylabel('correlation with the measured map')
ax.set_title('Model–data CC by resolution shell'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'cc_resolution_shells.png', dpi=150); plt.show()
del _hk_k, _qn_k; gc.collect()

## Predicted Rigid-Body ADPs vs. the Deposited 7tx0 ADPs

$\Sigma=\langle uu^{\mathsf T}\rangle=N^{-1}\sum_{\mathbf q}D(\mathbf q)^{-1}$ on an offset grid of
`BZ_NGRID`³ points (convergence printed, with a $1/n$ extrapolation), projected onto
each atom: $U=J_r\Sigma_{bb}J_r^{\mathsf T}$ and $B=\tfrac{8\pi^2}{3}\mathrm{Tr}\,U$.

**What can be compared.** The deposited ADPs come from another crystal and contain
every kind of motion and disorder, of which the rigid-body lattice vibrations are
one part. And the map is not on an absolute scale, so the predicted $U$ carry an
unknown overall factor. The cell therefore compares *patterns*: the correlation of
per-residue $B$ (fit and prior) with the deposited values, the single scale that
best matches them, and, for atoms with ANISOU records, the agreement of the
principal directions of the displacement ellipsoids (|cosine| between the major
axes; 0.5 is the mean for random directions in 3-D).

In [ ]:
Sigma_fit = bz_covariance(K_LAB_FIT)
Sigma_prior = bz_covariance(K_LAB_PRIOR)
U_fit, U_prior = all_adp_tensors(Sigma_fit), all_adp_tensors(Sigma_prior)
B_fit, B_prior = isotropic_B(U_fit), isotropic_B(U_prior)
ATOM_CHAIN = np.concatenate([[c]*len(CHAINS[c]['pos']) for c in BODY_CHAINS])
b_exp = np.concatenate([CHAINS[c]['b'] for c in BODY_CHAINS])
u_exp = np.concatenate([CHAINS[c]['U'] for c in BODY_CHAINS])
has_aniso = np.concatenate([CHAINS[c]['aniso'] for c in BODY_CHAINS])
_sym = max(np.abs(body_block(Sigma_fit, b) - S_of(BODIES[b]['R']) @ body_block(Sigma_fit, ASU_BODY[BODIES[b]['chain']])
                  @ S_of(BODIES[b]['R']).T).max()/np.abs(body_block(Sigma_fit, b)).max() for b in range(N_BODY))
print(f"symmetry of the covariance: max |Σ_bb - S_b Σ_cc S_b^T| / |Σ_bb| = {_sym:.1e}")
assert _sym < 1e-6
check_bz_convergence(K_LAB_FIT)

res_ids = [r for c in BODY_CHAINS for r in CHAINS[c]['res']]
res_index = {}; res_inverse = np.empty(len(res_ids), dtype=int)
for i, key in enumerate(res_ids):
    res_inverse[i] = res_index.setdefault(key, len(res_index))
n_res = len(res_index)
def per_residue_mean(x):
    return np.bincount(res_inverse, weights=x, minlength=n_res)/np.maximum(np.bincount(res_inverse, minlength=n_res), 1)
_res_chain = np.array([k[0] for k in res_index])
_lig_atom = np.concatenate([CHAINS[c]['lig'] for c in BODY_CHAINS])
_lig_res = np.array([bool(np.any(_lig_atom[res_inverse == r])) for r in range(n_res)])
Bf_r, Bp_r, Be_r = per_residue_mean(B_fit), per_residue_mean(B_prior), per_residue_mean(b_exp)
_poly = ~_lig_res
print(f"\n{'':8s} {'mean B (map units)':>19s} {'r(per residue, deposited)':>26s} {'best scale to deposited':>24s}")
for lab, Br in [('fit', Bf_r), ('prior', Bp_r)]:
    s = float(np.sum(Br[_poly]*Be_r[_poly])/np.sum(Br[_poly]**2))
    print(f"{lab:8s} {Br[_poly].mean():19.4g} {np.corrcoef(Br[_poly], Be_r[_poly])[0, 1]:26.3f} {s:24.4g}")
for c in BODY_CHAINS:
    m = (_res_chain == c) & _poly
    print(f"chain {c}: r(fit, deposited) = {np.corrcoef(Bf_r[m], Be_r[m])[0, 1]:.3f}; "
          f"deposited mean B {Be_r[m].mean():.1f} Å²")

def _major_axis(U):
    return np.linalg.eigh(U)[1][:, :, -1]
if has_aniso.mean() > 0.5:
    m = has_aniso & ~_lig_atom
    for lab, U in [('fit', U_fit), ('prior', U_prior)]:
        cosv = np.abs(np.sum(_major_axis(U[m])*_major_axis(u_exp[m]), axis=1))
        print(f"major axes, {lab} vs deposited ANISOU: median |cos| {np.median(cosv):.3f} "
              f"(random: 0.5), fraction within 30°: {np.mean(cosv > np.cos(np.radians(30))):.1%}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={'width_ratios': [2.2, 1]})
_s_f = float(np.sum(Bf_r[_poly]*Be_r[_poly])/np.sum(Bf_r[_poly]**2))
_s_p = float(np.sum(Bp_r[_poly]*Be_r[_poly])/np.sum(Bp_r[_poly]**2))
ax[0].plot(np.where(_poly, Be_r, np.nan), color='0.3', lw=1.2, label=f'deposited {PDB_ID}')
ax[0].plot(np.where(_poly, _s_f*Bf_r, np.nan), color='#2a78d6', lw=1.2, label='fit (scaled)')
ax[0].plot(np.where(_poly, _s_p*Bp_r, np.nan), color='#eb6834', lw=1.0, ls='--', label='prior (scaled)')
for c in BODY_CHAINS[1:]:
    ax[0].axvline(np.argmax(_res_chain == c) - 0.5, color='k', lw=0.8, ls=':')
ax[0].set_xlabel('residue index (chains in order)'); ax[0].set_ylabel('$B$ (Å², predicted scaled to deposited)')
ax[0].set_title('Per-residue B: rigid-body prediction vs deposited'); ax[0].legend(fontsize=8)
ax[1].scatter(Be_r[_poly], Bf_r[_poly], s=8, alpha=0.6, color='#2a78d6')
ax[1].set_xlabel('deposited B per residue (Å²)'); ax[1].set_ylabel('fitted rigid-body B (map units)')
ax[1].set_title('Per-residue B')
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'adp_vs_deposited.png', dpi=150); plt.show()
print("The rigid-body B varies smoothly with distance from each body's rotation centres; the")
print("deposited B adds loops, side chains and disorder, so a modest correlation is expected.")

## Two-Point Displacement Covariance Between Bodies

$$V_{ij}^{\alpha\beta}=\langle u_{i,\alpha}\,u_{j,\beta}\rangle$$

between atom $i$ of body $a$ in cell 0 and atom $j$ of body $b$ in cell $\mathbf n$ is the
real-space object the diffuse data measure. In the rigid-body model
$V_{ij}=J_i\,\Sigma_{ab}(\mathbf n)\,J_j^{\mathsf T}$, with $J_i$ the lever-arm map of atom
$i$ about its own body's centre of mass and

$$\Sigma_{ab}(\mathbf n)=\langle u_{a,\mathbf 0}u_{b,\mathbf n}^{\mathsf T}\rangle
=\frac1N\sum_{\mathbf q}\bigl[D(\mathbf q)^{+}\bigr]_{ab}\,e^{-i\mathbf q\cdot\mathbf R_{\mathbf n}}$$

(`lattice_covariance`, exact for a `V_NGRID`-periodic crystal; the supercell check
above tests the same Bloch algebra). The cell evaluates it for the Cα atoms of

* each asymmetric-unit chain with itself ($b=a$, $\mathbf n=0$): the ADPs and their
  correlations within one body;
* the representative bond of every orbit (the contacts);
* the nearest pairs of bodies that are **not** in contact.

It reports how far the fit moved $V$ from the calibrated prior,
$\|V_{\rm fit}-V_{\rm prior}\|_F/\|V_{\rm fit}\|_F$ over all Cα pairs and all $3\times3$
entries, next to the posterior spread of `V_SAMPLES` samples about the fit on the
same scale: a change well above the spread is one the data require. The pair correlation
$\rho_{ij}=\mathrm{tr}\,V_{ij}/\sqrt{\mathrm{tr}\,U_i\,\mathrm{tr}\,U_j}$ is mapped for the
A–A self pair and for the contact with the most atom pairs, fit and fit − prior.

In [ ]:
def lattice_covariance(K_lab, pairs, n_grid=16, offset=True, chunk=None):
    """Sigma_ab(n) (6x6) for every (a, b, n) in pairs, on an n_grid^3 offset BZ grid."""
    chunk = chunk or CHUNK_EVAL//4
    g = (np.arange(n_grid) + (0.5 if offset else 0.0))/n_grid
    Ig, Jg, Kg = np.meshgrid(g, g, g, indexing='ij')
    q_all = np.column_stack([Ig.ravel(), Jg.ravel(), Kg.ravel()]) @ B_recip.T
    Dref = dynamical_matrix_batch(q_all[:min(len(q_all), 2000)], unique, K_lab)
    floor = 1e-9*np.linalg.eigvalsh(Dref).max()
    R = np.array([A_orth @ np.array(n, float) for _, _, n in pairs])
    S = np.zeros((len(pairs), 6, 6), complex)
    for s0 in range(0, len(q_all), chunk):
        qb = q_all[s0:s0 + chunk]
        ev, U = np.linalg.eigh(dynamical_matrix_batch(qb, unique, K_lab))
        inv = np.where(ev > floor, 1.0/np.where(ev > floor, ev, 1.0), 0.0)
        ph = np.exp(-1j*(qb @ R.T))                                  # (nq, n_pairs)
        for k, (a, b, _) in enumerate(pairs):
            Ua, Ub = U[:, 6*a:6*a + 6, :], U[:, 6*b:6*b + 6, :]
            S[k] += np.tensordot(Ua*(inv*ph[:, k:k + 1])[:, None, :], Ub.conj(), axes=([0, 2], [0, 2]))
    return np.real(S)/len(q_all)

V_NGRID   = 12      # Brillouin-zone grid for Sigma(n); exact for a V_NGRID-periodic crystal
V_SAMPLES = 8       # posterior samples used for the uncertainty column
N_NONCONTACT = 4    # nearest non-contacting body pairs included for comparison

_CA = {c: np.array([i for i, (nm, lig) in enumerate(zip(CHAINS[c]['names'], CHAINS[c]['lig']))
                    if nm == 'CA' and not lig]) for c in BODY_CHAINS}
def _Jr_CA(b):
    c = BODIES[b]['chain']
    return rigid_jacobians(body_positions(b)[_CA[c]], BODIES[b]['com'])

V_PAIRS, V_KIND = [], []
for c in BODY_CHAINS:
    V_PAIRS.append((ASU_BODY[c], ASU_BODY[c], (0, 0, 0))); V_KIND.append('self')
for o in shell_order:
    V_PAIRS.append((unique[o]['a'], unique[o]['b'], unique[o]['n'])); V_KIND.append(f"contact {o}")
_cand = []
for a in [ASU_BODY[c] for c in BODY_CHAINS]:
    for b in range(N_BODY):
        for n in product(range(-1, 2), repeat=3):
            if (a == b and n == (0, 0, 0)) or bond_key(a, b, n) in BONDS:
                continue
            d = np.linalg.norm(BODIES[b]['com'] + A_orth @ np.array(n, float) - BODIES[a]['com'])
            _cand.append((d, a, b, n))
_seen = set()
for d, a, b, n in sorted(_cand):
    k = bond_key(a, b, n)
    if k in _seen: continue
    _seen.add(k); V_PAIRS.append((a, b, n)); V_KIND.append(f'no contact ({d:.0f} Å)')
    if len(_seen) >= N_NONCONTACT: break

def pair_covariances(K_lab):
    """V[k] (n_CA_a, n_CA_b, 3, 3) for each pair in V_PAIRS."""
    S = lattice_covariance(K_lab, V_PAIRS, V_NGRID)
    return [np.einsum('iab,jdb->ijad', _Jr_CA(a) @ S[k][None], _Jr_CA(b), optimize=True) for k, (a, b, _) in enumerate(V_PAIRS)]

def _self_trace(K_lab):
    """tr U of each Cα, per body (bodies of one chain agree by symmetry)."""
    S = lattice_covariance(K_lab, [(b, b, (0, 0, 0)) for b in range(N_BODY)], V_NGRID)
    return [np.trace(_Jr_CA(b) @ S[b][None] @ np.swapaxes(_Jr_CA(b), 1, 2), axis1=1, axis2=2) for b in range(N_BODY)]

_Vmods = {lab: pair_covariances(K) for lab, K in [('fit', K_LAB_FIT), ('prior', K_LAB_PRIOR)]}
_Vsamp = [pair_covariances(build_K(theta_to_Lmap(t))[1]) for t in _samples[:V_SAMPLES]]
_trU = {lab: _self_trace(K) for lab, K in [('fit', K_LAB_FIT), ('prior', K_LAB_PRIOR)]}

def _relerr(Va, Vb):
    return np.array([np.linalg.norm(x - y)/np.linalg.norm(y) for x, y in zip(Va, Vb)])

def _rho(V, trU):
    return [np.trace(Vk, axis1=2, axis2=3)/np.sqrt(np.outer(trU[a], trU[b])) for Vk, (a, b, _) in zip(V, V_PAIRS)]

_d_pf = _relerr(_Vmods['prior'], _Vmods['fit'])
_e_post = np.median([_relerr(Vs, _Vmods['fit']) for Vs in _Vsamp], axis=0)
_rho_f, _rho_p = _rho(_Vmods['fit'], _trU['fit']), _rho(_Vmods['prior'], _trU['prior'])
print(f"V_ij for the Cα atoms ({', '.join(f'{len(_CA[c])} in chain {c}' for c in BODY_CHAINS)})")
print(f"\n{'pair':>26s} {'kind':>22s} {'mean ρ fit':>11s} {'fit vs prior':>13s} {'posterior':>10s}")
_lbl = []
for k, (a, b, n) in enumerate(V_PAIRS):
    lab = f"{BODIES[a]['chain']}{BODIES[a]['op']}–{BODIES[b]['chain']}{BODIES[b]['op']}{'' if n == (0, 0, 0) else str(n)}"
    _lbl.append(lab)
    print(f"{lab:>26s} {V_KIND[k]:>22s} {_rho_f[k].mean():11.3f} {_d_pf[k]:13.1%} {_e_post[k]:10.1%}")

_kc = N_CHAIN + int(np.argmax([unique[o]['n_contacts'] for o in shell_order]))
fig, axes = plt.subplots(2, 2, figsize=(11, 9.5))
for r_, k in enumerate([0, _kc]):
    lim = np.abs(_rho_f[k]).max(); dlim = np.abs(_rho_f[k] - _rho_p[k]).max()
    for c_, (M, t_, lm) in enumerate([(_rho_f[k], 'fit', lim), (_rho_f[k] - _rho_p[k], 'fit − prior', dlim)]):
        im = axes[r_, c_].imshow(M, cmap='RdBu_r', vmin=-lm, vmax=lm, origin='lower', aspect='auto')
        axes[r_, c_].set_title(f"ρ_ij, {_lbl[k]} ({V_KIND[k]}): {t_}", fontsize=9)
        axes[r_, c_].set_xlabel('Cα index j'); axes[r_, c_].set_ylabel('Cα index i')
        fig.colorbar(im, ax=axes[r_, c_], fraction=0.046)
plt.tight_layout(); plt.savefig(OUT_PREFIX + 'pair_correlation_maps.png', dpi=150); plt.show()

## Phonon Mode Visualization

The lowest modes of the fitted model at `Q_VIZ` are rendered as looping GIFs of the
whole unit cell: all eight bodies, each moving rigidly by its own
$(\boldsymbol\Omega_b,\mathbf v_b)$ about its own centre of mass. With several bodies
per cell an eigenvector at a general $\mathbf q$ is genuinely complex: the bodies move
with different phases. Frame $t$ therefore shows $\mathrm{Re}(e\,e^{-2\pi it/T})$, the
exact motion of the cell over one period, rather than a standing wave. The table gives
each mode's frequency, its rotational share of the kinetic energy, how the energy is
shared between the chains, and the fraction of the eigenvector that is out of phase
(after removing the best global phase).

The surfaces are smooth envelopes of each chain (a Gaussian-blurred atom density,
largest connected component), shaded by chain; the asymmetric-unit bodies are
drawn darker. The animated displacement is scaled so the largest per-atom
displacement equals `AMPLITUDE`. The mode shapes do not depend on the map's scale; the
frequencies scale with its square root, so they are "THz" only if the map were absolute.

In [ ]:
Q_VIZ      = np.array([0.5, 0.25, 0.0])   # fractional — zone boundary, off-axis
N_MODES_VIZ = 4
AMPLITUDE  = 3.0    # Å — maximum per-atom displacement
N_FRAMES   = 20
GIF_FPS    = 10
SURF_SPACING, SURF_SIGMA = 2.0, 2.2   # Å — envelope grid and blur (coarse: rendering cost)

q_viz  = B_recip @ Q_VIZ
D_viz  = dynamical_matrix(q_viz, unique, K_LAB_FIT)
Dw_viz = Msq_inv @ D_viz @ Msq_inv.T
ev_v, evec_v = np.linalg.eigh(Dw_viz)
freqs_viz = np.sqrt(np.maximum(ev_v, 0))*freq_unit
evecs_lab = Msq_inv.T @ evec_v                       # complex, one column per mode
_phase = np.exp(-1j*np.angle(evecs_lab[np.abs(evecs_lab).argmax(axis=0), np.arange(N_DOF)]))
evecs_lab = evecs_lab*_phase[None, :]
_im_frac = np.linalg.norm(evecs_lab.imag, axis=0)/np.linalg.norm(evecs_lab, axis=0)

print(f"Lowest modes at q = {Q_VIZ}:")
print(f"{'mode':>5s} {'freq (THz)':>11s} {'rot KE %':>9s} " + " ".join(f"{'KE '+c+' %':>8s}" for c in BODY_CHAINS)
      + f" {'out of phase':>13s}")
def _ke(e, b):
    """(rotational, translational) kinetic-energy weight of body b in complex mode e."""
    return (np.real(e[6*b:6*b + 3].conj() @ BODIES[b]['J'] @ e[6*b:6*b + 3]),
            BODIES[b]['m']*np.real(e[6*b + 3:6*b + 6].conj() @ e[6*b + 3:6*b + 6]))
for s in range(max(N_MODES_VIZ, 8)):
    e = evecs_lab[:, s]
    ke_rot = sum(_ke(e, b)[0] for b in range(N_BODY)); ke_tr = sum(_ke(e, b)[1] for b in range(N_BODY))
    ke_ch = {c: sum(sum(_ke(e, b)) for b in range(N_BODY) if BODIES[b]['chain'] == c) for c in BODY_CHAINS}
    tot = ke_rot + ke_tr
    print(f"{s:5d} {freqs_viz[s]:11.5f} {100*ke_rot/tot:9.1f} " + " ".join(f"{100*ke_ch[c]/tot:8.1f}" for c in BODY_CHAINS)
          + f" {_im_frac[s]:13.3f}")

# --- envelope surface of each chain (in its own frame) -----------------------------------
def _chain_surface(c):
    P = CHAINS[c]['pos']
    lo, hi = P.min(0) - 3*SURF_SIGMA, P.max(0) + 3*SURF_SIGMA
    axes_ = [np.arange(l, h + SURF_SPACING, SURF_SPACING) for l, h in zip(lo, hi)]
    G3 = np.stack(np.meshgrid(*axes_, indexing='ij'), -1).reshape(-1, 3)
    dens = np.zeros(len(G3))
    tree = cKDTree(P)
    for i, nb in enumerate(tree.query_ball_point(G3, 3*SURF_SIGMA)):
        if nb:
            dens[i] = np.exp(-((P[nb] - G3[i])**2).sum(1)/(2*SURF_SIGMA**2)).sum()
    dens = dens.reshape([len(a) for a in axes_])
    verts, faces, _, _ = marching_cubes(dens, level=0.35*dens.max())
    verts = verts*SURF_SPACING + lo
    n = len(verts)
    adj = csr_matrix((np.ones(3*len(faces), np.int8),
                      (np.r_[faces[:, 0], faces[:, 1], faces[:, 2]], np.r_[faces[:, 1], faces[:, 2], faces[:, 0]])), shape=(n, n))
    _, labels = sc_connected_components(adj, directed=False)
    keep = np.bincount(labels).argmax()
    remap = np.full(n, -1); idx = np.where(labels == keep)[0]; remap[idx] = np.arange(len(idx))
    faces = remap[faces[(labels[faces] == keep).all(1)]]
    return verts[idx], faces

SURF = {c: _chain_surface(c) for c in BODY_CHAINS}
print("\n" + ", ".join(f"chain {c}: {len(SURF[c][1])} triangles" for c in BODY_CHAINS))

_LIGHT = np.array([0.5, 0.8, 1.0]); _LIGHT /= np.linalg.norm(_LIGHT)
CHAIN_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#8e5bd0']

def _shade(tri, hex_color, alpha, ambient=0.35):
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    n /= np.maximum(np.linalg.norm(n, axis=1, keepdims=True), 1e-12)
    inten = ambient + (1 - ambient)*np.abs(n @ _LIGHT)
    rgb = np.array([int(hex_color[k:k + 2], 16)/255 for k in (1, 3, 5)])
    return np.column_stack([inten[:, None]*rgb[None, :], np.full(len(tri), alpha)])

def _mode_scale(mode):
    """1 / (largest atom displacement amplitude) for the complex eigenvector."""
    e = evecs_lab[:, mode]
    dmax = max(np.linalg.norm(e[6*b + 3:6*b + 6][None] + np.cross(e[6*b:6*b + 3][None], body_positions(b) - BODIES[b]['com']),
                              axis=1).max() for b in range(N_BODY))
    return 1.0/max(dmax, 1e-12)

def body_vertices(b, mode, amp, phase=0.0):
    """Envelope vertices of body b displaced along Re(e exp(-i phase)), scaled so the
    largest atom displacement amplitude is amp (Å)."""
    B = BODIES[b]; V, _ = SURF[B['chain']]
    Vb = V @ B['R'].T + B['t']
    if mode is None or amp == 0:
        return Vb
    e = np.real(evecs_lab[:, mode]*np.exp(-1j*phase))*_mode_scale(mode)*amp
    om, v = e[6*b:6*b + 3], e[6*b + 3:6*b + 6]
    return Rot.from_rotvec(om).apply(Vb - B['com']) + B['com'] + v

_cell_edges = [(np.array(p), np.array(q)) for p in product((0, 1), repeat=3) for q in product((0, 1), repeat=3)
               if sum(abs(x - y) for x, y in zip(p, q)) == 1 and p < q]

def make_cell_mode_gif(mode, amplitude=AMPLITUDE, n_frames=N_FRAMES, fps=GIF_FPS, fname=None):
    fname = fname or f"{OUT_PREFIX}mode_{mode}_q{'_'.join(f'{x:.2f}' for x in Q_VIZ)}.gif"
    allv = np.vstack([body_vertices(b, None, 0) for b in range(N_BODY)])
    pad = 2.5*amplitude
    lims = [(allv[:, k].min() - pad, allv[:, k].max() + pad) for k in range(3)]
    e = evecs_lab[:, mode]
    ke_rot = sum(np.real(e[6*b:6*b + 3].conj() @ BODIES[b]['J'] @ e[6*b:6*b + 3]) for b in range(N_BODY))
    ke_tr = sum(BODIES[b]['m']*np.real(e[6*b + 3:6*b + 6].conj() @ e[6*b + 3:6*b + 6]) for b in range(N_BODY))
    rot_pct = 100*ke_rot/max(ke_rot + ke_tr, 1e-30)
    images = []
    for fi in range(n_frames):
        ph = 2*np.pi*fi/n_frames
        fig = plt.figure(figsize=(7, 6.5), facecolor='#0d1117')
        ax = fig.add_subplot(111, projection='3d', facecolor='#0d1117')
        try:
            for b in range(N_BODY):
                Vb = body_vertices(b, mode, amplitude, ph); tri = Vb[SURF[BODIES[b]['chain']][1]]
                col = CHAIN_COLORS[CHAIN_INDEX[BODIES[b]['chain']] % len(CHAIN_COLORS)]
                ax.add_collection3d(Poly3DCollection(tri, facecolors=_shade(tri, col, 0.85 if BODIES[b]['op'] == 0 else 0.45),
                                                     linewidths=0, edgecolor='none'))
            for p, q in _cell_edges:
                seg = np.array([A_orth @ p, A_orth @ q])
                ax.plot(*seg.T, color='0.6', lw=0.6)
            ax.set_xlim(*lims[0]); ax.set_ylim(*lims[1]); ax.set_zlim(*lims[2])
            ax.set_box_aspect([l[1] - l[0] for l in lims])
            ax.set_axis_off(); ax.view_init(elev=22, azim=-58)
            ax.set_title(f'Mode {mode}   {freqs_viz[mode]:.4f} THz   q=({Q_VIZ[0]:g}, {Q_VIZ[1]:g}, {Q_VIZ[2]:g})\n'
                         f'rot {rot_pct:.0f}%  trans {100 - rot_pct:.0f}%   ' +
                         '  '.join(f'{nm}: chain {c}' for nm, c in zip(['blue', 'orange', 'green', 'purple'], BODY_CHAINS)),
                         color='white', fontsize=8.5, pad=3)
            plt.tight_layout(pad=0.2)
            images.append(fig_to_image(fig))
        finally:
            plt.close(fig)
    imageio.mimsave(fname, images, fps=fps, loop=0)
    del images; gc.collect()
    return fname

print(f"\nGenerating unit-cell GIFs of the {N_MODES_VIZ} lowest modes at q = {Q_VIZ} "
      f"(amplitude {AMPLITUDE} Å) ...")
for s in range(N_MODES_VIZ):
    try:
        print(f"   mode {s}: {freqs_viz[s]:.5f} THz -> {make_cell_mode_gif(s)}")
    except Exception as e:
        print(f"   mode {s}: FAILED ({e})")
print(f"Load a GIF with: display(IPImage(filename='{OUT_PREFIX}mode_0_q....gif'))")

## Summary

| Output | File (prefix `experimental_7tx0_halo_`) | Description |
|--------|------|-------------|
| Solvent contrast, contacts, band structure | `solvent_contrast.png`, `contact_network.png`, `band_structure.png` | The model, as in the synthetic notebook |
| Data diagnostics | `resolution_diagnostics.png` | $I/\sigma$, finite and negative fractions, half-dataset $CC_{1/2}$ and $CC^*$ vs resolution |
| Halo selection | `halo_coupling.png` | $|qF_{\rm cell}|^2$ per Laue orbit and the selected halos |
| Sampling | `sampled_q_points.png` | Fitted and held-out voxels over the measured $l=0$, $\tfrac12$ layers |
| λ scan | `lambda_scan.png` | Cross-validated $R$ (folds grouped by halo), distance from the prior, $\chi^2$/dof |
| Fit | `refinement_fit_quality.png`, `isotropic_background.png`, `per_halo_amplitude_test.png` | Observed vs model, the background, per-halo amplitude test |
| What the data fix | `sloppiness_spectrum.png`, `posterior_bands.png`, `direction_block_composition.png`, `block_constraint_fit.png`, `elastic_tensor.png` | Invariant spectrum, posterior on the dispersion, composition of the determined directions, per-orbit block shares, elastic tensor |
| Maps | `map_measured_vs_fit_l0.png`, `map_measured_vs_fit_l0.5.png` | Measured, fitted and prior maps with normalized residuals |
| CC by resolution | `cc_resolution_shells.png` | Fit and prior against the half-dataset ceiling $CC^*$ |
| ADPs | `adp_vs_deposited.png` | Rigid-body B pattern vs deposited 7tx0 |
| Pair covariance | `pair_correlation_maps.png` | $\rho_{ij}$ of the fit, and its change from the prior |
| Modes | `band_structure_convergence.gif`, `mode_N_q*.gif` | Refinement path; lowest modes of the cell |

**Data.** mdx2 `merged.nxs` (CHESS 2025-11-05), read with `h5py`: `hkl_table` on a grid
subdivided `ndiv` times, one asymmetric unit of 4/m, with half-datasets. Bragg voxels
(integer, allowed) are never fitted. The indexing is tested against the structure
and the data are reindexed if a twinning-related indexing fits better.

**Model.** 7tx0 chains A and B as rigid bodies, eight per cell; 7 symmetry orbits of
contacts, 147 stiffness parameters; hybrid-map contrast transforms.

**Fit.** Halo voxels of the brightest halos (per Laue orbit) in 2.0–2.5 Å, natural
weights, a profiled isotropic background, the prior calibrated to the map's scale,
λ by grouped cross-validation. The map is not on an absolute scale: relative
stiffnesses, mode shapes and ratios are meaningful, absolute stiffnesses and ADPs are not.